# 10 — Continuous-Signal Pipeline and Inference Audit

This notebook records the later Paderborn signal-inspection and inference workflow. It examines raw MATLAB structure and named channels, inspects timing and sampling quality, prepares baseline feature data, checks the standalone inference module and evaluates the continuous-recording prediction path.

## Important separation

This is the Paderborn packaged 34-feature binary baseline / continuous-signal track. It is separate from the 22-feature five-recording sequence experiments and their Stage 1 / Stage 2 teacher models.

The notebook expects the original Paderborn recordings and locally generated processed files under the project data folders. Raw recordings and generated intermediate data are not included in Git.

In [ ]:
# ============================================================
# STEP 9A-1: RAW DATASET AND METADATA INSPECTION
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path

# ------------------------------------------------------------
# 1. Define project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"

BASELINE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "baseline"
)

DEVELOPMENT_CSV = (
    BASELINE_DIR / "paderborn_development_features.csv"
)

MODEL_CONFIG = (
    PROJECT_ROOT
    / "models"
    / "paderborn_baseline"
    / "feature_config.json"
)


# ------------------------------------------------------------
# 2. Verify directory and file availability
# ------------------------------------------------------------

assert PROJECT_ROOT.exists(), "Project directory not found."

assert RAW_DIR.exists(), (
    f"Raw data directory not found: {RAW_DIR}"
)

assert DEVELOPMENT_CSV.exists(), (
    "Development dataset not found."
)

assert MODEL_CONFIG.exists(), (
    "Baseline feature configuration not found."
)


# ------------------------------------------------------------
# 3. Load development metadata
# ------------------------------------------------------------

development_df = pd.read_csv(DEVELOPMENT_CSV)

print("DEVELOPMENT DATA")
print("-" * 50)

print("Shape:", development_df.shape)

print(
    "Physical bearings:",
    development_df["bearing"].nunique()
)

print(
    "Operating conditions:",
    development_df["operating_condition"].nunique()
)

print("\nFault class distribution:")

print(
    development_df["fault_class"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. Load the final feature configuration
# ------------------------------------------------------------

import json

with open(MODEL_CONFIG, "r") as f:
    feature_config = json.load(f)

FEATURES_FINAL = feature_config["features"]

print("\nMODEL FEATURE COUNT:", len(FEATURES_FINAL))


# ------------------------------------------------------------
# 5. Identify metadata columns
# ------------------------------------------------------------

metadata_columns = [
    column
    for column in development_df.columns
    if column not in FEATURES_FINAL
]

print("\nMETADATA COLUMNS")
print("-" * 50)

print(metadata_columns)


# ------------------------------------------------------------
# 6. Inspect representative recordings
# ------------------------------------------------------------

representative_records = (
    development_df[
        development_df["fault_class"].isin([0, 1, 2])
    ]
    .drop_duplicates(["fault_class", "bearing"])
    .groupby("fault_class", sort=True)
    .head(2)
)

print("\nREPRESENTATIVE BEARING RECORDS")

display(
    representative_records[metadata_columns]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Locate original MAT files
# ------------------------------------------------------------

mat_files = list(
    RAW_DIR.rglob("*.mat")
)

print("\nRAW MAT FILE INSPECTION")
print("-" * 50)

print("Total MAT files discovered:", len(mat_files))

print("\nFirst 15 MAT files:")

for file_path in mat_files[:15]:

    print(
        file_path.relative_to(PROJECT_ROOT),
        f" | Size: {file_path.stat().st_size / (1024 * 1024):.2f} MB"
    )


# ------------------------------------------------------------
# 8. Check dataset availability
# ------------------------------------------------------------

assert len(mat_files) > 0, (
    "No MAT files found in the raw dataset directory."
)

print("\nRaw dataset inspection completed successfully.")

In [ ]:
# ============================================================
# STEP 9A-2: INSPECT INTERNAL MAT FILE STRUCTURE
# ============================================================

import scipy.io as sio
import numpy as np

# ------------------------------------------------------------
# 1. Select a Healthy training recording
# ------------------------------------------------------------

selected_record = development_df[
    (development_df["bearing"] == "K001") &
    (development_df["operating_condition"] == "N09_M07_F10") &
    (development_df["measurement_number"] == 1)
].iloc[0]

mat_filename = selected_record["file"]
bearing_id = selected_record["bearing"]

mat_path = RAW_DIR / bearing_id / mat_filename

assert mat_path.exists(), "MAT file not found!"

print("SELECTED RECORDING")
print("-" * 50)

print("Bearing:", bearing_id)
print("Fault:", selected_record["fault_location"])
print("Condition:", selected_record["operating_condition"])
print("File:", mat_filename)


# ------------------------------------------------------------
# 2. Load MAT file
# ------------------------------------------------------------

mat_contents = sio.loadmat(
    mat_path,
    squeeze_me=True,
    struct_as_record=False
)

# Find actual MATLAB variable names.
mat_keys = [
    key for key in mat_contents.keys()
    if not key.startswith("__")
]

print("\nTOP-LEVEL VARIABLES")
print("-" * 50)

for key in mat_keys:
    value = mat_contents[key]

    print(
        key,
        "| Type:", type(value).__name__,
        "| Shape:", np.shape(value)
    )


# ------------------------------------------------------------
# 3. Access the actual main MATLAB structure
# ------------------------------------------------------------

assert len(mat_keys) == 1, (
    "Unexpected number of top-level variables."
)

root_key = mat_keys[0]

data_object = mat_contents[root_key]

print("\nMAIN STRUCTURE")
print("-" * 50)

print("Variable name:", root_key)
print("Object type:", type(data_object).__name__)

assert hasattr(data_object, "_fieldnames"), (
    "Main variable is not a MATLAB structure."
)


# ------------------------------------------------------------
# 4. Inspect all main structure fields
# ------------------------------------------------------------

print("\nMAIN STRUCTURE FIELDS")
print("-" * 50)

for field in data_object._fieldnames:

    value = getattr(data_object, field)

    print(
        f"{field}:",
        "Type:", type(value).__name__,
        "| Shape:", np.shape(value),
        "| Dtype:", getattr(value, "dtype", "N/A")
    )


# ------------------------------------------------------------
# 5. Inspect nested Y structure, if available
# ------------------------------------------------------------

if hasattr(data_object, "Y"):

    y_object = data_object.Y

    print("\nY FIELD DETAILS")
    print("-" * 50)

    print("Type:", type(y_object).__name__)
    print("Shape:", np.shape(y_object))
    print("Dtype:", getattr(y_object, "dtype", "N/A"))

    # Case 1: Y is itself a MATLAB structure.
    if hasattr(y_object, "_fieldnames"):

        print("\nY SUBFIELDS")

        for field in y_object._fieldnames:

            value = getattr(y_object, field)

            print(
                f"{field}:",
                "Type:", type(value).__name__,
                "| Shape:", np.shape(value),
                "| Dtype:", getattr(value, "dtype", "N/A")
            )

    # Case 2: Y contains MATLAB structures in an object array.
    elif isinstance(y_object, np.ndarray) and y_object.dtype == object:

        print("\nNESTED STRUCTURES INSIDE Y")

        for index, item in enumerate(y_object.flat[:10]):

            print(f"\nItem {index}:")
            print("Type:", type(item).__name__)

            if hasattr(item, "_fieldnames"):

                for field in item._fieldnames:

                    value = getattr(item, field)

                    print(
                        f"  {field}:",
                        "Type:", type(value).__name__,
                        "| Shape:", np.shape(value),
                        "| Dtype:", getattr(value, "dtype", "N/A")
                    )

    else:

        print("Y contains a non-structured array.")

else:

    print("\nY field is not directly available.")


print("\nMAT STRUCTURE INSPECTION COMPLETED.")

In [ ]:
# ============================================================
# STEP 9A-3: IDENTIFY MEASUREMENT CHANNELS
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Inspect all seven Y channels
# ------------------------------------------------------------

y_channels = data_object.Y

channel_records = []

for index, channel in enumerate(y_channels):

    channel_data = np.asarray(channel.Data).squeeze()

    channel_records.append({
        "channel_index": index,
        "name": channel.Name,
        "type": channel.Type,
        "unit": channel.Unit,
        "raster": channel.Raster,
        "device": channel.Device,
        "x_index": channel.XIndex,
        "down_sampling": channel.DownSampling,
        "number_of_samples": len(channel_data),
        "minimum": np.min(channel_data),
        "maximum": np.max(channel_data)
    })


channel_summary = pd.DataFrame(channel_records)

print("MEASUREMENT CHANNEL SUMMARY")
print("-" * 60)

display(channel_summary)


# ------------------------------------------------------------
# 2. Inspect the X structure
# ------------------------------------------------------------

print("\nX STRUCTURE INSPECTION")
print("-" * 60)

x_channels = data_object.X

print("Number of X items:", len(x_channels))

for index, channel in enumerate(x_channels):

    print(f"\nX item {index}")

    if hasattr(channel, "_fieldnames"):

        for field in channel._fieldnames:

            value = getattr(channel, field)

            if isinstance(value, np.ndarray):
                description = (
                    f"Shape: {value.shape}, "
                    f"Dtype: {value.dtype}"
                )
            else:
                description = str(value)

            print(f"  {field}: {description}")

    else:

        print("  Type:", type(channel).__name__)
        print("  Value:", channel)


print("\nCHANNEL IDENTIFICATION COMPLETED.")

In [ ]:
# ============================================================
# STEP 9A-3 (CONTINUED): MAP Y CHANNELS TO X REFERENCES
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Inspect X reference structures
# ------------------------------------------------------------

x_channels = data_object.X

x_records = []

for index, x_channel in enumerate(x_channels):

    x_data = np.asarray(x_channel.Data).squeeze()

    x_records.append({
        "x_index": index,
        "raster": x_channel.Raster,
        "samples": len(x_data),
        "start_value": x_data[0] if len(x_data) else np.nan,
        "end_value": x_data[-1] if len(x_data) else np.nan,
        "median_increment": (
            np.median(np.diff(x_data))
            if len(x_data) > 1 else np.nan
        )
    })

x_reference_df = pd.DataFrame(x_records)

print("X REFERENCE SUMMARY")
display(x_reference_df)


# ------------------------------------------------------------
# 2. Inspect all Y measurement channels
# ------------------------------------------------------------

y_channels = data_object.Y

y_records = []

for index, y_channel in enumerate(y_channels):

    signal = np.asarray(y_channel.Data).squeeze()

    y_records.append({
        "y_index": index,
        "name": y_channel.Name,
        "unit": y_channel.Unit,
        "raster": y_channel.Raster,
        "device": y_channel.Device,
        "x_index": y_channel.XIndex,
        "down_sampling": y_channel.DownSampling,
        "samples": len(signal),
        "minimum": np.min(signal),
        "maximum": np.max(signal)
    })

y_channel_df = pd.DataFrame(y_records)

print("\nY MEASUREMENT CHANNEL SUMMARY")
display(y_channel_df)


# ------------------------------------------------------------
# 3. Print explicit Y-to-X reference information
# ------------------------------------------------------------

print("\nY TO X REFERENCE MAPPING")
print("-" * 65)

for _, row in y_channel_df.iterrows():

    print(
        f"Y[{row['y_index']}] | "
        f"Name: {row['name']} | "
        f"XIndex: {row['x_index']} | "
        f"Samples: {row['samples']} | "
        f"Raster: {row['raster']} | "
        f"DownSampling: {row['down_sampling']}"
    )

print("\nChannel mapping inspection completed.")

In [ ]:
# ============================================================
# STEP 9A-4: INSPECT RAW VIBRATION WAVEFORMS
# ============================================================

import scipy.io as sio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Select representative training recordings
# ------------------------------------------------------------

selected_bearings = {
    "Healthy": "K001",
    "Outer": "KA01",
    "Inner": "KI01"
}

selected_condition = "N09_M07_F10"
selected_measurement = 1

signal_records = {}

# ------------------------------------------------------------
# 2. Load each recording and identify its channels
# ------------------------------------------------------------

for class_name, bearing in selected_bearings.items():

    record = development_df[
        (development_df["bearing"] == bearing) &
        (development_df["operating_condition"] == selected_condition) &
        (development_df["measurement_number"] == selected_measurement)
    ]

    assert len(record) == 1, (
        f"Expected one recording for {bearing}, found {len(record)}"
    )

    record = record.iloc[0]

    mat_path = RAW_DIR / bearing / record["file"]

    mat_contents = sio.loadmat(
        mat_path,
        squeeze_me=True,
        struct_as_record=False
    )

    # Identify the main MATLAB structure dynamically.
    mat_keys = [
        key for key in mat_contents.keys()
        if not key.startswith("__")
    ]

    assert len(mat_keys) == 1

    root = mat_contents[mat_keys[0]]

    x_channels = root.X
    y_channels = root.Y

    # Build a dictionary using the actual channel names.
    channel_map = {
        channel.Name: channel
        for channel in y_channels
    }

    # Extract vibration and speed.
    vibration_channel = channel_map["vibration_1"]
    speed_channel = channel_map["speed"]

    vibration = np.asarray(
        vibration_channel.Data,
        dtype=np.float64
    ).squeeze()

    speed = np.asarray(
        speed_channel.Data,
        dtype=np.float64
    ).squeeze()

    # XIndex uses MATLAB-style one-based indexing.
    vibration_time = np.asarray(
        x_channels[vibration_channel.XIndex - 1].Data,
        dtype=np.float64
    ).squeeze()

    speed_time = np.asarray(
        x_channels[speed_channel.XIndex - 1].Data,
        dtype=np.float64
    ).squeeze()

    assert len(vibration) == len(vibration_time)
    assert len(speed) == len(speed_time)

    # Calculate sampling frequencies from timestamps.
    vibration_fs = 1 / np.mean(np.diff(vibration_time))
    speed_fs = 1 / np.mean(np.diff(speed_time))

    signal_records[class_name] = {
        "bearing": bearing,
        "vibration": vibration,
        "vibration_time": vibration_time,
        "vibration_fs": vibration_fs,
        "speed": speed,
        "speed_time": speed_time,
        "speed_fs": speed_fs
    }


# ------------------------------------------------------------
# 3. Display signal information
# ------------------------------------------------------------

signal_summary = []

for class_name, data in signal_records.items():

    vibration = data["vibration"]
    speed = data["speed"]
    time = data["vibration_time"]

    signal_summary.append({
        "class": class_name,
        "bearing": data["bearing"],
        "vibration_samples": len(vibration),
        "vibration_sampling_rate": data["vibration_fs"],
        "duration_seconds": time[-1] - time[0],
        "vibration_rms": np.sqrt(np.mean(vibration ** 2)),
        "vibration_min": np.min(vibration),
        "vibration_max": np.max(vibration),
        "average_speed_rpm": np.mean(speed),
        "minimum_speed_rpm": np.min(speed),
        "maximum_speed_rpm": np.max(speed)
    })


signal_summary_df = pd.DataFrame(signal_summary)

print("RAW SIGNAL SUMMARY")

display(signal_summary_df.round(3))


# ------------------------------------------------------------
# 4. Plot first 0.05 seconds of vibration
# ------------------------------------------------------------

plot_duration = 0.05

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(14, 10)
)

for ax, (class_name, data) in zip(
    axes,
    signal_records.items()
):

    time = data["vibration_time"]
    vibration = data["vibration"]

    # Use relative time, starting from zero.
    relative_time = time - time[0]

    mask = relative_time <= plot_duration

    ax.plot(
        relative_time[mask],
        vibration[mask],
        linewidth=0.7
    )

    ax.set_title(
        f"{class_name} Bearing - {data['bearing']}"
    )

    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Vibration amplitude")
    ax.grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 9A-5: VERIFY VIBRATION TIMESTAMP CONSISTENCY
# ============================================================

import numpy as np
import pandas as pd

NOMINAL_FS = 64000.0

sampling_records = []

for class_name, data in signal_records.items():

    time = np.asarray(
        data["vibration_time"],
        dtype=np.float64
    )

    dt = np.diff(time)

    # Calculate sampling interval statistics.
    mean_dt = np.mean(dt)
    median_dt = np.median(dt)

    # Calculate variation in timestamp intervals.
    deviation_pct = (
        np.abs(dt - median_dt) / median_dt
    ) * 100

    duration = time[-1] - time[0]

    # Sampling frequency estimated using mean interval.
    fs_from_mean = 1 / mean_dt

    # Sampling frequency estimated using median interval.
    fs_from_median = 1 / median_dt

    # Expected samples at nominal 64 kHz.
    expected_samples = round(
        duration * NOMINAL_FS
    ) + 1

    sampling_records.append({
        "class": class_name,
        "bearing": data["bearing"],
        "samples": len(time),
        "duration_seconds": duration,
        "mean_dt_us": mean_dt * 1e6,
        "median_dt_us": median_dt * 1e6,
        "std_dt_us": np.std(dt) * 1e6,
        "minimum_dt_us": np.min(dt) * 1e6,
        "maximum_dt_us": np.max(dt) * 1e6,
        "fs_from_mean": fs_from_mean,
        "fs_from_median": fs_from_median,
        "expected_samples_64k": expected_samples,
        "sample_difference": len(time) - expected_samples,
        "nonpositive_intervals": np.sum(dt <= 0),
        "intervals_deviating_over_1pct": np.sum(
            deviation_pct > 1
        ),
        "interval_deviation_pct_mean": np.mean(
            deviation_pct
        )
    })


sampling_diagnostics_df = pd.DataFrame(
    sampling_records
)

print("VIBRATION TIMESTAMP DIAGNOSTICS")
print("-" * 60)

display(
    sampling_diagnostics_df.round(6)
)


# ------------------------------------------------------------
# Additional checks
# ------------------------------------------------------------

print("\nTIMESTAMP INTEGRITY CHECK")
print("-" * 60)

for class_name, data in signal_records.items():

    time = data["vibration_time"]
    dt = np.diff(time)

    print(f"\n{class_name} - {data['bearing']}")

    print("Strictly increasing timestamps:", np.all(dt > 0))

    print(
        "Approximately uniform intervals:",
        np.std(dt) / np.mean(dt) < 0.01
    )

print("\nTimestamp diagnostic completed.")

In [ ]:
# ============================================================
# STEP 9A-5B: INVESTIGATE TIMESTAMP IRREGULARITIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOMINAL_DT = 1 / 64000

interval_diagnostics = []

fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(15, 11)
)

for row, (class_name, data) in enumerate(signal_records.items()):

    time = data["vibration_time"]

    dt = np.diff(time)

    # Convert intervals to microseconds.
    dt_us = dt * 1e6

    # Calculate percent deviation from nominal 64 kHz interval.
    deviation_pct = (
        np.abs(dt - NOMINAL_DT) / NOMINAL_DT
    ) * 100

    # Calculate interval percentiles.
    percentiles = np.percentile(
        dt_us,
        [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]
    )

    interval_diagnostics.append({
        "bearing": data["bearing"],
        "minimum_us": percentiles[0],
        "p01_us": percentiles[1],
        "p05_us": percentiles[2],
        "p10_us": percentiles[3],
        "p25_us": percentiles[4],
        "median_us": percentiles[5],
        "p75_us": percentiles[6],
        "p90_us": percentiles[7],
        "p95_us": percentiles[8],
        "p99_us": percentiles[9],
        "maximum_us": percentiles[10],
        "within_1pct_nominal": (
            100 * np.mean(deviation_pct <= 1)
        ),
        "within_5pct_nominal": (
            100 * np.mean(deviation_pct <= 5)
        )
    })

    # --------------------------------------------------------
    # Plot 1: Interval histogram
    # --------------------------------------------------------

    axes[row, 0].hist(
        dt_us,
        bins=60
    )

    axes[row, 0].axvline(
        NOMINAL_DT * 1e6,
        linestyle="--",
        label="Nominal 64 kHz interval"
    )

    axes[row, 0].set_title(
        f"{class_name} - {data['bearing']} Timestamp Distribution"
    )

    axes[row, 0].set_xlabel("Interval (microseconds)")
    axes[row, 0].set_ylabel("Frequency")
    axes[row, 0].legend()

    # --------------------------------------------------------
    # Plot 2: Timestamp intervals against sample index
    # --------------------------------------------------------

    sample_limit = min(20000, len(dt_us))

    axes[row, 1].plot(
        np.arange(sample_limit),
        dt_us[:sample_limit],
        linewidth=0.5
    )

    axes[row, 1].axhline(
        NOMINAL_DT * 1e6,
        linestyle="--",
        label="Nominal interval"
    )

    axes[row, 1].set_title(
        f"{class_name} - First {sample_limit} Intervals"
    )

    axes[row, 1].set_xlabel("Sample index")
    axes[row, 1].set_ylabel("Interval (microseconds)")
    axes[row, 1].legend()


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Display numerical interval statistics
# ------------------------------------------------------------

interval_diagnostics_df = pd.DataFrame(
    interval_diagnostics
)

print("TIMESTAMP INTERVAL DISTRIBUTION")

display(
    interval_diagnostics_df.round(5)
)

In [ ]:
# ============================================================
# STEP 9A-5C: QUANTIFY TIMESTAMP INTERVAL PATTERNS
# ============================================================

import numpy as np
import pandas as pd

NOMINAL_FS = 64000.0
NOMINAL_DT_US = (1 / NOMINAL_FS) * 1e6

timestamp_pattern_results = []

for class_name, data in signal_records.items():

    time = np.asarray(
        data["vibration_time"],
        dtype=np.float64
    )

    dt_us = np.diff(time) * 1e6

    # Round intervals to three decimal places
    # to group the dominant timestamp values.
    rounded_intervals = np.round(dt_us, 3)

    interval_counts = (
        pd.Series(rounded_intervals)
        .value_counts()
        .sort_values(ascending=False)
    )

    # Identify the two most frequent intervals.
    dominant_intervals = interval_counts.head(2)

    print(f"\n{class_name} - {data['bearing']}")
    print("-" * 55)

    print("Total intervals:", len(dt_us))

    print("\nMost frequent intervals:")

    dominant_labels = {}

    for interval, count in dominant_intervals.items():

        percentage = 100 * count / len(dt_us)

        print(
            f"{interval:.3f} us : "
            f"{count} occurrences "
            f"({percentage:.2f}%)"
        )

    # Assign dominant interval categories.
    short_interval = dominant_intervals.index[0]
    long_interval = dominant_intervals.index[1]

    labels = np.where(
        np.abs(rounded_intervals - short_interval) < 0.001,
        "Short",
        np.where(
            np.abs(rounded_intervals - long_interval) < 0.001,
            "Long",
            "Other"
        )
    )

    transitions = np.sum(labels[1:] != labels[:-1])

    print("\nInterval transitions:", transitions)

    print(
        "Other interval count:",
        np.sum(labels == "Other")
    )

    print("\nFirst 30 interval categories:")
    print(labels[:30])

    # Fit an average linear timebase against sample index.
    sample_index = np.arange(len(time))

    slope, intercept = np.polyfit(
        sample_index,
        time,
        1
    )

    fitted_time = intercept + slope * sample_index

    residual_us = (
        time - fitted_time
    ) * 1e6

    fitted_fs = 1 / slope

    timestamp_pattern_results.append({
        "bearing": data["bearing"],
        "samples": len(time),
        "short_interval_us": short_interval,
        "long_interval_us": long_interval,
        "short_interval_pct": (
            100 * np.mean(labels == "Short")
        ),
        "long_interval_pct": (
            100 * np.mean(labels == "Long")
        ),
        "other_interval_pct": (
            100 * np.mean(labels == "Other")
        ),
        "transition_count": transitions,
        "fitted_sampling_rate": fitted_fs,
        "residual_std_us": np.std(residual_us),
        "maximum_absolute_residual_us": (
            np.max(np.abs(residual_us))
        )
    })


timestamp_pattern_df = pd.DataFrame(
    timestamp_pattern_results
)

print("\n\nTIMESTAMP PATTERN SUMMARY")

display(timestamp_pattern_df.round(5))

In [ ]:
# ============================================================
# STEP 9A-5D: INVESTIGATE K001 TIMESTAMP ANOMALIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Inspect K001 timestamps
# ------------------------------------------------------------

data = signal_records["Healthy"]

time = np.asarray(
    data["vibration_time"],
    dtype=np.float64
)

dt_us = np.diff(time) * 1e6

# Identify the two dominant timestamp intervals.
rounded_dt = np.round(dt_us, 3)

interval_counts = (
    pd.Series(rounded_dt)
    .value_counts()
    .head(2)
)

dominant_intervals = np.sort(
    interval_counts.index.to_numpy()
)

short_interval = dominant_intervals[0]
long_interval = dominant_intervals[1]

print("K001 DOMINANT INTERVALS")
print("-" * 50)

print("Short interval:", short_interval, "us")
print("Long interval:", long_interval, "us")


# ------------------------------------------------------------
# 2. Identify unusual intervals
# ------------------------------------------------------------

normal_mask = np.isin(
    rounded_dt,
    dominant_intervals
)

anomaly_indices = np.where(~normal_mask)[0]

anomaly_values = dt_us[~normal_mask]

print("\nANOMALY SUMMARY")
print("-" * 50)

print("Total intervals:", len(dt_us))
print("Normal intervals:", normal_mask.sum())
print("Unusual intervals:", len(anomaly_indices))

if len(anomaly_values) > 0:

    print("\nUnusual interval statistics (microseconds):")

    print(
        pd.Series(anomaly_values).describe(
            percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
        )
    )


# ------------------------------------------------------------
# 3. Identify consecutive anomaly groups
# ------------------------------------------------------------

if len(anomaly_indices) > 0:

    group_breaks = np.where(
        np.diff(anomaly_indices) > 1
    )[0]

    group_starts = np.r_[
        0,
        group_breaks + 1
    ]

    group_ends = np.r_[
        group_breaks,
        len(anomaly_indices) - 1
    ]

    anomaly_groups = []

    for start, end in zip(group_starts, group_ends):

        indices = anomaly_indices[start:end + 1]

        anomaly_groups.append({
            "start_interval_index": int(indices[0]),
            "end_interval_index": int(indices[-1]),
            "number_of_intervals": len(indices),
            "minimum_interval_us": dt_us[indices].min(),
            "maximum_interval_us": dt_us[indices].max()
        })

    anomaly_groups_df = pd.DataFrame(anomaly_groups)

    anomaly_groups_df = anomaly_groups_df.sort_values(
        "number_of_intervals",
        ascending=False
    )

    print("\nLARGEST CONSECUTIVE ANOMALY GROUPS")

    display(
        anomaly_groups_df.head(20).round(5)
    )


# ------------------------------------------------------------
# 4. Display intervals around the first unusual samples
# ------------------------------------------------------------

print("\nFIRST UNUSUAL INTERVALS WITH NEIGHBOURS")

context_records = []

for index in anomaly_indices[:10]:

    for neighbour in range(
        max(0, index - 2),
        min(len(dt_us), index + 3)
    ):

        context_records.append({
            "interval_index": neighbour,
            "time_seconds": time[neighbour + 1],
            "interval_us": dt_us[neighbour],
            "is_anomaly": not normal_mask[neighbour]
        })

context_df = (
    pd.DataFrame(context_records)
    .drop_duplicates("interval_index")
    .sort_values("interval_index")
)

display(context_df.round(6))


# ------------------------------------------------------------
# 5. Plot timestamp residual against an average linear grid
# ------------------------------------------------------------

sample_indices = np.arange(len(time))

slope, intercept = np.polyfit(
    sample_indices,
    time,
    1
)

fitted_time = (
    intercept + slope * sample_indices
)

residual_us = (
    time - fitted_time
) * 1e6

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 8)
)

# Complete residual pattern.
axes[0].plot(
    time - time[0],
    residual_us,
    linewidth=0.5
)

axes[0].set_title(
    "K001: Timestamp Residual from Average Linear Timebase"
)

axes[0].set_xlabel("Time (seconds)")
axes[0].set_ylabel("Residual (microseconds)")
axes[0].grid(alpha=0.3)


# First 0.1 seconds.
early_mask = (time - time[0]) <= 0.1

axes[1].plot(
    time[early_mask] - time[0],
    residual_us[early_mask],
    linewidth=0.7
)

axes[1].set_title(
    "K001: Timestamp Residual During First 0.1 Seconds"
)

axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Residual (microseconds)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


print("\nK001 anomaly investigation completed.")

In [ ]:
# ============================================================
# STEP 9A-5E: INSPECT K001 SIGNAL AROUND TIMESTAMP ANOMALY
# ============================================================

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Identify anomaly position in K001 measurement 1
# ------------------------------------------------------------

k001_data = signal_records["Healthy"]

vibration_1 = k001_data["vibration"]
time_1 = k001_data["vibration_time"]

dt_us_1 = np.diff(time_1) * 1e6

rounded_dt_1 = np.round(dt_us_1, 3)

dominant_intervals_1 = (
    pd.Series(rounded_dt_1)
    .value_counts()
    .head(2)
    .index.to_numpy()
)

normal_mask_1 = np.isin(
    rounded_dt_1,
    dominant_intervals_1
)

anomaly_indices_1 = np.where(
    ~normal_mask_1
)[0]

assert len(anomaly_indices_1) > 0

start_idx = int(anomaly_indices_1[0])
end_idx = int(anomaly_indices_1[-1] + 1)

print("K001 MEASUREMENT 1 ANOMALY")
print("-" * 55)

print("Start index:", start_idx)
print("End index:", end_idx)
print("Affected samples:", end_idx - start_idx + 1)

print(
    "Start timestamp:",
    time_1[start_idx]
)

print(
    "End timestamp:",
    time_1[end_idx]
)


# ------------------------------------------------------------
# 2. Compare signal statistics around anomaly
# ------------------------------------------------------------

block_length = end_idx - start_idx + 1

before_start = max(0, start_idx - block_length)
after_end = min(
    len(vibration_1),
    end_idx + 1 + block_length
)

segments = {
    "Before anomaly": vibration_1[
        before_start:start_idx
    ],

    "During anomaly": vibration_1[
        start_idx:end_idx + 1
    ],

    "After anomaly": vibration_1[
        end_idx + 1:after_end
    ]
}

segment_results = []

for name, signal in segments.items():

    segment_results.append({
        "segment": name,
        "samples": len(signal),
        "rms": np.sqrt(np.mean(signal ** 2)),
        "mean": np.mean(signal),
        "std": np.std(signal),
        "minimum": np.min(signal),
        "maximum": np.max(signal)
    })

segment_summary = pd.DataFrame(segment_results)

print("\nSIGNAL STATISTICS AROUND ANOMALY")

display(segment_summary.round(5))


# ------------------------------------------------------------
# 3. Plot vibration against sample index
# ------------------------------------------------------------

plot_start = max(0, start_idx - 1000)

plot_end = min(
    len(vibration_1),
    end_idx + 1001
)

sample_indices = np.arange(
    plot_start,
    plot_end
)

plt.figure(figsize=(14, 5))

plt.plot(
    sample_indices,
    vibration_1[plot_start:plot_end],
    linewidth=0.6
)

plt.axvspan(
    start_idx,
    end_idx,
    alpha=0.2,
    label="Timestamp anomaly region"
)

plt.axvline(
    start_idx,
    linestyle="--",
    label="Anomaly start"
)

plt.axvline(
    end_idx,
    linestyle="--",
    label="Anomaly end"
)

plt.title(
    "K001 Measurement 1: Vibration Around Timestamp Anomaly"
)

plt.xlabel("Sample index")
plt.ylabel("Vibration amplitude")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 4. Plot timestamp intervals around anomaly
# ------------------------------------------------------------

local_start = max(0, start_idx - 100)
local_end = min(
    len(dt_us_1),
    end_idx + 101
)

plt.figure(figsize=(14, 4))

plt.plot(
    np.arange(local_start, local_end),
    dt_us_1[local_start:local_end],
    linewidth=0.8
)

plt.axvspan(
    start_idx,
    end_idx,
    alpha=0.2,
    label="Timestamp anomaly region"
)

plt.axhline(
    1 / 64000 * 1e6,
    linestyle="--",
    label="Nominal 64 kHz interval"
)

plt.title("K001 Measurement 1: Local Timestamp Intervals")

plt.xlabel("Interval index")
plt.ylabel("Interval (microseconds)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. Load another K001 recording under the same condition
# ------------------------------------------------------------

second_record = development_df[
    (development_df["bearing"] == "K001") &
    (development_df["operating_condition"] == "N09_M07_F10") &
    (development_df["measurement_number"] == 2)
]

assert len(second_record) == 1, (
    "Second K001 recording not found."
)

second_record = second_record.iloc[0]

second_path = (
    RAW_DIR /
    "K001" /
    second_record["file"]
)

mat_contents_2 = sio.loadmat(
    second_path,
    squeeze_me=True,
    struct_as_record=False
)

root_key_2 = [
    key for key in mat_contents_2
    if not key.startswith("__")
][0]

root_2 = mat_contents_2[root_key_2]

channels_2 = {
    channel.Name: channel
    for channel in root_2.Y
}

vibration_channel_2 = channels_2["vibration_1"]

vibration_2 = np.asarray(
    vibration_channel_2.Data,
    dtype=np.float64
).squeeze()

time_2 = np.asarray(
    root_2.X[
        vibration_channel_2.XIndex - 1
    ].Data,
    dtype=np.float64
).squeeze()

dt_us_2 = np.diff(time_2) * 1e6

rounded_dt_2 = np.round(dt_us_2, 3)

interval_counts_2 = (
    pd.Series(rounded_dt_2)
    .value_counts()
    .head(3)
)

dominant_intervals_2 = (
    interval_counts_2.index.to_numpy()
)

normal_mask_2 = np.isin(
    rounded_dt_2,
    dominant_intervals_2[:2]
)

anomaly_count_2 = int(
    (~normal_mask_2).sum()
)


# ------------------------------------------------------------
# 6. Compare the two K001 recordings
# ------------------------------------------------------------

recording_comparison = pd.DataFrame([
    {
        "measurement": 1,
        "file": k001_data["bearing"],
        "samples": len(vibration_1),
        "duration": time_1[-1] - time_1[0],
        "interval_anomalies": len(anomaly_indices_1)
    },
    {
        "measurement": 2,
        "file": second_record["file"],
        "samples": len(vibration_2),
        "duration": time_2[-1] - time_2[0],
        "interval_anomalies": anomaly_count_2
    }
])

print("\nK001 RECORDING COMPARISON")

display(recording_comparison.round(5))

print("\nSecond recording's most frequent intervals:")
print(interval_counts_2)

print("\nK001 anomaly inspection completed.")

In [ ]:
# ============================================================
# STEP 9A-5F: CORRECTED K001 RECORDING INVESTIGATION
# ============================================================

import scipy.io as sio
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Retrieve the exact K001 recordings from metadata
# ------------------------------------------------------------

def get_recording_path(bearing, condition, measurement):

    matched = development_df[
        (development_df["bearing"] == bearing) &
        (development_df["operating_condition"] == condition) &
        (development_df["measurement_number"] == measurement)
    ]

    assert len(matched) == 1, (
        f"Expected one record for {bearing}, "
        f"measurement {measurement}; found {len(matched)}"
    )

    record = matched.iloc[0]

    path = RAW_DIR / bearing / record["file"]

    assert path.exists(), f"File not found: {path}"

    return record, path


record_1, path_1 = get_recording_path(
    "K001", "N09_M07_F10", 1
)

record_2, path_2 = get_recording_path(
    "K001", "N09_M07_F10", 2
)

print("EXACT FILES SELECTED")
print("-" * 60)

print("Measurement 1:", path_1)
print("Measurement 2:", path_2)


# ------------------------------------------------------------
# 2. Load both recordings independently
# ------------------------------------------------------------

loaded_recordings = {}

for measurement, path in [
    (1, path_1),
    (2, path_2)
]:

    contents = sio.loadmat(
        path,
        squeeze_me=True,
        struct_as_record=False
    )

    root_key = [
        key for key in contents
        if not key.startswith("__")
    ][0]

    root = contents[root_key]

    channel_map = {
        channel.Name: channel
        for channel in root.Y
    }

    vibration_channel = channel_map["vibration_1"]

    vibration = np.asarray(
        vibration_channel.Data,
        dtype=np.float64
    ).squeeze()

    time = np.asarray(
        root.X[vibration_channel.XIndex - 1].Data,
        dtype=np.float64
    ).squeeze()

    loaded_recordings[measurement] = {
        "root": root,
        "file": path.name,
        "vibration": vibration,
        "time": time,
        "measurement_id": root.Info.MeasurementID
    }


# ------------------------------------------------------------
# 3. Confirm identities and measurement metadata
# ------------------------------------------------------------

identity_records = []

for measurement, data in loaded_recordings.items():

    root = data["root"]

    measurement_info = root.Description.Measurement

    identity_records.append({
        "measurement": measurement,
        "filename": data["file"],
        "measurement_id": data["measurement_id"],
        "vibration_samples": len(data["vibration"]),
        "timestamp_samples": len(data["time"]),
        "duration_seconds": (
            data["time"][-1] - data["time"][0]
        ),
        "metadata_length": measurement_info.Length,
        "start_timestamp": measurement_info.StartTimestamp,
        "stop_timestamp": measurement_info.StopTimestamp,
        "x_axis_offset": measurement_info.XAxisOffset
    })


identity_df = pd.DataFrame(identity_records)

print("\nRECORDING IDENTITY AND METADATA")

display(identity_df)


# ------------------------------------------------------------
# 4. Inspect nested acquisition description
# ------------------------------------------------------------

def inspect_nested_structure(structure, name):

    print(f"\n{name}")
    print("-" * 50)

    for field in structure._fieldnames:

        value = getattr(structure, field)

        if hasattr(value, "_fieldnames"):

            print(f"\n{field}:")

            for subfield in value._fieldnames:

                subvalue = getattr(value, subfield)

                print(f"  {subfield}: {subvalue}")

        else:

            print(f"{field}: {value}")


for measurement, data in loaded_recordings.items():

    print("\n" + "=" * 60)
    print(f"MEASUREMENT {measurement}")
    print("=" * 60)

    inspect_nested_structure(
        data["root"].Description,
        "DESCRIPTION"
    )


# ------------------------------------------------------------
# 5. Recalculate timestamp anomalies for Measurement 1
# ------------------------------------------------------------

time_1 = loaded_recordings[1]["time"]
time_2 = loaded_recordings[2]["time"]

vibration_1 = loaded_recordings[1]["vibration"]
vibration_2 = loaded_recordings[2]["vibration"]

dt_us_1 = np.diff(time_1) * 1e6

rounded_dt_1 = np.round(dt_us_1, 3)

dominant_intervals = (
    pd.Series(rounded_dt_1)
    .value_counts()
    .head(2)
    .index.to_numpy()
)

normal_mask = np.isin(
    rounded_dt_1,
    dominant_intervals
)

anomaly_indices = np.where(~normal_mask)[0]

print("\nMEASUREMENT 1 TIMESTAMP ANOMALIES")
print("-" * 50)

print("Total unusual intervals:", len(anomaly_indices))

if len(anomaly_indices) > 0:

    anomaly_start = int(anomaly_indices[0])
    anomaly_end = int(anomaly_indices[-1] + 1)

    print("Anomaly start index:", anomaly_start)
    print("Anomaly end index:", anomaly_end)

    print("Start time:", time_1[anomaly_start])
    print("End time:", time_1[anomaly_end])

else:

    print("No timestamp anomalies detected.")


# ------------------------------------------------------------
# 6. Compare vibration continuity around the anomaly
# ------------------------------------------------------------

assert len(anomaly_indices) > 0

block_length = anomaly_end - anomaly_start + 1

regions = {
    "Before anomaly": (
        max(0, anomaly_start - block_length),
        anomaly_start
    ),

    "During anomaly": (
        anomaly_start,
        anomaly_end + 1
    ),

    "After anomaly": (
        anomaly_end + 1,
        anomaly_end + 1 + block_length
    )
}

continuity_records = []

for measurement, vibration in [
    (1, vibration_1),
    (2, vibration_2)
]:

    for region_name, (start, stop) in regions.items():

        assert stop <= len(vibration)

        segment = vibration[start:stop]

        differences = np.diff(segment)

        continuity_records.append({
            "measurement": measurement,
            "region": region_name,
            "samples": len(segment),
            "signal_rms": np.sqrt(
                np.mean(segment ** 2)
            ),
            "difference_rms": np.sqrt(
                np.mean(differences ** 2)
            ),
            "difference_p99_abs": np.percentile(
                np.abs(differences), 99
            ),
            "maximum_absolute_difference": np.max(
                np.abs(differences)
            )
        })


continuity_df = pd.DataFrame(continuity_records)

print("\nCORRECTED SIGNAL CONTINUITY COMPARISON")

display(continuity_df.round(5))


print("\nCORRECTED RECORDING INVESTIGATION COMPLETED.")

In [ ]:
# ============================================================
# STEP 9A-5G: AUDIT K001 RECORDINGS 1-20
# ============================================================

import scipy.io as sio
import numpy as np
import pandas as pd

NOMINAL_FS = 64000.0
NOMINAL_DT_US = 1e6 / NOMINAL_FS

# Select all 20 K001 recordings under the same condition.
k001_records = (
    development_df[
        (development_df["bearing"] == "K001") &
        (development_df["operating_condition"] == "N09_M07_F10")
    ]
    .sort_values("measurement_number")
    .reset_index(drop=True)
)

assert len(k001_records) == 20, (
    f"Expected 20 recordings, found {len(k001_records)}"
)

results = []

for _, record in k001_records.iterrows():

    measurement = int(record["measurement_number"])

    file_path = RAW_DIR / "K001" / record["file"]

    mat = sio.loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False
    )

    root_key = [
        key for key in mat
        if not key.startswith("__")
    ][0]

    root = mat[root_key]

    channels = {
        channel.Name: channel
        for channel in root.Y
    }

    vibration_channel = channels["vibration_1"]

    vibration = np.asarray(
        vibration_channel.Data,
        dtype=np.float64
    ).squeeze()

    time = np.asarray(
        root.X[vibration_channel.XIndex - 1].Data,
        dtype=np.float64
    ).squeeze()

    assert len(vibration) == len(time)

    dt_us = np.diff(time) * 1e6

    # Flag intervals differing from nominal 64 kHz
    # by more than 5%.
    anomaly_mask = (
        np.abs(dt_us - NOMINAL_DT_US)
        > 0.05 * NOMINAL_DT_US
    )

    anomaly_indices = np.where(anomaly_mask)[0]

    # Find the longest consecutive anomalous interval group.
    if len(anomaly_indices) > 0:

        breaks = np.where(
            np.diff(anomaly_indices) > 1
        )[0]

        group_starts = np.r_[0, breaks + 1]
        group_ends = np.r_[breaks, len(anomaly_indices) - 1]

        run_lengths = (
            group_ends - group_starts + 1
        )

        longest_run = int(run_lengths.max())

        anomaly_start_time = float(
            time[anomaly_indices[0]]
        )

    else:

        longest_run = 0
        anomaly_start_time = np.nan

    measurement_metadata = root.Description.Measurement

    results.append({
        "measurement": measurement,
        "file": record["file"],
        "measurement_id": root.Info.MeasurementID,
        "samples": len(vibration),
        "metadata_length": measurement_metadata.Length,
        "duration_seconds": time[-1] - time[0],
        "mean_interval_us": np.mean(dt_us),
        "median_interval_us": np.median(dt_us),
        "minimum_interval_us": np.min(dt_us),
        "maximum_interval_us": np.max(dt_us),
        "anomalous_intervals": int(anomaly_mask.sum()),
        "longest_anomaly_run": longest_run,
        "anomaly_start_time": anomaly_start_time
    })


k001_timebase_audit = pd.DataFrame(results)

print("K001 RECORDING TIMEBASE AUDIT")

display(
    k001_timebase_audit.round(5)
)


# Summary of sample-count consistency.
print("\nSAMPLE COUNT DISTRIBUTION")

display(
    k001_timebase_audit["samples"]
    .value_counts()
    .rename_axis("sample_count")
    .reset_index(name="recording_count")
)


# Overall audit summary.
print("\nAUDIT SUMMARY")

print(
    "Recordings with timestamp anomalies:",
    (k001_timebase_audit["anomalous_intervals"] > 0).sum()
)

print(
    "Recordings with non-standard sample count:",
    (k001_timebase_audit["samples"] != 256001).sum()
)

In [ ]:
# ============================================================
# STEP 9A-5H: CROSS-BEARING TIMESTAMP QUALITY AUDIT
# ============================================================

import scipy.io as sio
import numpy as np
import pandas as pd

NOMINAL_FS = 64000.0
NOMINAL_DT_US = 1e6 / NOMINAL_FS

BEARINGS_TO_AUDIT = {
    "K001": "Healthy",
    "KA01": "Outer",
    "KI01": "Inner"
}

CONDITION_TO_AUDIT = "N09_M07_F10"

audit_records = []

for bearing, class_name in BEARINGS_TO_AUDIT.items():

    print(f"\nAuditing {bearing} ({class_name})...")

    bearing_records = development_df[
        (development_df["bearing"] == bearing) &
        (
            development_df["operating_condition"]
            == CONDITION_TO_AUDIT
        )
    ].sort_values("measurement_number")

    assert len(bearing_records) == 20, (
        f"Expected 20 recordings for {bearing}, "
        f"found {len(bearing_records)}"
    )

    for _, record in bearing_records.iterrows():

        measurement = int(record["measurement_number"])

        mat_path = RAW_DIR / bearing / record["file"]

        mat = sio.loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )

        root_key = [
            key for key in mat
            if not key.startswith("__")
        ][0]

        root = mat[root_key]

        channels = {
            channel.Name: channel
            for channel in root.Y
        }

        vibration_channel = channels["vibration_1"]

        vibration = np.asarray(
            vibration_channel.Data,
            dtype=np.float64
        ).squeeze()

        time = np.asarray(
            root.X[vibration_channel.XIndex - 1].Data,
            dtype=np.float64
        ).squeeze()

        assert len(vibration) == len(time)

        dt_us = np.diff(time) * 1e6

        # Diagnostic threshold: 5% deviation from nominal 64 kHz.
        anomaly_mask = (
            np.abs(dt_us - NOMINAL_DT_US)
            > 0.05 * NOMINAL_DT_US
        )

        anomaly_indices = np.where(anomaly_mask)[0]

        # Identify consecutive anomaly groups.
        if len(anomaly_indices) > 0:

            breaks = np.where(
                np.diff(anomaly_indices) > 1
            )[0]

            starts = np.r_[0, breaks + 1]
            ends = np.r_[breaks, len(anomaly_indices) - 1]

            run_lengths = (
                ends - starts + 1
            )

            longest_run = int(run_lengths.max())

            anomaly_start_time = float(
                time[anomaly_indices[0]]
            )

        else:

            longest_run = 0
            anomaly_start_time = np.nan

        metadata = root.Description.Measurement

        audit_records.append({
            "bearing": bearing,
            "class_name": class_name,
            "measurement": measurement,
            "measurement_id": root.Info.MeasurementID,
            "file": record["file"],
            "samples": len(vibration),
            "sample_difference": len(vibration) - 256001,
            "duration_seconds": time[-1] - time[0],
            "mean_interval_us": np.mean(dt_us),
            "minimum_interval_us": np.min(dt_us),
            "maximum_interval_us": np.max(dt_us),
            "anomalous_intervals": int(anomaly_mask.sum()),
            "longest_anomaly_run": longest_run,
            "anomaly_start_time": anomaly_start_time,
            "metadata_length": metadata.Length
        })


cross_bearing_audit_df = pd.DataFrame(audit_records)


# ------------------------------------------------------------
# 1. Overall audit results
# ------------------------------------------------------------

print("\nCROSS-BEARING AUDIT RESULTS")

display(
    cross_bearing_audit_df.round(5)
)


# ------------------------------------------------------------
# 2. Summary by physical bearing
# ------------------------------------------------------------

bearing_quality_summary = (
    cross_bearing_audit_df
    .groupby(["bearing", "class_name"])
    .agg(
        total_recordings=("measurement", "count"),

        recordings_with_anomalies=(
            "anomalous_intervals",
            lambda x: (x > 0).sum()
        ),

        recordings_with_long_runs=(
            "longest_anomaly_run",
            lambda x: (x >= 50).sum()
        ),

        recordings_with_extra_samples=(
            "sample_difference",
            lambda x: (x > 1).sum()
        ),

        recordings_with_fewer_samples=(
            "sample_difference",
            lambda x: (x < -1).sum()
        ),

        maximum_anomalous_intervals=(
            "anomalous_intervals",
            "max"
        ),

        maximum_anomaly_run=(
            "longest_anomaly_run",
            "max"
        ),

        minimum_samples=("samples", "min"),
        maximum_samples=("samples", "max")
    )
    .reset_index()
)

print("\nQUALITY SUMMARY BY BEARING")

display(
    bearing_quality_summary
)


# ------------------------------------------------------------
# 3. Display recordings with substantial anomalies
# ------------------------------------------------------------

problematic_recordings = cross_bearing_audit_df[
    (cross_bearing_audit_df["longest_anomaly_run"] >= 50) |
    (cross_bearing_audit_df["sample_difference"].abs() > 100)
]

print("\nRECORDINGS REQUIRING FURTHER INVESTIGATION")

display(
    problematic_recordings[
        [
            "bearing",
            "class_name",
            "measurement",
            "samples",
            "sample_difference",
            "anomalous_intervals",
            "longest_anomaly_run",
            "anomaly_start_time",
            "minimum_interval_us",
            "maximum_interval_us"
        ]
    ].round(5)
)


print("\nCROSS-BEARING AUDIT COMPLETED.")

In [ ]:
# ============================================================
# STEP 9A-5I: AUDIT ALL FOUR OPERATING CONDITIONS
# ============================================================

import scipy.io as sio
import numpy as np
import pandas as pd

NOMINAL_FS = 64000.0
NOMINAL_DT_US = 1e6 / NOMINAL_FS

BEARINGS_TO_AUDIT = {
    "K001": "Healthy",
    "KA01": "Outer",
    "KI01": "Inner"
}

audit_all_conditions = []


# ------------------------------------------------------------
# 1. Loop through bearings and operating conditions
# ------------------------------------------------------------

for bearing, class_name in BEARINGS_TO_AUDIT.items():

    for condition in sorted(
        development_df["operating_condition"].unique()
    ):

        records = development_df[
            (development_df["bearing"] == bearing) &
            (development_df["operating_condition"] == condition)
        ].sort_values("measurement_number")

        print(
            f"Inspecting {bearing} | {condition} "
            f"| Recordings: {len(records)}"
        )

        assert len(records) == 20, (
            f"Expected 20 recordings for "
            f"{bearing}, {condition}; found {len(records)}"
        )

        for _, record in records.iterrows():

            measurement = int(record["measurement_number"])

            file_path = RAW_DIR / bearing / record["file"]

            mat = sio.loadmat(
                file_path,
                squeeze_me=True,
                struct_as_record=False
            )

            root_key = [
                key for key in mat
                if not key.startswith("__")
            ][0]

            root = mat[root_key]

            channels = {
                channel.Name: channel
                for channel in root.Y
            }

            vibration_channel = channels["vibration_1"]

            vibration = np.asarray(
                vibration_channel.Data,
                dtype=np.float64
            ).squeeze()

            time = np.asarray(
                root.X[vibration_channel.XIndex - 1].Data,
                dtype=np.float64
            ).squeeze()

            assert len(vibration) == len(time)

            dt_us = np.diff(time) * 1e6

            # Flag intervals deviating more than 5%
            # from the nominal 64 kHz interval.
            anomaly_mask = (
                np.abs(dt_us - NOMINAL_DT_US)
                > 0.05 * NOMINAL_DT_US
            )

            anomaly_indices = np.where(anomaly_mask)[0]

            # Find consecutive anomaly groups.
            if len(anomaly_indices) > 0:

                breaks = np.where(
                    np.diff(anomaly_indices) > 1
                )[0]

                starts = np.r_[0, breaks + 1]

                ends = np.r_[
                    breaks,
                    len(anomaly_indices) - 1
                ]

                longest_run = int(
                    np.max(ends - starts + 1)
                )

                anomaly_start_time = float(
                    time[anomaly_indices[0]]
                )

            else:

                longest_run = 0
                anomaly_start_time = np.nan

            metadata = root.Description.Measurement

            audit_all_conditions.append({
                "bearing": bearing,
                "class_name": class_name,
                "operating_condition": condition,
                "measurement": measurement,
                "file": record["file"],
                "measurement_id": root.Info.MeasurementID,
                "samples": len(vibration),
                "sample_difference": len(vibration) - 256001,
                "duration_seconds": time[-1] - time[0],
                "mean_interval_us": np.mean(dt_us),
                "minimum_interval_us": np.min(dt_us),
                "maximum_interval_us": np.max(dt_us),
                "anomalous_intervals": int(anomaly_mask.sum()),
                "longest_anomaly_run": longest_run,
                "anomaly_start_time": anomaly_start_time,
                "metadata_length": metadata.Length
            })


# ------------------------------------------------------------
# 2. Create complete audit DataFrame
# ------------------------------------------------------------

cross_condition_audit_df = pd.DataFrame(
    audit_all_conditions
)

print("\nTOTAL RECORDINGS INSPECTED:")
print(len(cross_condition_audit_df))


# ------------------------------------------------------------
# 3. Summary by bearing and operating condition
# ------------------------------------------------------------

condition_quality_summary = (
    cross_condition_audit_df
    .groupby([
        "bearing",
        "class_name",
        "operating_condition"
    ])
    .agg(
        total_recordings=("measurement", "count"),

        recordings_with_anomalies=(
            "anomalous_intervals",
            lambda x: (x > 0).sum()
        ),

        recordings_with_long_runs=(
            "longest_anomaly_run",
            lambda x: (x >= 50).sum()
        ),

        recordings_with_nonstandard_samples=(
            "sample_difference",
            lambda x: (x.abs() > 1).sum()
        ),

        maximum_anomalous_intervals=(
            "anomalous_intervals",
            "max"
        ),

        maximum_anomaly_run=(
            "longest_anomaly_run",
            "max"
        ),

        minimum_samples=("samples", "min"),
        maximum_samples=("samples", "max")
    )
    .reset_index()
)

print("\nQUALITY SUMMARY BY OPERATING CONDITION")

display(
    condition_quality_summary
)


# ------------------------------------------------------------
# 4. Overall summary by bearing
# ------------------------------------------------------------

bearing_summary = (
    cross_condition_audit_df
    .groupby(["bearing", "class_name"])
    .agg(
        total_recordings=("measurement", "count"),

        recordings_with_anomalies=(
            "anomalous_intervals",
            lambda x: (x > 0).sum()
        ),

        recordings_with_long_runs=(
            "longest_anomaly_run",
            lambda x: (x >= 50).sum()
        ),

        recordings_with_nonstandard_samples=(
            "sample_difference",
            lambda x: (x.abs() > 1).sum()
        )
    )
    .reset_index()
)

print("\nOVERALL BEARING QUALITY SUMMARY")

display(
    bearing_summary
)


# ------------------------------------------------------------
# 5. Largest timestamp anomaly recordings
# ------------------------------------------------------------

largest_anomalies = (
    cross_condition_audit_df
    .sort_values(
        "anomalous_intervals",
        ascending=False
    )
    .head(20)
)

print("\n20 RECORDINGS WITH THE MOST TIMESTAMP ANOMALIES")

display(
    largest_anomalies[
        [
            "bearing",
            "class_name",
            "operating_condition",
            "measurement",
            "samples",
            "sample_difference",
            "anomalous_intervals",
            "longest_anomaly_run",
            "anomaly_start_time",
            "minimum_interval_us",
            "maximum_interval_us"
        ]
    ].round(5)
)


print("\nCROSS-CONDITION AUDIT COMPLETED.")

In [ ]:
# ============================================================
# STEP 9A-5J
# INVESTIGATE A SEVERELY AFFECTED RECORDING
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.io import loadmat


# ------------------------------------------------------------
# 1. SELECT THE RECORDING
# ------------------------------------------------------------

SELECTED_BEARING = "KA01"
SELECTED_CONDITION = "N15_M07_F04"
MEASUREMENT_NUMBER = 16

NOMINAL_FS = 64000
NOMINAL_DT = 1 / NOMINAL_FS

ANOMALY_TOLERANCE = 0.05       # 5% deviation from nominal interval
CONTEXT_SECONDS = 0.05         # 50 ms before and after anomaly


print("=" * 70)
print("STEP 9A-5J: SEVERELY AFFECTED RECORDING INVESTIGATION")
print("=" * 70)

print(f"Bearing              : {SELECTED_BEARING}")
print(f"Operating condition  : {SELECTED_CONDITION}")
print(f"Measurement number   : {MEASUREMENT_NUMBER}")
print(f"Nominal sampling rate: {NOMINAL_FS} Hz")
print(f"Nominal interval     : {NOMINAL_DT * 1e6:.6f} microseconds")


# ------------------------------------------------------------
# 2. IDENTIFY THE RECORDING FROM DEVELOPMENT METADATA
# ------------------------------------------------------------

selected_rows = development_df[
    (development_df["bearing"] == SELECTED_BEARING)
    & (development_df["operating_condition"] == SELECTED_CONDITION)
    & (
        development_df["measurement_number"].astype(int)
        == MEASUREMENT_NUMBER
    )
]

if selected_rows.empty:
    raise ValueError(
        "The selected recording was not found in development_df. "
        "Check the bearing, operating condition and measurement number."
    )

selected_row = selected_rows.iloc[0]

print("\nRecording metadata:")
print(selected_row)


# ------------------------------------------------------------
# 3. LOCATE THE MAT FILE
# ------------------------------------------------------------

filename = (
    f"{SELECTED_CONDITION}_"
    f"{SELECTED_BEARING}_"
    f"{MEASUREMENT_NUMBER}.mat"
)

mat_path = Path(RAW_DIR) / SELECTED_BEARING / filename

# Fallback: search recursively if the expected path does not exist.
if not mat_path.exists():

    matching_files = list(Path(RAW_DIR).rglob(filename))

    if not matching_files:
        raise FileNotFoundError(
            f"Could not locate {filename} inside {RAW_DIR}"
        )

    mat_path = matching_files[0]


print("\nMAT file:")
print(mat_path)


# ------------------------------------------------------------
# 4. LOAD THE MATLAB STRUCTURE
# ------------------------------------------------------------

mat_contents = loadmat(
    mat_path,
    squeeze_me=True,
    struct_as_record=False
)

mat_key = mat_path.stem

if mat_key not in mat_contents:
    raise KeyError(
        f"MAT variable '{mat_key}' was not found in the file."
    )

data_object = mat_contents[mat_key]


# ------------------------------------------------------------
# 5. HELPER FUNCTIONS FOR MATLAB STRUCTURES
# ------------------------------------------------------------

def get_mat_field(obj, field_name):
    """
    Read a field from a MATLAB structure.
    Supports scipy mat_struct objects and dictionaries.
    """

    if isinstance(obj, dict):
        return obj[field_name]

    if hasattr(obj, field_name):
        return getattr(obj, field_name)

    raise AttributeError(
        f"MATLAB structure does not contain field: {field_name}"
    )


def get_struct_items(value):
    """
    Convert MATLAB structure arrays into a Python list.
    """

    if isinstance(value, np.ndarray):
        return list(value.reshape(-1))

    return [value]


def mat_value_to_string(value):
    """
    Convert MATLAB string/character-array values into Python strings.
    """

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    if isinstance(value, np.ndarray):

        value = value.squeeze()

        if value.ndim == 0:
            value = value.item()

        elif value.dtype.kind in ("U", "S"):
            value = "".join(map(str, value.reshape(-1)))

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    return str(value).strip()


# ------------------------------------------------------------
# 6. ACCESS X AND Y CHANNEL STRUCTURES
# ------------------------------------------------------------

x_items = get_struct_items(
    get_mat_field(data_object, "X")
)

y_items = get_struct_items(
    get_mat_field(data_object, "Y")
)


print("\nNumber of X time-axis entries:", len(x_items))
print("Number of Y signal channels  :", len(y_items))


# ------------------------------------------------------------
# 7. EXTRACT A CHANNEL AND ITS TIMESTAMPS
# ------------------------------------------------------------

def extract_channel_with_time(channel_name):

    selected_channel = None

    for channel in y_items:

        current_name = mat_value_to_string(
            get_mat_field(channel, "Name")
        ).lower()

        if current_name == channel_name.lower():
            selected_channel = channel
            break

    if selected_channel is None:
        raise ValueError(
            f"Channel '{channel_name}' was not found in the MAT file."
        )

    # Extract signal values
    signal = np.asarray(
        get_mat_field(selected_channel, "Data"),
        dtype=float
    ).reshape(-1)

    # XIndex is one-based in the MATLAB structure.
    # Python uses zero-based indexing.
    x_index = int(
        np.asarray(
            get_mat_field(selected_channel, "XIndex")
        ).squeeze()
    )

    python_x_index = x_index - 1

    if python_x_index < 0 or python_x_index >= len(x_items):
        raise IndexError(
            f"Invalid XIndex={x_index} for channel '{channel_name}'."
        )

    time_axis = x_items[python_x_index]

    timestamps = np.asarray(
        get_mat_field(time_axis, "Data"),
        dtype=float
    ).reshape(-1)

    if len(signal) != len(timestamps):
        raise ValueError(
            f"Length mismatch for {channel_name}: "
            f"{len(signal)} signal samples and "
            f"{len(timestamps)} timestamps."
        )

    if not np.all(np.isfinite(signal)):
        raise ValueError(
            f"Non-finite signal values found in {channel_name}."
        )

    if not np.all(np.isfinite(timestamps)):
        raise ValueError(
            f"Non-finite timestamps found in {channel_name}."
        )

    return signal, timestamps


# ------------------------------------------------------------
# 8. EXTRACT VIBRATION AND CURRENT SIGNALS
# ------------------------------------------------------------

vibration_signal, vibration_time = extract_channel_with_time(
    "vibration_1"
)

current_signal, current_time = extract_channel_with_time(
    "phase_current_1"
)


print("\nSignal information:")
print("-" * 70)

print(
    f"Vibration samples     : {len(vibration_signal):,}"
)

print(
    f"Vibration timestamps : {len(vibration_time):,}"
)

print(
    f"Current samples       : {len(current_signal):,}"
)

print(
    f"Current timestamps    : {len(current_time):,}"
)

print(
    f"Vibration duration    : "
    f"{vibration_time[-1] - vibration_time[0]:.6f} seconds"
)


# ------------------------------------------------------------
# 9. CALCULATE TIMESTAMP INTERVALS
# ------------------------------------------------------------

dt = np.diff(vibration_time)

if len(dt) == 0:
    raise ValueError(
        "The vibration recording does not contain enough timestamps."
    )

if np.any(dt <= 0):
    raise ValueError(
        "Non-increasing timestamps detected. "
        "Investigate the timestamp sequence before continuing."
    )


# An interval is anomalous when it differs from nominal dt
# by more than 5%.

tolerance = NOMINAL_DT * ANOMALY_TOLERANCE

anomaly_mask = (
    np.abs(dt - NOMINAL_DT) > tolerance
)

anomaly_indices = np.flatnonzero(anomaly_mask)


print("\nTimestamp interval investigation:")
print("-" * 70)

print(f"Total intervals          : {len(dt):,}")
print(f"Anomalous intervals      : {len(anomaly_indices):,}")
print(
    f"Percentage anomalous    : "
    f"{np.mean(anomaly_mask) * 100:.4f}%"
)


# ------------------------------------------------------------
# 10. IDENTIFY CONTIGUOUS ANOMALY RUNS
# ------------------------------------------------------------

if len(anomaly_indices) == 0:
    raise ValueError(
        "No anomalous intervals were detected in this recording. "
        "Check the selected recording and anomaly threshold."
    )


# Split the anomaly indices wherever there is a gap.
split_positions = (
    np.where(np.diff(anomaly_indices) > 1)[0] + 1
)

anomaly_groups = np.split(
    anomaly_indices,
    split_positions
)


run_records = []

for group in anomaly_groups:

    if len(group) == 0:
        continue

    start_idx = int(group[0])
    end_idx = int(group[-1])

    run_count = len(group)

    start_time = vibration_time[start_idx]
    end_time = vibration_time[end_idx + 1]

    run_duration = end_time - start_time

    run_records.append({
        "start_interval_index": start_idx,
        "end_interval_index": end_idx,
        "interval_count": run_count,
        "start_time_s": start_time,
        "end_time_s": end_time,
        "actual_duration_s": run_duration,
        "nominal_duration_s": run_count * NOMINAL_DT,
        "mean_interval_us": np.mean(dt[group]) * 1e6,
        "median_interval_us": np.median(dt[group]) * 1e6,
        "std_interval_us": np.std(dt[group]) * 1e6,
        "minimum_interval_us": np.min(dt[group]) * 1e6,
        "maximum_interval_us": np.max(dt[group]) * 1e6,
        "effective_sampling_rate": (
            1 / np.mean(dt[group])
        )
    })


runs_df = pd.DataFrame(run_records)

runs_df = runs_df.sort_values(
    "interval_count",
    ascending=False
).reset_index(drop=True)


print("\nLargest contiguous anomaly runs:")
print("-" * 70)

print(
    runs_df.head(10).to_string(index=False)
)


# ------------------------------------------------------------
# 11. SELECT THE LONGEST ANOMALY RUN
# ------------------------------------------------------------

longest_run = runs_df.iloc[0]

start_interval_idx = int(
    longest_run["start_interval_index"]
)

end_interval_idx = int(
    longest_run["end_interval_index"]
)

anomaly_start_time = float(
    longest_run["start_time_s"]
)

anomaly_end_time = float(
    longest_run["end_time_s"]
)

anomaly_interval_count = int(
    longest_run["interval_count"]
)


print("\nLONGEST ANOMALY RUN")
print("=" * 70)

print(
    f"Start interval index    : {start_interval_idx}"
)

print(
    f"End interval index      : {end_interval_idx}"
)

print(
    f"Number of intervals     : {anomaly_interval_count:,}"
)

print(
    f"Start time              : {anomaly_start_time:.6f} s"
)

print(
    f"End time                : {anomaly_end_time:.6f} s"
)

print(
    f"Actual duration         : "
    f"{longest_run['actual_duration_s']:.6f} s"
)

print(
    f"Expected duration       : "
    f"{longest_run['nominal_duration_s']:.6f} s"
)

print(
    f"Mean interval           : "
    f"{longest_run['mean_interval_us']:.6f} microseconds"
)

print(
    f"Median interval         : "
    f"{longest_run['median_interval_us']:.6f} microseconds"
)

print(
    f"Minimum interval        : "
    f"{longest_run['minimum_interval_us']:.6f} microseconds"
)

print(
    f"Maximum interval        : "
    f"{longest_run['maximum_interval_us']:.6f} microseconds"
)

print(
    f"Effective sampling rate : "
    f"{longest_run['effective_sampling_rate']:.2f} Hz"
)


# ------------------------------------------------------------
# 12. DIVIDE RECORDING INTO BEFORE, DURING AND AFTER SECTIONS
# ------------------------------------------------------------

# These are interval arrays, not signal arrays.
# An interval at index i connects timestamp[i] and timestamp[i+1].

section_intervals = {

    "Before anomaly": dt[:start_interval_idx],

    "During anomaly": dt[
        start_interval_idx:end_interval_idx + 1
    ],

    "After anomaly": dt[
        end_interval_idx + 1:
    ]
}


section_summary = []
valid_section_intervals = {}


print("\nINTERVAL STATISTICS BY SECTION")
print("=" * 70)


for section_name, intervals in section_intervals.items():

    intervals = np.asarray(intervals)

    # IMPORTANT FIX:
    # Skip empty sections before applying np.min()/np.max().
    if intervals.size == 0:

        print(
            f"\nSkipping {section_name}: "
            "it contains no timestamp intervals."
        )

        continue

    valid_section_intervals[section_name] = intervals

    section_summary.append({

        "section": section_name,

        "interval_count": len(intervals),

        "mean_interval_us": np.mean(intervals) * 1e6,

        "median_interval_us": np.median(intervals) * 1e6,

        "std_interval_us": np.std(intervals) * 1e6,

        "minimum_interval_us": np.min(intervals) * 1e6,

        "maximum_interval_us": np.max(intervals) * 1e6,

        "effective_sampling_rate": (
            1 / np.mean(intervals)
        )
    })


if section_summary:

    section_summary_df = pd.DataFrame(section_summary)

    print(
        section_summary_df.to_string(index=False)
    )

else:

    section_summary_df = pd.DataFrame()

    print("No non-empty sections were available.")


# ------------------------------------------------------------
# 13. COMPARE SIGNAL STATISTICS BEFORE, DURING AND AFTER
# ------------------------------------------------------------

def calculate_signal_statistics(signal):

    signal = np.asarray(signal, dtype=float)

    if signal.size == 0:
        return {
            "sample_count": 0,
            "mean": np.nan,
            "std": np.nan,
            "rms": np.nan,
            "minimum": np.nan,
            "maximum": np.nan
        }

    return {

        "sample_count": len(signal),

        "mean": np.mean(signal),

        "std": np.std(signal),

        "rms": np.sqrt(np.mean(signal ** 2)),

        "minimum": np.min(signal),

        "maximum": np.max(signal)
    }


# Use time-based boundaries so the signal sections are selected
# independently using each channel's own timestamp vector.

signal_section_masks = {

    "Before anomaly": (
        vibration_time < anomaly_start_time
    ),

    "During anomaly": (
        (vibration_time >= anomaly_start_time)
        & (vibration_time <= anomaly_end_time)
    ),

    "After anomaly": (
        vibration_time > anomaly_end_time
    )
}


current_section_masks = {

    "Before anomaly": (
        current_time < anomaly_start_time
    ),

    "During anomaly": (
        (current_time >= anomaly_start_time)
        & (current_time <= anomaly_end_time)
    ),

    "After anomaly": (
        current_time > anomaly_end_time
    )
}


signal_statistics = []


for section_name in signal_section_masks:

    # Vibration statistics
    vibration_mask = signal_section_masks[section_name]

    vibration_stats = calculate_signal_statistics(
        vibration_signal[vibration_mask]
    )

    signal_statistics.append({

        "channel": "Vibration",

        "section": section_name,

        **vibration_stats
    })


    # Phase current statistics
    current_mask = current_section_masks[section_name]

    current_stats = calculate_signal_statistics(
        current_signal[current_mask]
    )

    signal_statistics.append({

        "channel": "Phase current 1",

        "section": section_name,

        **current_stats
    })


signal_statistics_df = pd.DataFrame(
    signal_statistics
)


print("\nSIGNAL STATISTICS BY SECTION")
print("=" * 70)

print(
    signal_statistics_df.to_string(index=False)
)


# ------------------------------------------------------------
# 14. PLOT 1: TIMESTAMP INTERVALS OVER THE FULL RECORDING
# ------------------------------------------------------------

midpoint_times = (
    vibration_time[:-1] + vibration_time[1:]
) / 2

plt.figure(figsize=(14, 5))

plt.plot(
    midpoint_times,
    dt * 1e6,
    linewidth=0.6,
    label="Timestamp interval"
)

# Highlight anomalous intervals
plt.scatter(
    midpoint_times[anomaly_mask],
    dt[anomaly_mask] * 1e6,
    s=5,
    label="Anomalous intervals"
)

plt.axhline(
    NOMINAL_DT * 1e6,
    linestyle="--",
    label="Nominal 64 kHz interval"
)

plt.axvspan(
    anomaly_start_time,
    anomaly_end_time,
    alpha=0.2,
    label="Longest anomaly run"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Timestamp interval (microseconds)")

plt.title(
    "Vibration Timestamp Intervals - Full Recording"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 15. PLOT 2: INTERVAL DISTRIBUTION BY SECTION
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

histogram_created = False

for section_name, intervals in valid_section_intervals.items():

    if intervals.size == 0:
        continue

    plt.hist(
        intervals * 1e6,
        bins=50,
        alpha=0.5,
        label=section_name
    )

    histogram_created = True


if histogram_created:

    plt.axvline(
        NOMINAL_DT * 1e6,
        linestyle="--",
        label="Nominal 64 kHz interval"
    )

    plt.xlabel("Timestamp interval (microseconds)")
    plt.ylabel("Frequency")

    plt.title(
        "Timestamp Interval Distribution Before, During and After Anomaly"
    )

    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

else:

    print(
        "\nSkipping interval histogram: "
        "no non-empty sections are available."
    )


# ------------------------------------------------------------
# 16. PLOT 3: VIBRATION AND CURRENT AROUND THE ANOMALY
# ------------------------------------------------------------

plot_start_time = max(
    vibration_time[0],
    anomaly_start_time - CONTEXT_SECONDS
)

plot_end_time = min(
    vibration_time[-1],
    anomaly_end_time + CONTEXT_SECONDS
)


vibration_plot_mask = (
    (vibration_time >= plot_start_time)
    & (vibration_time <= plot_end_time)
)

current_plot_mask = (
    (current_time >= plot_start_time)
    & (current_time <= plot_end_time)
)


fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 8),
    sharex=True
)


# Vibration
axes[0].plot(
    vibration_time[vibration_plot_mask],
    vibration_signal[vibration_plot_mask],
    linewidth=0.7
)

axes[0].axvspan(
    anomaly_start_time,
    anomaly_end_time,
    alpha=0.2,
    label="Longest anomaly run"
)

axes[0].set_ylabel("Vibration amplitude")
axes[0].set_title("Vibration Signal Around Anomalous Region")
axes[0].legend()
axes[0].grid(True, alpha=0.3)


# Phase current
axes[1].plot(
    current_time[current_plot_mask],
    current_signal[current_plot_mask],
    linewidth=0.7
)

axes[1].axvspan(
    anomaly_start_time,
    anomaly_end_time,
    alpha=0.2,
    label="Longest anomaly run"
)

axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Current amplitude")
axes[1].set_title("Phase Current 1 Around Anomalous Region")
axes[1].legend()
axes[1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 17. FINAL DIAGNOSTIC SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 9A-5J COMPLETED")
print("=" * 70)

print(f"Bearing: {SELECTED_BEARING}")
print(f"Condition: {SELECTED_CONDITION}")
print(f"Measurement: {MEASUREMENT_NUMBER}")

print(
    f"\nTotal vibration samples: {len(vibration_signal):,}"
)

print(
    f"Total anomalous intervals: {len(anomaly_indices):,}"
)

print(
    f"Number of contiguous anomaly runs: {len(runs_df):,}"
)

print(
    f"Longest anomaly run: {anomaly_interval_count:,} intervals"
)

print(
    f"Anomaly time range: "
    f"{anomaly_start_time:.6f} s to "
    f"{anomaly_end_time:.6f} s"
)

print(
    f"Anomaly duration: "
    f"{longest_run['actual_duration_s']:.6f} seconds"
)

print("\nNo samples have been removed, resampled or modified.")
print("This step is diagnostic only.")

In [ ]:
# ============================================================
# STEP 9A-5K
# INSPECT ACQUISITION AND SAMPLING METADATA
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat


print("=" * 75)
print("STEP 9A-5K: MAT FILE ACQUISITION METADATA")
print("=" * 75)


# ------------------------------------------------------------
# 1. RELOAD THE EXACT RECORDING
# ------------------------------------------------------------

selected_mat_path = (
    Path(RAW_DIR)
    / "KA01"
    / "N15_M07_F04_KA01_16.mat"
)

if not selected_mat_path.exists():
    raise FileNotFoundError(selected_mat_path)

mat_contents = loadmat(
    selected_mat_path,
    squeeze_me=True,
    struct_as_record=False
)

data_object_metadata = mat_contents[
    selected_mat_path.stem
]

print("\nFile:", selected_mat_path)


# ------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------

def read_field(obj, field_name):

    if isinstance(obj, dict):
        return obj.get(field_name, None)

    if hasattr(obj, field_name):
        return getattr(obj, field_name)

    return None


def get_field_names(obj):

    if isinstance(obj, dict):
        return list(obj.keys())

    return list(
        getattr(obj, "_fieldnames", [])
    )


def format_metadata_value(value):

    if value is None:
        return "Not available"

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace"
        )

    if isinstance(value, np.ndarray):

        if value.size <= 10:
            return np.array2string(
                value,
                precision=10
            )

        return (
            f"Array: shape={value.shape}, "
            f"dtype={value.dtype}"
        )

    if isinstance(value, np.generic):
        return value.item()

    if hasattr(value, "_fieldnames"):
        return (
            "MATLAB structure with fields: "
            + ", ".join(value._fieldnames)
        )

    return value


# ------------------------------------------------------------
# 3. ACCESS X AND Y STRUCTURES
# ------------------------------------------------------------

x_items = np.asarray(
    read_field(data_object_metadata, "X"),
    dtype=object
).reshape(-1)

y_items = np.asarray(
    read_field(data_object_metadata, "Y"),
    dtype=object
).reshape(-1)


print("\nNumber of X entries:", len(x_items))
print("Number of Y entries:", len(y_items))


# ------------------------------------------------------------
# 4. INSPECT ALL X TIME AXES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("X TIME-AXIS METADATA")
print("=" * 75)

x_metadata_records = []

for i, x_axis in enumerate(x_items):

    print(f"\nX[{i}]")
    print("-" * 50)

    field_names = get_field_names(x_axis)

    print("Available fields:", field_names)

    record = {
        "X_index_python": i,
        "X_index_matlab": i + 1
    }

    for field in [
        "Name",
        "Raster",
        "Unit",
        "Type",
        "Device",
        "DownSampling"
    ]:

        value = read_field(
            x_axis,
            field
        )

        formatted_value = format_metadata_value(
            value
        )

        record[field] = formatted_value

        print(f"{field}: {formatted_value}")

    time_data = read_field(
        x_axis,
        "Data"
    )

    if time_data is not None:

        time_data = np.asarray(
            time_data
        ).reshape(-1)

        print("Data length:", len(time_data))

        if len(time_data) > 1:

            intervals = np.diff(
                time_data.astype(float)
            )

            print(
                "Mean timestamp interval (us):",
                np.mean(intervals) * 1e6
            )

            print(
                "Median timestamp interval (us):",
                np.median(intervals) * 1e6
            )

            print(
                "Minimum timestamp interval (us):",
                np.min(intervals) * 1e6
            )

            print(
                "Maximum timestamp interval (us):",
                np.max(intervals) * 1e6
            )

    x_metadata_records.append(record)


# ------------------------------------------------------------
# 5. INSPECT VIBRATION AND CURRENT CHANNEL METADATA
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("Y CHANNEL METADATA")
print("=" * 75)

selected_channel_names = [
    "vibration_1",
    "phase_current_1",
    "phase_current_2"
]

y_metadata_records = []

for channel in y_items:

    channel_name = read_field(
        channel,
        "Name"
    )

    if isinstance(channel_name, np.ndarray):
        channel_name = channel_name.squeeze()

    channel_name = str(channel_name).strip()

    if channel_name.lower() not in selected_channel_names:
        continue

    print(f"\nCHANNEL: {channel_name}")
    print("-" * 60)

    print(
        "Available fields:",
        get_field_names(channel)
    )

    record = {
        "channel": channel_name
    }

    for field in [
        "Type",
        "Unit",
        "Raster",
        "Device",
        "XIndex",
        "DownSampling"
    ]:

        value = read_field(
            channel,
            field
        )

        formatted_value = format_metadata_value(
            value
        )

        record[field] = formatted_value

        print(
            f"{field}: {formatted_value}"
        )

    signal_data = read_field(
        channel,
        "Data"
    )

    if signal_data is not None:

        signal_data = np.asarray(
            signal_data
        ).reshape(-1)

        record["sample_count"] = len(
            signal_data
        )

        print(
            "Data length:",
            len(signal_data)
        )

    y_metadata_records.append(record)


# ------------------------------------------------------------
# 6. CREATE SUMMARY TABLES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("X AXIS SUMMARY TABLE")
print("=" * 75)

x_metadata_df = pd.DataFrame(
    x_metadata_records
)

display(x_metadata_df)


print("\n" + "=" * 75)
print("Y CHANNEL SUMMARY TABLE")
print("=" * 75)

y_metadata_df = pd.DataFrame(
    y_metadata_records
)

display(y_metadata_df)


# ------------------------------------------------------------
# 7. CHECK CHANNEL XINDEX REFERENCES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CHANNEL-TO-TIME-AXIS MAPPING")
print("=" * 75)

for channel in y_items:

    channel_name = read_field(
        channel,
        "Name"
    )

    if isinstance(channel_name, np.ndarray):
        channel_name = channel_name.squeeze()

    channel_name = str(channel_name).strip()

    if channel_name.lower() not in selected_channel_names:
        continue

    x_index = read_field(
        channel,
        "XIndex"
    )

    if x_index is not None:

        x_index = int(
            np.asarray(x_index).squeeze()
        )

        print(
            f"{channel_name}: "
            f"XIndex={x_index} (MATLAB), "
            f"X[{x_index - 1}] (Python)"
        )

print("\nSTEP 9A-5K COMPLETED")

In [ ]:
# ============================================================
# STEP 9A-5L
# COMPARE SIGNALS IN THE SAMPLE-INDEX DOMAIN
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import welch


print("=" * 75)
print("STEP 9A-5L: SAMPLE-INDEX DOMAIN COMPARISON")
print("=" * 75)


# ------------------------------------------------------------
# 1. VERIFY PREVIOUS STEP VARIABLES
# ------------------------------------------------------------

required_variables = [
    "vibration_signal",
    "current_signal",
    "start_interval_idx",
    "end_interval_idx"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required variables are missing: "
        + ", ".join(missing_variables)
        + ". Run Step 9A-5J again before this step."
    )


# ------------------------------------------------------------
# 2. SELECT EQUAL-SIZED SAMPLE WINDOWS
# ------------------------------------------------------------

# The anomaly run starts and ends at these interval indices.
# An interval connects two consecutive samples.
#
# Therefore, the corresponding anomalous sample section is:
# start_interval_idx : end_interval_idx + 2

anomaly_sample_start = start_interval_idx
anomaly_sample_end = end_interval_idx + 2

anomaly_sample_count = (
    anomaly_sample_end - anomaly_sample_start
)

# Select an equally sized section immediately before the anomaly.
before_sample_end = anomaly_sample_start

before_sample_start = (
    before_sample_end - anomaly_sample_count
)

if before_sample_start < 0:
    raise ValueError(
        "Not enough samples before the anomaly."
    )


# Extract signal sections
current_before = current_signal[
    before_sample_start:before_sample_end
]

current_during = current_signal[
    anomaly_sample_start:anomaly_sample_end
]

vibration_before = vibration_signal[
    before_sample_start:before_sample_end
]

vibration_during = vibration_signal[
    anomaly_sample_start:anomaly_sample_end
]


# Verify equal lengths
assert len(current_before) == len(current_during)
assert len(vibration_before) == len(vibration_during)

print("\nSelected sample windows:")
print("-" * 65)

print(
    f"Before anomaly: samples "
    f"{before_sample_start:,} to "
    f"{before_sample_end - 1:,}"
)

print(
    f"During anomaly: samples "
    f"{anomaly_sample_start:,} to "
    f"{anomaly_sample_end - 1:,}"
)

print(
    f"Samples in each window: {anomaly_sample_count:,}"
)

print(
    f"Expected duration at 64 kHz: "
    f"{anomaly_sample_count / 64000:.6f} seconds"
)


# ------------------------------------------------------------
# 3. CALCULATE FREQUENCY CHARACTERISTICS
# ------------------------------------------------------------

# We use 64 kHz as a reference sampling rate.
# This is an assumption for comparison, NOT proof that the
# anomalous section was actually acquired at 64 kHz.

REFERENCE_FS = 64000


def analyse_frequency(signal, channel_name, section_name):

    signal = np.asarray(signal, dtype=float)

    signal = signal - np.mean(signal)

    frequencies, power = welch(
        signal,
        fs=REFERENCE_FS,
        window="hann",
        nperseg=min(8192, len(signal)),
        detrend="constant"
    )

    # Restrict the frequency range to avoid selecting
    # DC or very high-frequency noise.

    if channel_name == "Phase current":
        frequency_range = (10, 1000)
    else:
        frequency_range = (10, 10000)

    frequency_mask = (
        (frequencies >= frequency_range[0])
        & (frequencies <= frequency_range[1])
    )

    if not np.any(frequency_mask):
        raise ValueError(
            f"No frequencies available for {channel_name}."
        )

    valid_frequencies = frequencies[frequency_mask]
    valid_power = power[frequency_mask]

    dominant_index = np.argmax(valid_power)

    dominant_frequency = valid_frequencies[
        dominant_index
    ]

    return {
        "channel": channel_name,
        "section": section_name,
        "sample_count": len(signal),
        "rms": np.sqrt(np.mean(signal ** 2)),
        "dominant_frequency_hz": dominant_frequency,
        "cycles_per_sample": (
            dominant_frequency / REFERENCE_FS
        )
    }


frequency_records = []


frequency_records.append(
    analyse_frequency(
        current_before,
        "Phase current",
        "Before anomaly"
    )
)

frequency_records.append(
    analyse_frequency(
        current_during,
        "Phase current",
        "During anomaly"
    )
)

frequency_records.append(
    analyse_frequency(
        vibration_before,
        "Vibration",
        "Before anomaly"
    )
)

frequency_records.append(
    analyse_frequency(
        vibration_during,
        "Vibration",
        "During anomaly"
    )
)


frequency_comparison_df = pd.DataFrame(
    frequency_records
)


print("\nFREQUENCY COMPARISON")
print("=" * 75)

print(
    frequency_comparison_df.to_string(index=False)
)


# ------------------------------------------------------------
# 4. COMPARE PHASE-CURRENT FREQUENCY DIRECTLY
# ------------------------------------------------------------

current_before_frequency = (
    frequency_comparison_df.loc[
        (frequency_comparison_df["channel"] == "Phase current")
        & (
            frequency_comparison_df["section"]
            == "Before anomaly"
        ),
        "dominant_frequency_hz"
    ].iloc[0]
)

current_during_frequency = (
    frequency_comparison_df.loc[
        (frequency_comparison_df["channel"] == "Phase current")
        & (
            frequency_comparison_df["section"]
            == "During anomaly"
        ),
        "dominant_frequency_hz"
    ].iloc[0]
)


frequency_change_percent = (
    abs(
        current_during_frequency
        - current_before_frequency
    )
    / current_before_frequency
) * 100


print("\nPHASE-CURRENT COMPARISON")
print("-" * 65)

print(
    f"Before anomaly: "
    f"{current_before_frequency:.3f} Hz"
)

print(
    f"During anomaly: "
    f"{current_during_frequency:.3f} Hz"
)

print(
    f"Percentage frequency difference: "
    f"{frequency_change_percent:.3f}%"
)


# ------------------------------------------------------------
# 5. PLOT PHASE CURRENT USING SAMPLE INDEX
# ------------------------------------------------------------

# Plot the first 4096 samples of each equal-sized section.
# The X-axis is sample position, NOT recorded timestamp.

PLOT_SAMPLES = min(
    4096,
    anomaly_sample_count
)

sample_positions = np.arange(PLOT_SAMPLES)


fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 8)
)


axes[0].plot(
    sample_positions,
    current_before[:PLOT_SAMPLES],
    linewidth=0.8
)

axes[0].set_title(
    "Phase Current Before Anomaly - Sample Index Domain"
)

axes[0].set_ylabel("Current amplitude")
axes[0].set_xlabel("Sample position within selected window")
axes[0].grid(True, alpha=0.3)


axes[1].plot(
    sample_positions,
    current_during[:PLOT_SAMPLES],
    linewidth=0.8
)

axes[1].set_title(
    "Phase Current During Anomaly - Sample Index Domain"
)

axes[1].set_ylabel("Current amplitude")
axes[1].set_xlabel("Sample position within selected window")
axes[1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 6. PLOT VIBRATION USING SAMPLE INDEX
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 8)
)


axes[0].plot(
    sample_positions,
    vibration_before[:PLOT_SAMPLES],
    linewidth=0.6
)

axes[0].set_title(
    "Vibration Before Anomaly - Sample Index Domain"
)

axes[0].set_ylabel("Vibration amplitude")
axes[0].set_xlabel("Sample position within selected window")
axes[0].grid(True, alpha=0.3)


axes[1].plot(
    sample_positions,
    vibration_during[:PLOT_SAMPLES],
    linewidth=0.6
)

axes[1].set_title(
    "Vibration During Anomaly - Sample Index Domain"
)

axes[1].set_ylabel("Vibration amplitude")
axes[1].set_xlabel("Sample position within selected window")
axes[1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9A-5L COMPLETED")
print("=" * 75)

print(
    "Both sections contain exactly the same number of samples."
)

print(
    "Frequency comparison uses sample values, "
    "with 64 kHz as a common reference."
)

print(
    "No timestamps or signal values were modified."
)

In [ ]:
# ============================================================
# STEP 9A-5M
# INVESTIGATE THE INDEPENDENT MOTOR-SPEED CHANNEL
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.io import loadmat


print("=" * 75)
print("STEP 9A-5M: INDEPENDENT MOTOR-SPEED INVESTIGATION")
print("=" * 75)


# ------------------------------------------------------------
# 1. LOAD THE SAME RECORDING
# ------------------------------------------------------------

selected_mat_path = (
    Path(RAW_DIR)
    / "KA01"
    / "N15_M07_F04_KA01_16.mat"
)

mat_contents = loadmat(
    selected_mat_path,
    squeeze_me=True,
    struct_as_record=False
)

data_object = mat_contents[
    selected_mat_path.stem
]


# ------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------

def get_field(obj, field_name):

    if isinstance(obj, dict):
        return obj.get(field_name, None)

    if hasattr(obj, field_name):
        return getattr(obj, field_name)

    return None


def get_items(value):

    if isinstance(value, np.ndarray):
        return list(value.reshape(-1))

    return [value]


def convert_matlab_string(value):

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace"
        ).strip()

    if isinstance(value, np.ndarray):

        value = np.squeeze(value)

        if value.dtype.kind in ("U", "S"):
            return "".join(
                map(str, value.reshape(-1))
            ).strip()

        if value.ndim == 0:
            value = value.item()

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace"
        ).strip()

    return str(value).strip()


# ------------------------------------------------------------
# 3. ACCESS MATLAB X AND Y STRUCTURES
# ------------------------------------------------------------

x_items = get_items(
    get_field(data_object, "X")
)

y_items = get_items(
    get_field(data_object, "Y")
)


# ------------------------------------------------------------
# 4. FIND THE SPEED CHANNEL
# ------------------------------------------------------------

speed_channel = None

for channel in y_items:

    channel_name = convert_matlab_string(
        get_field(channel, "Name")
    )

    if channel_name.lower() == "speed":
        speed_channel = channel
        break


if speed_channel is None:
    raise ValueError(
        "The speed channel was not found in the selected MAT file."
    )


# ------------------------------------------------------------
# 5. EXTRACT SPEED SIGNAL AND ITS OWN TIME AXIS
# ------------------------------------------------------------

speed_signal = np.asarray(
    get_field(speed_channel, "Data"),
    dtype=float
).reshape(-1)


speed_x_index = int(
    np.asarray(
        get_field(speed_channel, "XIndex")
    ).squeeze()
)


# MATLAB XIndex is one-based.
speed_time_axis = x_items[
    speed_x_index - 1
]


speed_timestamps = np.asarray(
    get_field(speed_time_axis, "Data"),
    dtype=float
).reshape(-1)


if len(speed_signal) != len(speed_timestamps):
    raise ValueError(
        "Speed signal and timestamp lengths do not match."
    )


if len(speed_signal) < 2:
    raise ValueError(
        "Insufficient speed samples."
    )


if np.any(np.diff(speed_timestamps) <= 0):
    raise ValueError(
        "Speed timestamps are not strictly increasing."
    )


print("\nSPEED CHANNEL METADATA")
print("-" * 65)

for field in [
    "Name",
    "Unit",
    "Raster",
    "Device",
    "XIndex",
    "DownSampling"
]:

    print(
        f"{field}:",
        convert_matlab_string(
            get_field(speed_channel, field)
        )
    )


print("\nSPEED TIME-AXIS INFORMATION")
print("-" * 65)

print("Number of samples:", len(speed_signal))

print(
    "Start time:",
    speed_timestamps[0]
)

print(
    "End time:",
    speed_timestamps[-1]
)

speed_dt = np.diff(speed_timestamps)

print(
    "Mean interval (microseconds):",
    np.mean(speed_dt) * 1e6
)

print(
    "Median interval (microseconds):",
    np.median(speed_dt) * 1e6
)

print(
    "Effective sampling rate (Hz):",
    1 / np.mean(speed_dt)
)


# ------------------------------------------------------------
# 6. DEFINE BEFORE AND DURING ANOMALY WINDOWS
# ------------------------------------------------------------

ANOMALY_START = 3.876035
ANOMALY_END = 3.999998

ANOMALY_DURATION = (
    ANOMALY_END - ANOMALY_START
)

# Select an equal-duration period immediately before
# the timestamp anomaly.

BEFORE_START = (
    ANOMALY_START - ANOMALY_DURATION
)

BEFORE_END = ANOMALY_START


before_mask = (
    (speed_timestamps >= BEFORE_START)
    & (speed_timestamps < BEFORE_END)
)

during_mask = (
    (speed_timestamps >= ANOMALY_START)
    & (speed_timestamps <= ANOMALY_END)
)


speed_before = speed_signal[before_mask]
speed_during = speed_signal[during_mask]


# ------------------------------------------------------------
# 7. CALCULATE SPEED STATISTICS
# ------------------------------------------------------------

def speed_statistics(values, section_name):

    values = np.asarray(values, dtype=float)

    if values.size == 0:

        return {
            "section": section_name,
            "sample_count": 0,
            "mean": np.nan,
            "median": np.nan,
            "std": np.nan,
            "minimum": np.nan,
            "maximum": np.nan
        }

    return {

        "section": section_name,

        "sample_count": len(values),

        "mean": np.mean(values),

        "median": np.median(values),

        "std": np.std(values),

        "minimum": np.min(values),

        "maximum": np.max(values)
    }


speed_comparison = pd.DataFrame([

    speed_statistics(
        speed_before,
        "Before anomaly"
    ),

    speed_statistics(
        speed_during,
        "During anomaly"
    )

])


print("\nMOTOR SPEED COMPARISON")
print("=" * 75)

print(speed_comparison.to_string(index=False))


# ------------------------------------------------------------
# 8. PLOT THE FULL SPEED SIGNAL
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

plt.plot(
    speed_timestamps,
    speed_signal,
    linewidth=0.8,
    label="Measured speed"
)

plt.axvspan(
    ANOMALY_START,
    ANOMALY_END,
    alpha=0.2,
    label="HostService timestamp anomaly"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Speed signal (stored units)")

plt.title(
    "Motor Speed - Full Recording"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 9. PLOT THE SPEED AROUND THE ANOMALY
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

plt.plot(
    speed_timestamps,
    speed_signal,
    linewidth=1,
    label="Measured speed"
)

plt.axvspan(
    ANOMALY_START,
    ANOMALY_END,
    alpha=0.2,
    label="HostService timestamp anomaly"
)

plt.xlim(
    BEFORE_START - 0.01,
    ANOMALY_END + 0.005
)

plt.xlabel("Time (seconds)")
plt.ylabel("Speed signal (stored units)")

plt.title(
    "Motor Speed Before and During HostService Timestamp Anomaly"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9A-5M COMPLETED")
print("=" * 75)

print(
    f"Before anomaly: "
    f"{BEFORE_START:.6f} to {BEFORE_END:.6f} seconds"
)

print(
    f"During anomaly: "
    f"{ANOMALY_START:.6f} to {ANOMALY_END:.6f} seconds"
)

print(
    f"Speed samples before: {len(speed_before)}"
)

print(
    f"Speed samples during: {len(speed_during)}"
)

print(
    "\nNo signal values or timestamps were modified."
)

In [ ]:
# ============================================================
# STEP 9A-5N
# COMPARE FREQUENCY USING ACTUAL TIMESTAMPS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.io import loadmat


print("=" * 75)
print("STEP 9A-5N: TIMESTAMP-BASED FREQUENCY COMPARISON")
print("=" * 75)


# ------------------------------------------------------------
# 1. LOAD THE SAME RECORDING
# ------------------------------------------------------------

selected_mat_path = (
    Path(RAW_DIR)
    / "KA01"
    / "N15_M07_F04_KA01_16.mat"
)

mat_contents = loadmat(
    selected_mat_path,
    squeeze_me=True,
    struct_as_record=False
)

root = mat_contents[selected_mat_path.stem]


def read_field(obj, name):

    if isinstance(obj, dict):
        return obj.get(name)

    return getattr(obj, name, None)


x_items = np.asarray(
    read_field(root, "X"),
    dtype=object
).reshape(-1)

y_items = np.asarray(
    read_field(root, "Y"),
    dtype=object
).reshape(-1)


def extract_signal_and_time(channel_name):

    for channel in y_items:

        name = read_field(channel, "Name")

        name = str(name).strip()

        if name.lower() == channel_name.lower():

            signal = np.asarray(
                read_field(channel, "Data"),
                dtype=float
            ).reshape(-1)

            x_index = int(
                np.asarray(
                    read_field(channel, "XIndex")
                ).squeeze()
            )

            time_axis = x_items[x_index - 1]

            timestamps = np.asarray(
                read_field(time_axis, "Data"),
                dtype=float
            ).reshape(-1)

            if len(signal) != len(timestamps):
                raise ValueError(
                    f"Length mismatch in {channel_name}"
                )

            return signal, timestamps

    raise ValueError(
        f"Channel {channel_name} not found."
    )


current_signal_n, current_time_n = extract_signal_and_time(
    "phase_current_1"
)

speed_signal_n, speed_time_n = extract_signal_and_time(
    "speed"
)


# ------------------------------------------------------------
# 2. DEFINE THE TWO EQUAL-SIZED SAMPLE SECTIONS
# ------------------------------------------------------------

if (
    "start_interval_idx" not in globals()
    or "end_interval_idx" not in globals()
):
    raise RuntimeError(
        "Run Step 9A-5J before this cell."
    )


during_start = int(start_interval_idx)
during_end = int(end_interval_idx) + 2

sample_count = during_end - during_start

before_end = during_start
before_start = before_end - sample_count

if before_start < 0:
    raise ValueError(
        "Insufficient samples before the anomaly."
    )


before_signal = current_signal_n[
    before_start:before_end
]

before_time = current_time_n[
    before_start:before_end
]

during_signal = current_signal_n[
    during_start:during_end
]

during_time = current_time_n[
    during_start:during_end
]


assert len(before_signal) == len(during_signal)
assert len(before_time) == len(during_time)


print("\nSELECTED SECTIONS")
print("-" * 65)

print(
    f"Before: samples {before_start:,} to "
    f"{before_end - 1:,}"
)

print(
    f"During: samples {during_start:,} to "
    f"{during_end - 1:,}"
)

print("Samples in each section:", sample_count)

print(
    "Before timestamp duration:",
    before_time[-1] - before_time[0]
)

print(
    "During timestamp duration:",
    during_time[-1] - during_time[0]
)


# ------------------------------------------------------------
# 3. DETECT POSITIVE-GOING ZERO CROSSINGS
# ------------------------------------------------------------

def calculate_zero_crossing_frequency(
    signal,
    timestamps,
    reference_fs=64000
):

    signal = np.asarray(signal, dtype=float)
    timestamps = np.asarray(timestamps, dtype=float)

    # Remove the section's DC component.
    centered_signal = signal - np.mean(signal)

    # Positive-going zero crossings.
    crossing_indices = np.flatnonzero(
        (centered_signal[:-1] <= 0)
        & (centered_signal[1:] > 0)
    )

    if len(crossing_indices) < 2:
        raise ValueError(
            "Insufficient zero crossings to calculate frequency."
        )

    # Linear interpolation between consecutive samples.
    y0 = centered_signal[crossing_indices]
    y1 = centered_signal[crossing_indices + 1]

    fractions = -y0 / (y1 - y0)

    # Fractional sample positions of the crossings.
    crossing_positions = (
        crossing_indices + fractions
    )

    # Corresponding timestamp positions.
    crossing_times = (
        timestamps[crossing_indices]
        + fractions
        * (
            timestamps[crossing_indices + 1]
            - timestamps[crossing_indices]
        )
    )

    cycles = len(crossing_times) - 1

    timestamp_duration = (
        crossing_times[-1] - crossing_times[0]
    )

    sample_span = (
        crossing_positions[-1] - crossing_positions[0]
    )

    timestamp_frequency = (
        cycles / timestamp_duration
    )

    sample_index_frequency = (
        cycles * reference_fs / sample_span
    )

    intervals = np.diff(timestamps)

    return {

        "crossing_count": len(crossing_times),

        "cycle_count": cycles,

        "timestamp_duration_s": timestamp_duration,

        "sample_span": sample_span,

        "frequency_from_timestamps_hz": timestamp_frequency,

        "frequency_from_sample_index_at_64k_hz": (
            sample_index_frequency
        ),

        "median_timestamp_interval_us": (
            np.median(intervals) * 1e6
        ),

        "effective_sampling_rate_hz": (
            1 / np.median(intervals)
        )
    }


# ------------------------------------------------------------
# 4. CALCULATE FREQUENCY FOR BOTH SECTIONS
# ------------------------------------------------------------

before_result = calculate_zero_crossing_frequency(
    before_signal,
    before_time
)

during_result = calculate_zero_crossing_frequency(
    during_signal,
    during_time
)


comparison_df = pd.DataFrame([

    {
        "section": "Before anomaly",
        **before_result
    },

    {
        "section": "During anomaly",
        **during_result
    }

])


print("\nPHASE-CURRENT FREQUENCY COMPARISON")
print("=" * 75)

print(
    comparison_df.to_string(index=False)
)


# ------------------------------------------------------------
# 5. COMPARE THE FREQUENCY RATIOS
# ------------------------------------------------------------

timestamp_frequency_ratio = (
    during_result["frequency_from_timestamps_hz"]
    / before_result["frequency_from_timestamps_hz"]
)

sample_frequency_ratio = (
    during_result["frequency_from_sample_index_at_64k_hz"]
    / before_result["frequency_from_sample_index_at_64k_hz"]
)

print("\nFREQUENCY RATIOS")
print("-" * 65)

print(
    "During / Before (timestamp-based):",
    round(timestamp_frequency_ratio, 4)
)

print(
    "During / Before (sample-index-based):",
    round(sample_frequency_ratio, 4)
)


# ------------------------------------------------------------
# 6. COMPARE INDEPENDENT MOTOR SPEED
# ------------------------------------------------------------

before_start_time = before_time[0]
before_end_time = before_time[-1]

during_start_time = during_time[0]
during_end_time = during_time[-1]


speed_before_mask = (
    (speed_time_n >= before_start_time)
    & (speed_time_n <= before_end_time)
)

speed_during_mask = (
    (speed_time_n >= during_start_time)
    & (speed_time_n <= during_end_time)
)


speed_before_section = speed_signal_n[
    speed_before_mask
]

speed_during_section = speed_signal_n[
    speed_during_mask
]


speed_comparison = pd.DataFrame([

    {
        "section": "Before anomaly",
        "sample_count": len(speed_before_section),
        "mean_speed_stored_units": np.mean(
            speed_before_section
        )
    },

    {
        "section": "During anomaly",
        "sample_count": len(speed_during_section),
        "mean_speed_stored_units": np.mean(
            speed_during_section
        )
    }

])


print("\nINDEPENDENT MOTOR SPEED COMPARISON")
print("=" * 75)

print(speed_comparison.to_string(index=False))


# ------------------------------------------------------------
# 7. PLOT BOTH SECTIONS AGAINST THEIR ACTUAL TIMESTAMPS
# ------------------------------------------------------------

before_relative_time = (
    before_time - before_time[0]
)

during_relative_time = (
    during_time - during_time[0]
)


fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 8)
)


axes[0].plot(
    before_relative_time,
    before_signal,
    linewidth=0.8
)

axes[0].set_title(
    "Phase Current Before Anomaly - Actual Timestamp Axis"
)

axes[0].set_xlabel("Elapsed time (seconds)")
axes[0].set_ylabel("Current amplitude")
axes[0].grid(True, alpha=0.3)


axes[1].plot(
    during_relative_time,
    during_signal,
    linewidth=0.8
)

axes[1].set_title(
    "Phase Current During Anomaly - Actual Timestamp Axis"
)

axes[1].set_xlabel("Elapsed time (seconds)")
axes[1].set_ylabel("Current amplitude")
axes[1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9A-5N COMPLETED")
print("=" * 75)

print(
    "Timestamp-based frequency ratio:",
    round(timestamp_frequency_ratio, 4)
)

print(
    "Sample-index-based frequency ratio:",
    round(sample_frequency_ratio, 4)
)

print(
    "\nNo samples or timestamps were modified."
)

In [ ]:
# ============================================================
# STEP 9A-5O
# COMPARE HOSTSERVICE AND MECHANICAL SAMPLE COUNTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


print("=" * 75)
print("STEP 9A-5O: CROSS-CHANNEL SAMPLE RATE CONSISTENCY")
print("=" * 75)


# ------------------------------------------------------------
# 1. VERIFY REQUIRED VARIABLES
# ------------------------------------------------------------

required_variables = [
    "current_time_n",
    "speed_time_n"
]

missing = [
    variable for variable in required_variables
    if variable not in globals()
]

if missing:
    raise RuntimeError(
        "Missing variables: " + ", ".join(missing)
        + ". Run Step 9A-5N first."
    )


# ------------------------------------------------------------
# 2. DEFINE COMMON TIME RANGE
# ------------------------------------------------------------

common_start = max(
    current_time_n[0],
    speed_time_n[0]
)

common_end = min(
    current_time_n[-1],
    speed_time_n[-1]
)


BIN_WIDTH = 0.05

bin_edges = np.arange(
    common_start,
    common_end,
    BIN_WIDTH
)

# Ensure the final timestamp is included.
if bin_edges[-1] < common_end:
    bin_edges = np.append(
        bin_edges,
        common_end
    )


# ------------------------------------------------------------
# 3. COUNT SAMPLES IN EACH TIME BIN
# ------------------------------------------------------------

host_counts, _ = np.histogram(
    current_time_n,
    bins=bin_edges
)

mechanical_counts, _ = np.histogram(
    speed_time_n,
    bins=bin_edges
)


records = []

for i in range(len(bin_edges) - 1):

    start = bin_edges[i]
    end = bin_edges[i + 1]

    duration = end - start

    host_count = host_counts[i]
    mechanical_count = mechanical_counts[i]

    if mechanical_count == 0:
        ratio = np.nan
    else:
        ratio = host_count / mechanical_count

    records.append({

        "start_time_s": start,

        "end_time_s": end,

        "host_samples": host_count,

        "mechanical_samples": mechanical_count,

        "host_samples_per_second": (
            host_count / duration
        ),

        "mechanical_samples_per_second": (
            mechanical_count / duration
        ),

        "host_to_mechanical_ratio": ratio
    })


sample_ratio_df = pd.DataFrame(records)


# ------------------------------------------------------------
# 4. DISPLAY THE RESULT
# ------------------------------------------------------------

print("\nCROSS-CHANNEL SAMPLE COUNT TABLE")
print("=" * 75)

print(
    sample_ratio_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ------------------------------------------------------------
# 5. SUMMARY BEFORE AND DURING THE ANOMALY
# ------------------------------------------------------------

ANOMALY_START = 3.876035

before_bins = sample_ratio_df[
    sample_ratio_df["end_time_s"] <= ANOMALY_START
]

during_bins = sample_ratio_df[
    sample_ratio_df["start_time_s"] >= ANOMALY_START
]


summary_records = []

for section_name, section in [
    ("Before anomaly", before_bins),
    ("During anomaly", during_bins)
]:

    if section.empty:
        continue

    summary_records.append({

        "section": section_name,

        "number_of_bins": len(section),

        "mean_host_rate": (
            section["host_samples_per_second"].mean()
        ),

        "mean_mechanical_rate": (
            section["mechanical_samples_per_second"].mean()
        ),

        "mean_sample_ratio": (
            section["host_to_mechanical_ratio"].mean()
        )
    })


cross_channel_summary = pd.DataFrame(
    summary_records
)


print("\nCROSS-CHANNEL SUMMARY")
print("=" * 75)

print(
    cross_channel_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 6. PLOT HOSTSERVICE / MECHANICAL SAMPLE RATIO
# ------------------------------------------------------------

bin_centres = (
    sample_ratio_df["start_time_s"].to_numpy()
    + sample_ratio_df["end_time_s"].to_numpy()
) / 2


plt.figure(figsize=(14, 5))

plt.plot(
    bin_centres,
    sample_ratio_df["host_to_mechanical_ratio"],
    marker="o",
    linewidth=1.5,
    label="Observed sample ratio"
)

plt.axhline(
    16,
    linestyle="--",
    label="Expected ratio (64 kHz / 4 kHz)"
)

plt.axvline(
    ANOMALY_START,
    linestyle=":",
    label="Start of timestamp anomaly"
)

plt.xlabel("Time (seconds)")
plt.ylabel("HostService samples per mechanical sample")

plt.title(
    "Cross-Channel Sample Count Consistency"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9A-5O COMPLETED")
print("=" * 75)

print("Expected sample ratio: 16")
print(
    "No signal samples or timestamps were modified."
)

In [ ]:
# ============================================================
# STEP 9A-5P
# DATASET-WIDE HOSTSERVICE TIMESTAMP QUALITY AUDIT
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.io import loadmat


print("=" * 75)
print("STEP 9A-5P: DATASET-WIDE TIMESTAMP QUALITY AUDIT")
print("=" * 75)


# ------------------------------------------------------------
# 1. LOAD DEVELOPMENT AND TEST RECORDING METADATA
# ------------------------------------------------------------

PROJECT_ROOT = Path(PROJECT_ROOT)
RAW_DIR = Path(RAW_DIR)
BASELINE_DIR = Path(BASELINE_DIR)

TEST_CSV = (
    BASELINE_DIR / "paderborn_integrated_test_features.csv"
)

if not TEST_CSV.exists():
    raise FileNotFoundError(
        f"Integrated test feature file not found: {TEST_CSV}"
    )

test_metadata = pd.read_csv(TEST_CSV)

development_metadata = development_df.copy()
development_metadata["audit_subset"] = (
    development_metadata["split"].astype(str)
)

test_metadata["audit_subset"] = "test"

audit_metadata = pd.concat(
    [development_metadata, test_metadata],
    ignore_index=True
)

required_columns = [
    "bearing",
    "file",
    "audit_subset"
]

missing_columns = [
    col for col in required_columns
    if col not in audit_metadata.columns
]

if missing_columns:
    raise ValueError(
        "Missing metadata columns: "
        + ", ".join(missing_columns)
    )


# Avoid processing the same physical recording twice.
audit_metadata = audit_metadata.drop_duplicates(
    subset=["bearing", "file"]
).reset_index(drop=True)


print("\nRECORDING COUNTS")
print("-" * 65)

print(
    "Development recordings:",
    len(development_metadata)
)

print(
    "Test recordings:",
    len(test_metadata)
)

print(
    "Unique recordings to audit:",
    len(audit_metadata)
)


# ------------------------------------------------------------
# 2. MATLAB STRUCTURE HELPERS
# ------------------------------------------------------------

def get_mat_field(obj, field_name):

    if isinstance(obj, dict):
        return obj.get(field_name)

    if hasattr(obj, field_name):
        return getattr(obj, field_name)

    return None


def get_mat_items(value):

    if isinstance(value, np.ndarray):
        return list(value.reshape(-1))

    return [value]


def mat_string(value):

    if value is None:
        return ""

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace"
        ).strip()

    if isinstance(value, np.ndarray):

        value = np.squeeze(value)

        if value.dtype.kind in ("U", "S"):
            return "".join(
                map(str, value.reshape(-1))
            ).strip()

        if value.ndim == 0:
            value = value.item()

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace"
        ).strip()

    return str(value).strip()


def find_y_channel(y_items, channel_name):

    for channel in y_items:

        name = mat_string(
            get_mat_field(channel, "Name")
        )

        if name.lower() == channel_name.lower():
            return channel

    return None


# ------------------------------------------------------------
# 3. CONFIGURATION
# ------------------------------------------------------------

NOMINAL_FS = 64000
NOMINAL_DT = 1 / NOMINAL_FS

ANOMALY_TOLERANCE = 0.05

EXPECTED_HOST_TO_MECHANICAL_RATIO = 16

EXPECTED_SAMPLES_PER_4_SECONDS = 256001

LONG_RUN_THRESHOLD = 50


# ------------------------------------------------------------
# 4. AUDIT EACH RECORDING
# ------------------------------------------------------------

audit_records = []

error_records = []

total_recordings = len(audit_metadata)


for position, (_, row) in enumerate(
    audit_metadata.iterrows(),
    start=1
):

    bearing = str(row["bearing"])
    filename = str(row["file"])
    subset = str(row["audit_subset"])

    mat_path = RAW_DIR / bearing / filename

    try:

        if not mat_path.exists():

            raise FileNotFoundError(
                f"Missing file: {mat_path}"
            )

        # Load MATLAB file.
        contents = loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )

        mat_key = mat_path.stem

        if mat_key not in contents:
            raise KeyError(
                f"MAT variable '{mat_key}' not found."
            )

        root = contents[mat_key]

        x_items = get_mat_items(
            get_mat_field(root, "X")
        )

        y_items = get_mat_items(
            get_mat_field(root, "Y")
        )

        vibration_channel = find_y_channel(
            y_items,
            "vibration_1"
        )

        speed_channel = find_y_channel(
            y_items,
            "speed"
        )

        if vibration_channel is None:
            raise ValueError(
                "vibration_1 channel not found."
            )

        if speed_channel is None:
            raise ValueError(
                "speed channel not found."
            )

        # Vibration samples.
        vibration_data = np.asarray(
            get_mat_field(vibration_channel, "Data"),
            dtype=float
        ).reshape(-1)

        # Vibration time axis.
        vibration_xindex = int(
            np.asarray(
                get_mat_field(
                    vibration_channel,
                    "XIndex"
                )
            ).squeeze()
        )

        vibration_time_axis = x_items[
            vibration_xindex - 1
        ]

        vibration_timestamps = np.asarray(
            get_mat_field(
                vibration_time_axis,
                "Data"
            ),
            dtype=float
        ).reshape(-1)

        # Mechanical speed sample count.
        speed_data = np.asarray(
            get_mat_field(speed_channel, "Data"),
            dtype=float
        ).reshape(-1)

        # Check signal/time-axis length consistency.
        if len(vibration_data) != len(vibration_timestamps):
            raise ValueError(
                "Vibration data/time-axis length mismatch."
            )

        if len(vibration_timestamps) < 2:
            raise ValueError(
                "Insufficient vibration timestamps."
            )

        dt = np.diff(vibration_timestamps)

        finite_mask = np.isfinite(dt)

        if not np.all(finite_mask):
            raise ValueError(
                "Non-finite timestamp intervals detected."
            )

        # Detect non-increasing timestamps.
        non_increasing_count = int(
            np.sum(dt <= 0)
        )

        # Deviations beyond 5% of nominal 64 kHz interval.
        anomaly_mask = (
            np.abs(dt - NOMINAL_DT)
            > ANOMALY_TOLERANCE * NOMINAL_DT
        )

        anomaly_indices = np.flatnonzero(
            anomaly_mask
        )

        # Find longest consecutive anomaly run.
        if len(anomaly_indices) > 0:

            split_positions = (
                np.where(
                    np.diff(anomaly_indices) > 1
                )[0] + 1
            )

            groups = np.split(
                anomaly_indices,
                split_positions
            )

            longest_run = max(
                len(group) for group in groups
            )

        else:
            longest_run = 0

        median_dt = np.median(dt)

        mean_dt = np.mean(dt)

        timestamp_duration = (
            vibration_timestamps[-1]
            - vibration_timestamps[0]
        )

        host_sample_count = len(vibration_data)

        mechanical_sample_count = len(speed_data)

        if mechanical_sample_count > 0:

            host_mechanical_ratio = (
                host_sample_count
                / mechanical_sample_count
            )

        else:

            host_mechanical_ratio = np.nan

        anomaly_percentage = (
            len(anomaly_indices)
            / len(dt)
        ) * 100

        audit_records.append({

            "bearing": bearing,

            "file": filename,

            "audit_subset": subset,

            "fault_class": row.get(
                "fault_class",
                np.nan
            ),

            "fault_location": row.get(
                "fault_location",
                "unknown"
            ),

            "sample_count": host_sample_count,

            "mechanical_sample_count": (
                mechanical_sample_count
            ),

            "host_mechanical_count_ratio": (
                host_mechanical_ratio
            ),

            "timestamp_duration_s": (
                timestamp_duration
            ),

            "median_interval_us": median_dt * 1e6,

            "mean_interval_us": mean_dt * 1e6,

            "median_interval_fs": 1 / median_dt,

            "anomalous_interval_count": (
                len(anomaly_indices)
            ),

            "anomalous_interval_pct": (
                anomaly_percentage
            ),

            "longest_anomaly_run": (
                longest_run
            ),

            "non_increasing_intervals": (
                non_increasing_count
            ),

            "status": "OK"
        })


    except Exception as error:

        error_records.append({

            "bearing": bearing,

            "file": filename,

            "audit_subset": subset,

            "error": str(error)
        })


    # Print periodic progress.
    if position % 100 == 0 or position == total_recordings:

        print(
            f"Processed {position:,} / "
            f"{total_recordings:,} recordings"
        )


# ------------------------------------------------------------
# 5. CREATE AUDIT DATAFRAMES
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_records)

errors_df = pd.DataFrame(error_records)


print("\n" + "=" * 75)
print("AUDIT EXECUTION SUMMARY")
print("=" * 75)

print("Successfully audited:", len(audit_df))
print("Files with errors:", len(errors_df))


if audit_df.empty:
    raise RuntimeError(
        "No recordings were successfully audited."
    )


# ------------------------------------------------------------
# 6. ADD QUALITY FLAGS
# ------------------------------------------------------------

audit_df["has_timestamp_anomaly"] = (
    audit_df["anomalous_interval_pct"] > 0
)

audit_df["has_long_anomaly_run"] = (
    audit_df["longest_anomaly_run"]
    >= LONG_RUN_THRESHOLD
)

audit_df["ratio_outside_expected_range"] = (
    np.abs(
        audit_df["host_mechanical_count_ratio"]
        - EXPECTED_HOST_TO_MECHANICAL_RATIO
    )
    > 0.05 * EXPECTED_HOST_TO_MECHANICAL_RATIO
)

audit_df["sample_count_difference"] = (
    audit_df["sample_count"]
    - EXPECTED_SAMPLES_PER_4_SECONDS
)


# ------------------------------------------------------------
# 7. OVERALL QUALITY SUMMARY
# ------------------------------------------------------------

overall_summary = pd.DataFrame([{

    "recordings_audited": len(audit_df),

    "recordings_with_timestamp_anomalies": int(
        audit_df["has_timestamp_anomaly"].sum()
    ),

    "recordings_with_long_anomaly_runs": int(
        audit_df["has_long_anomaly_run"].sum()
    ),

    "recordings_with_ratio_deviation": int(
        audit_df["ratio_outside_expected_range"].sum()
    ),

    "median_sample_count": (
        audit_df["sample_count"].median()
    ),

    "minimum_sample_count": (
        audit_df["sample_count"].min()
    ),

    "maximum_sample_count": (
        audit_df["sample_count"].max()
    ),

    "median_host_mechanical_ratio": (
        audit_df["host_mechanical_count_ratio"].median()
    )
}])


print("\nOVERALL AUDIT SUMMARY")
print("=" * 75)

print(
    overall_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 8. SUMMARY BY DEVELOPMENT / TEST PARTITION
# ------------------------------------------------------------

partition_summary = (
    audit_df
    .groupby("audit_subset")
    .agg(

        recordings=("file", "count"),

        recordings_with_anomalies=(
            "has_timestamp_anomaly",
            "sum"
        ),

        recordings_with_long_runs=(
            "has_long_anomaly_run",
            "sum"
        ),

        recordings_with_ratio_deviation=(
            "ratio_outside_expected_range",
            "sum"
        ),

        median_samples=("sample_count", "median"),

        min_samples=("sample_count", "min"),

        max_samples=("sample_count", "max"),

        median_host_mechanical_ratio=(
            "host_mechanical_count_ratio",
            "median"
        )
    )
    .reset_index()
)


print("\nSUMMARY BY DATASET PARTITION")
print("=" * 75)

print(
    partition_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 9. SUMMARY BY FAULT CLASS
# ------------------------------------------------------------

class_summary = (
    audit_df
    .groupby(
        ["fault_class", "fault_location"],
        dropna=False
    )
    .agg(

        recordings=("file", "count"),

        recordings_with_anomalies=(
            "has_timestamp_anomaly",
            "sum"
        ),

        recordings_with_long_runs=(
            "has_long_anomaly_run",
            "sum"
        ),

        median_samples=("sample_count", "median"),

        median_anomaly_pct=(
            "anomalous_interval_pct",
            "median"
        ),

        median_host_mechanical_ratio=(
            "host_mechanical_count_ratio",
            "median"
        )
    )
    .reset_index()
)


print("\nSUMMARY BY FAULT CLASS")
print("=" * 75)

print(
    class_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 10. DISPLAY THE 20 MOST AFFECTED RECORDINGS
# ------------------------------------------------------------

worst_recordings = (
    audit_df
    .sort_values(
        [
            "longest_anomaly_run",
            "anomalous_interval_pct"
        ],
        ascending=False
    )
    .head(20)
)


print("\n20 MOST AFFECTED RECORDINGS")
print("=" * 75)

display(
    worst_recordings[
        [
            "bearing",
            "file",
            "audit_subset",
            "sample_count",
            "mechanical_sample_count",
            "host_mechanical_count_ratio",
            "median_interval_us",
            "anomalous_interval_pct",
            "longest_anomaly_run"
        ]
    ]
)


# ------------------------------------------------------------
# 11. CHECK OUR PREVIOUSLY INVESTIGATED RECORDING
# ------------------------------------------------------------

selected_recording = audit_df[
    (audit_df["bearing"] == "KA01")
    & (
        audit_df["file"]
        == "N15_M07_F04_KA01_16.mat"
    )
]


print("\nPREVIOUSLY INVESTIGATED RECORDING")
print("=" * 75)

if selected_recording.empty:

    print(
        "The selected recording was not present "
        "in the audited metadata."
    )

else:

    display(
        selected_recording[
            [
                "bearing",
                "file",
                "sample_count",
                "mechanical_sample_count",
                "host_mechanical_count_ratio",
                "median_interval_us",
                "anomalous_interval_pct",
                "longest_anomaly_run"
            ]
        ]
    )


# ------------------------------------------------------------
# 12. SHOW FILES WITH LOADING ERRORS
# ------------------------------------------------------------

print("\nFILES WITH LOADING ERRORS")
print("=" * 75)

if errors_df.empty:

    print("No file-loading errors.")

else:

    display(errors_df)


# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9A-5P COMPLETED")
print("=" * 75)

print(
    "This audit covers the development and "
    "integrated held-out test recordings."
)

print(
    "No signal samples were removed, resampled "
    "or modified."
)

In [ ]:
# ============================================================
# STEP 9A-5Q
# CLASSIFY RECORDINGS BY SAMPLE COUNT AND TIMESTAMP QUALITY
# ============================================================

import numpy as np
import pandas as pd


print("=" * 75)
print("STEP 9A-5Q: SAMPLE COUNT VS TIMESTAMP QUALITY")
print("=" * 75)


# ------------------------------------------------------------
# 1. VERIFY AUDIT DATAFRAME
# ------------------------------------------------------------

if "audit_df" not in globals():
    raise RuntimeError(
        "audit_df is not available. Run Step 9A-5P first."
    )

EXPECTED_SAMPLES = 256001
COUNT_TOLERANCE = 1
LONG_RUN_THRESHOLD = 50

quality_df = audit_df.copy()


# ------------------------------------------------------------
# 2. CLASSIFY SAMPLE COUNTS
# ------------------------------------------------------------

quality_df["sample_count_difference"] = (
    quality_df["sample_count"] - EXPECTED_SAMPLES
)


def classify_sample_count(difference):

    if abs(difference) <= COUNT_TOLERANCE:
        return "Nominal"

    elif difference < 0:
        return "Below nominal"

    else:
        return "Above nominal"


quality_df["sample_count_category"] = (
    quality_df["sample_count_difference"]
    .apply(classify_sample_count)
)


# ------------------------------------------------------------
# 3. CLASSIFY LONG TIMESTAMP ANOMALY RUNS
# ------------------------------------------------------------

quality_df["timestamp_run_category"] = np.where(
    quality_df["longest_anomaly_run"] >= LONG_RUN_THRESHOLD,
    "Long anomaly run",
    "No long anomaly run"
)


# ------------------------------------------------------------
# 4. CROSS-TABULATE THE TWO PROBLEMS
# ------------------------------------------------------------

quality_cross_tab = pd.crosstab(
    quality_df["sample_count_category"],
    quality_df["timestamp_run_category"],
    margins=True
)


print("\nSAMPLE COUNT VS TIMESTAMP QUALITY")
print("=" * 75)

print(quality_cross_tab)


# ------------------------------------------------------------
# 5. CALCULATE PERCENTAGES
# ------------------------------------------------------------

quality_percentage = pd.crosstab(
    quality_df["sample_count_category"],
    quality_df["timestamp_run_category"],
    normalize="all"
) * 100


print("\nPERCENTAGE DISTRIBUTION")
print("=" * 75)

print(
    quality_percentage.round(2)
)


# ------------------------------------------------------------
# 6. SUMMARISE EACH CATEGORY
# ------------------------------------------------------------

category_summary = (

    quality_df

    .groupby(
        [
            "sample_count_category",
            "timestamp_run_category"
        ],
        dropna=False
    )

    .agg(

        recordings=("file", "count"),

        median_samples=("sample_count", "median"),

        minimum_samples=("sample_count", "min"),

        maximum_samples=("sample_count", "max"),

        median_anomaly_pct=(
            "anomalous_interval_pct",
            "median"
        ),

        maximum_anomaly_pct=(
            "anomalous_interval_pct",
            "max"
        ),

        median_longest_run=(
            "longest_anomaly_run",
            "median"
        ),

        maximum_longest_run=(
            "longest_anomaly_run",
            "max"
        )
    )

    .reset_index()
)


print("\nCATEGORY DETAILS")
print("=" * 75)

print(
    category_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 7. QUALITY BREAKDOWN BY DATASET PARTITION
# ------------------------------------------------------------

partition_quality = (

    quality_df

    .groupby(
        [
            "audit_subset",
            "sample_count_category",
            "timestamp_run_category"
        ]
    )

    .size()

    .reset_index(name="recordings")
)


print("\nQUALITY BREAKDOWN BY PARTITION")
print("=" * 75)

print(
    partition_quality.to_string(index=False)
)


# ------------------------------------------------------------
# 8. REPRESENTATIVE RECORDINGS FROM EACH CATEGORY
# ------------------------------------------------------------

representative_groups = [

    (
        "Nominal count + long anomaly run",
        "Nominal",
        "Long anomaly run"
    ),

    (
        "Above nominal count + long anomaly run",
        "Above nominal",
        "Long anomaly run"
    ),

    (
        "Below nominal count + long anomaly run",
        "Below nominal",
        "Long anomaly run"
    ),

    (
        "Above nominal count + no long anomaly run",
        "Above nominal",
        "No long anomaly run"
    ),

    (
        "Nominal count + no long anomaly run",
        "Nominal",
        "No long anomaly run"
    )
]


representative_records = []


for label, count_category, run_category in representative_groups:

    subset = quality_df[
        (quality_df["sample_count_category"] == count_category)
        & (
            quality_df["timestamp_run_category"]
            == run_category
        )
    ]

    if subset.empty:
        continue

    selected = subset.sort_values(
        "longest_anomaly_run",
        ascending=False
    ).iloc[0]

    representative_records.append({

        "category": label,

        "bearing": selected["bearing"],

        "file": selected["file"],

        "partition": selected["audit_subset"],

        "fault_location": selected["fault_location"],

        "sample_count": selected["sample_count"],

        "sample_count_difference": (
            selected["sample_count_difference"]
        ),

        "anomaly_percentage": (
            selected["anomalous_interval_pct"]
        ),

        "longest_anomaly_run": (
            selected["longest_anomaly_run"]
        )
    })


representative_df = pd.DataFrame(
    representative_records
)


print("\nREPRESENTATIVE RECORDINGS")
print("=" * 75)

print(
    representative_df.to_string(index=False)
)


# ------------------------------------------------------------
# 9. SAVE AUDIT TABLES FOR REPRODUCIBILITY
# ------------------------------------------------------------

AUDIT_OUTPUT_DIR = (
    PROJECT_ROOT / "results" / "signal_quality_audit"
)

AUDIT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


quality_df.to_csv(
    AUDIT_OUTPUT_DIR / "recording_quality_audit.csv",
    index=False
)

category_summary.to_csv(
    AUDIT_OUTPUT_DIR / "quality_category_summary.csv",
    index=False
)

representative_df.to_csv(
    AUDIT_OUTPUT_DIR / "representative_recordings.csv",
    index=False
)


print("\nSaved audit outputs to:")
print(AUDIT_OUTPUT_DIR)


# ------------------------------------------------------------
# 10. COMPLETION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9A-5Q COMPLETED")
print("=" * 75)

print(
    "No raw signals, timestamps or existing model files "
    "were modified."
)

In [ ]:
# ============================================================
# STEP 9A-5R
# CHECK RECORDING DURATION VS SAMPLE COUNT
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


print("=" * 75)
print("STEP 9A-5R: RECORDING DURATION CONSISTENCY")
print("=" * 75)


# ------------------------------------------------------------
# 1. VERIFY EXISTING AUDIT DATA
# ------------------------------------------------------------

if "audit_df" not in globals():
    raise RuntimeError(
        "audit_df is unavailable. Run Step 9A-5P first."
    )

duration_df = audit_df.copy()
# Recreate the categories generated in Step 9A-5Q.
# They were added to quality_df, not the original audit_df.

EXPECTED_SAMPLES = 256001
COUNT_TOLERANCE = 1
LONG_RUN_THRESHOLD = 50

duration_df["sample_count_difference"] = (
    duration_df["sample_count"] - EXPECTED_SAMPLES
)

duration_df["sample_count_category"] = np.select(
    [
        duration_df["sample_count_difference"].abs()
        <= COUNT_TOLERANCE,

        duration_df["sample_count_difference"]
        < -COUNT_TOLERANCE
    ],
    [
        "Nominal",
        "Below nominal"
    ],
    default="Above nominal"
)

duration_df["timestamp_run_category"] = np.where(
    duration_df["longest_anomaly_run"] >= LONG_RUN_THRESHOLD,
    "Long anomaly run",
    "No long anomaly run"
)

required_columns = [
    "bearing",
    "file",
    "audit_subset",
    "sample_count",
    "timestamp_duration_s",
    "median_interval_us",
    "longest_anomaly_run",
    "anomalous_interval_pct"
]

missing = [
    col for col in required_columns
    if col not in duration_df.columns
]

if missing:
    raise ValueError(
        "Missing columns: " + ", ".join(missing)
    )


# ------------------------------------------------------------
# 2. CALCULATE EFFECTIVE SAMPLING RATE
# ------------------------------------------------------------

NOMINAL_FS = 64000
EXPECTED_DURATION = 4.0

duration_df["effective_host_sampling_rate"] = (
    (duration_df["sample_count"] - 1)
    / duration_df["timestamp_duration_s"]
)

duration_df["expected_samples_for_duration"] = (
    duration_df["timestamp_duration_s"]
    * NOMINAL_FS
    + 1
)

duration_df["sample_count_difference_from_duration"] = (
    duration_df["sample_count"]
    - duration_df["expected_samples_for_duration"]
)

duration_df["duration_difference_from_4s"] = (
    duration_df["timestamp_duration_s"]
    - EXPECTED_DURATION
)

duration_df["duration_category"] = np.select(
    [
        duration_df["timestamp_duration_s"] < 3.95,
        duration_df["timestamp_duration_s"] > 4.05
    ],
    [
        "Shorter than 4 seconds",
        "Longer than 4 seconds"
    ],
    default="Approximately 4 seconds"
)

duration_df["rate_deviation_pct"] = (
    (
        duration_df["effective_host_sampling_rate"]
        - NOMINAL_FS
    )
    / NOMINAL_FS
) * 100


# ------------------------------------------------------------
# 3. OVERALL DURATION STATISTICS
# ------------------------------------------------------------

print("\nOVERALL RECORDING DURATION")
print("=" * 75)

print(
    duration_df["timestamp_duration_s"].describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    ).to_string()
)


# ------------------------------------------------------------
# 4. DURATION BY SAMPLE COUNT AND TIMESTAMP CATEGORY
# ------------------------------------------------------------

duration_summary = (

    duration_df

    .groupby(
        [
            "sample_count_category",
            "timestamp_run_category"
        ],
        dropna=False
    )

    .agg(

        recordings=("file", "count"),

        median_sample_count=("sample_count", "median"),

        median_duration_s=(
            "timestamp_duration_s",
            "median"
        ),

        minimum_duration_s=(
            "timestamp_duration_s",
            "min"
        ),

        maximum_duration_s=(
            "timestamp_duration_s",
            "max"
        ),

        p05_duration_s=(
            "timestamp_duration_s",
            lambda x: x.quantile(0.05)
        ),

        p95_duration_s=(
            "timestamp_duration_s",
            lambda x: x.quantile(0.95)
        ),

        median_effective_fs=(
            "effective_host_sampling_rate",
            "median"
        ),

        median_rate_deviation_pct=(
            "rate_deviation_pct",
            "median"
        )
    )

    .reset_index()
)


print("\nDURATION BY QUALITY CATEGORY")
print("=" * 75)

print(
    duration_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 5. CHECK HOW MANY RECORDINGS ARE APPROXIMATELY 4 SECONDS
# ------------------------------------------------------------

duration_category_summary = (

    duration_df

    .groupby(
        [
            "sample_count_category",
            "duration_category"
        ]
    )

    .agg(

        recordings=("file", "count"),

        median_samples=("sample_count", "median"),

        median_duration_s=(
            "timestamp_duration_s",
            "median"
        ),

        median_effective_fs=(
            "effective_host_sampling_rate",
            "median"
        )
    )

    .reset_index()
)


print("\nSAMPLE COUNT VS DURATION CATEGORY")
print("=" * 75)

print(
    duration_category_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 6. CHECK EFFECTIVE RATE CONSISTENCY
# ------------------------------------------------------------

duration_df["rate_consistency"] = np.where(
    duration_df["effective_host_sampling_rate"].between(
        NOMINAL_FS * 0.95,
        NOMINAL_FS * 1.05
    ),
    "Within 5% of 64 kHz",
    "Outside 5% of 64 kHz"
)


rate_summary = (

    duration_df

    .groupby(
        [
            "sample_count_category",
            "timestamp_run_category",
            "rate_consistency"
        ]
    )

    .size()

    .reset_index(name="recordings")
)


print("\nEFFECTIVE SAMPLING RATE CONSISTENCY")
print("=" * 75)

print(
    rate_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 7. INSPECT RECORDINGS WITH EXCESS SAMPLES
#    BUT NO LONG ANOMALY RUN
# ------------------------------------------------------------

excess_without_long_run = duration_df[
    (
        duration_df["sample_count_category"]
        == "Above nominal"
    )
    &
    (
        duration_df["timestamp_run_category"]
        == "No long anomaly run"
    )
]


print("\nEXCESS SAMPLES WITHOUT LONG ANOMALY RUN")
print("=" * 75)

print(
    "Number of recordings:",
    len(excess_without_long_run)
)

largest_excess_recordings = (
    excess_without_long_run
    .sort_values(
        "sample_count",
        ascending=False
    )
    .head(15)
)


display(
    largest_excess_recordings[
        [
            "bearing",
            "file",
            "audit_subset",
            "sample_count",
            "timestamp_duration_s",
            "median_interval_us",
            "effective_host_sampling_rate",
            "rate_deviation_pct",
            "anomalous_interval_pct",
            "longest_anomaly_run"
        ]
    ]
)


# ------------------------------------------------------------
# 8. INSPECT RECORDINGS WITH LONG ANOMALY RUNS
# ------------------------------------------------------------

long_run_recordings = duration_df[
    duration_df["timestamp_run_category"]
    == "Long anomaly run"
]


print("\nRECORDINGS WITH LONG ANOMALY RUNS")
print("=" * 75)

print(
    "Number of recordings:",
    len(long_run_recordings)
)

largest_long_runs = (
    long_run_recordings
    .sort_values(
        "longest_anomaly_run",
        ascending=False
    )
    .head(15)
)


display(
    largest_long_runs[
        [
            "bearing",
            "file",
            "audit_subset",
            "sample_count",
            "timestamp_duration_s",
            "median_interval_us",
            "effective_host_sampling_rate",
            "rate_deviation_pct",
            "anomalous_interval_pct",
            "longest_anomaly_run"
        ]
    ]
)


# ------------------------------------------------------------
# 9. PLOT SAMPLE COUNT AGAINST DURATION
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

for category in [
    "Nominal",
    "Above nominal",
    "Below nominal"
]:

    subset = duration_df[
        duration_df["sample_count_category"] == category
    ]

    plt.scatter(
        subset["timestamp_duration_s"],
        subset["sample_count"],
        s=12,
        alpha=0.5,
        label=category
    )


duration_range = np.linspace(
    duration_df["timestamp_duration_s"].min(),
    duration_df["timestamp_duration_s"].max(),
    200
)


plt.plot(
    duration_range,
    NOMINAL_FS * duration_range + 1,
    linestyle="--",
    label="Expected count at 64 kHz"
)


plt.axvline(
    4.0,
    linestyle=":",
    label="Expected 4-second duration"
)

plt.xlabel("Recorded timestamp duration (seconds)")
plt.ylabel("Number of HostService samples")

plt.title(
    "HostService Sample Count vs Recorded Duration"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 10. PLOT EFFECTIVE SAMPLING RATE DISTRIBUTION
# ------------------------------------------------------------

plt.figure(figsize=(12, 6))

for category in [
    "Nominal",
    "Above nominal",
    "Below nominal"
]:

    subset = duration_df[
        duration_df["sample_count_category"] == category
    ]

    plt.hist(
        subset["effective_host_sampling_rate"],
        bins=50,
        alpha=0.45,
        label=category
    )


plt.axvline(
    NOMINAL_FS,
    linestyle="--",
    label="Nominal 64 kHz"
)

plt.xlabel("Effective sampling rate (Hz)")
plt.ylabel("Number of recordings")

plt.title(
    "Effective HostService Sampling Rate Distribution"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 11. SAVE THE DURATION AUDIT
# ------------------------------------------------------------

AUDIT_OUTPUT_DIR = (
    PROJECT_ROOT / "results" / "signal_quality_audit"
)

AUDIT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

duration_df.to_csv(
    AUDIT_OUTPUT_DIR / "recording_duration_audit.csv",
    index=False
)

duration_summary.to_csv(
    AUDIT_OUTPUT_DIR / "duration_quality_summary.csv",
    index=False
)

rate_summary.to_csv(
    AUDIT_OUTPUT_DIR / "sampling_rate_consistency.csv",
    index=False
)


print("\nSaved duration audit outputs to:")
print(AUDIT_OUTPUT_DIR)


# ------------------------------------------------------------
# 12. COMPLETION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9A-5R COMPLETED")
print("=" * 75)

print(
    "No raw data, timestamps or model files were modified."
)

In [ ]:
# ============================================================
# STEP 9B-1
# REPRODUCE AND VERIFY THE ORIGINAL 34 FEATURES
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.io import loadmat
from scipy.stats import kurtosis, skew
from scipy.signal import butter, sosfiltfilt, hilbert
from scipy.fft import rfft, rfftfreq


print("=" * 75)
print("STEP 9B-1: ORIGINAL FEATURE REPRODUCTION")
print("=" * 75)


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

TARGET_FILE = "N15_M07_F04_KA01_16.mat"
TARGET_BEARING = "KA01"

FS = 64000
LOWCUT = 2000
HIGHCUT = 10000
HARMONIC_TOLERANCE = 3

N_BALLS = 8
BALL_DIAMETER = 6.75
PITCH_DIAMETER = 29.05
PRESSURE_ANGLE_DEG = 0

COS_THETA = np.cos(
    np.deg2rad(PRESSURE_ANGLE_DEG)
)


# ------------------------------------------------------------
# 2. DEFINE THE ORIGINAL 22 FEATURES
# ------------------------------------------------------------

FEATURES_22 = [
    "mean",
    "std",
    "min",
    "max",
    "kurtosis",
    "skewness",
    "crest_factor",
    "dominant_frequency",
    "dominant_magnitude",
    "spectral_centroid",
    "spectral_bandwidth",
    "band_power_0_500",
    "band_power_500_1000",
    "band_power_1000_2000",
    "band_power_2000_3000",
    "band_power_3000_4000",
    "band_power_4000_5000",
    "band_power_5000_6000",
    "band_power_6000_7000",
    "band_power_7000_8000",
    "band_power_8000_9000",
    "band_power_9000_10000"
]


# ------------------------------------------------------------
# 3. DEFINE THE ORIGINAL 12 FAULT FEATURES
# ------------------------------------------------------------

FAULT_FEATURES_12 = (

    [
        f"bpfo_{h}x_energy_ratio"
        for h in range(1, 6)
    ]

    + ["bpfo_1to5_total"]

    + [
        f"bpfi_{h}x_energy_ratio"
        for h in range(1, 6)
    ]

    + ["bpfi_1to5_total"]
)


UNIFIED_FEATURES = (
    FEATURES_22 + FAULT_FEATURES_12
)


print("\nExpected feature count:", len(UNIFIED_FEATURES))


# ------------------------------------------------------------
# 4. LOCATE AND LOAD THE RECORDING
# ------------------------------------------------------------

mat_path = (
    Path(RAW_DIR)
    / TARGET_BEARING
    / TARGET_FILE
)

if not mat_path.exists():
    raise FileNotFoundError(mat_path)


mat = loadmat(
    mat_path,
    squeeze_me=False,
    struct_as_record=False
)


data_keys = [
    key for key in mat.keys()
    if not key.startswith("__")
]


if len(data_keys) != 1:
    raise ValueError(
        f"Unexpected MATLAB structure: {mat_path}"
    )


data = mat[data_keys[0]][0, 0]


# ------------------------------------------------------------
# 5. HELPER TO EXTRACT MATLAB CHANNELS
# ------------------------------------------------------------

def extract_channel_exact(data, channel_name):

    y_channels = np.asarray(
        data.Y
    ).flatten()

    for channel in y_channels:

        name = channel.Name

        if isinstance(name, np.ndarray):
            name = name.flatten()[0]

        if isinstance(name, bytes):
            name = name.decode(
                "utf-8",
                errors="replace"
            )

        name = str(name).strip()

        if name == channel_name:

            return np.asarray(
                channel.Data
            ).squeeze().astype(np.float64)

    raise ValueError(
        f"{channel_name} not found in {TARGET_FILE}"
    )


# Extract complete vibration signal and measured RPM.
signal = extract_channel_exact(
    data,
    "vibration_1"
)

speed = extract_channel_exact(
    data,
    "speed"
)

rpm = float(np.mean(speed))


print("\nRAW SIGNAL INFORMATION")
print("-" * 65)

print("File:", TARGET_FILE)
print("Vibration samples:", len(signal))
print("Measured RPM:", rpm)
print("Expected BPFO/BPFI sample segment:", FS)


# ------------------------------------------------------------
# 6. ORIGINAL 26-FEATURE FUNCTION
# ------------------------------------------------------------

def extract_vibration_features_exact(
    signal,
    fs=FS
):

    signal = np.asarray(
        signal,
        dtype=np.float64
    ).flatten()

    if len(signal) == 0:
        raise ValueError("Signal is empty.")

    # Time-domain features
    mean_value = np.mean(signal)

    std_value = np.std(signal)

    rms_value = np.sqrt(
        np.mean(signal ** 2)
    )

    min_value = np.min(signal)

    max_value = np.max(signal)

    peak_to_peak = max_value - min_value

    kurtosis_value = kurtosis(
        signal,
        fisher=True,
        bias=False
    )

    skewness_value = skew(
        signal,
        bias=False
    )

    crest_factor = (
        np.max(np.abs(signal)) / rms_value
        if rms_value != 0 else 0
    )

    # Frequency-domain features
    n = len(signal)

    centered_signal = signal - mean_value

    fft_values = np.fft.rfft(
        centered_signal
    )

    frequencies = np.fft.rfftfreq(
        n,
        d=1 / fs
    )

    magnitude = np.abs(fft_values)

    magnitude_no_dc = magnitude.copy()

    if len(magnitude_no_dc) > 1:
        magnitude_no_dc[0] = 0

    dominant_index = np.argmax(
        magnitude_no_dc
    )

    dominant_frequency = frequencies[
        dominant_index
    ]

    dominant_magnitude = magnitude[
        dominant_index
    ]

    power = magnitude ** 2

    total_power = np.sum(power)

    if total_power > 0:

        spectral_centroid = (
            np.sum(frequencies * power)
            / total_power
        )

        spectral_bandwidth = np.sqrt(
            np.sum(
                ((frequencies - spectral_centroid) ** 2)
                * power
            )
            / total_power
        )

        probability = power / total_power

        spectral_entropy = -np.sum(
            probability[probability > 0]
            * np.log2(probability[probability > 0])
        )

    else:

        spectral_centroid = 0.0
        spectral_bandwidth = 0.0
        spectral_entropy = 0.0

    spectral_rms = np.sqrt(
        np.mean(power)
    )

    bands = [
        (0, 500),
        (500, 1000),
        (1000, 2000),
        (2000, 3000),
        (3000, 4000),
        (4000, 5000),
        (5000, 6000),
        (6000, 7000),
        (7000, 8000),
        (8000, 9000),
        (9000, 10000)
    ]

    features = {

        "mean": mean_value,
        "std": std_value,
        "rms": rms_value,
        "min": min_value,
        "max": max_value,
        "peak_to_peak": peak_to_peak,
        "kurtosis": kurtosis_value,
        "skewness": skewness_value,
        "crest_factor": crest_factor,
        "dominant_frequency": dominant_frequency,
        "dominant_magnitude": dominant_magnitude,
        "spectral_centroid": spectral_centroid,
        "spectral_bandwidth": spectral_bandwidth,
        "spectral_rms": spectral_rms,
        "spectral_entropy": spectral_entropy
    }

    for low, high in bands:

        mask = (
            (frequencies >= low)
            & (frequencies < high)
        )

        features[
            f"band_power_{low}_{high}"
        ] = np.sum(power[mask])

    return features


# ------------------------------------------------------------
# 7. ORIGINAL ENVELOPE SPECTRUM FUNCTION
# ------------------------------------------------------------

def envelope_spectrum_exact(
    signal,
    fs,
    lowcut,
    highcut,
    max_freq
):

    signal = signal[:fs].copy()

    signal = signal - np.mean(signal)

    sos = butter(
        4,
        [lowcut, highcut],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    filtered = sosfiltfilt(
        sos,
        signal
    )

    analytic_signal = hilbert(
        filtered
    )

    envelope = np.abs(
        analytic_signal
    )

    envelope = envelope - np.mean(envelope)

    spectrum = np.abs(
        rfft(envelope)
    )

    frequencies = rfftfreq(
        len(envelope),
        d=1 / fs
    )

    mask = frequencies <= max_freq

    return frequencies[mask], spectrum[mask]


# ------------------------------------------------------------
# 8. ORIGINAL BEARING FAULT FREQUENCY CALCULATIONS
# ------------------------------------------------------------

def calculate_fault_frequencies_exact(rpm):

    fr = rpm / 60.0

    bpfo = (
        N_BALLS / 2
        * fr
        * (
            1
            - (BALL_DIAMETER / PITCH_DIAMETER)
            * COS_THETA
        )
    )

    bpfi = (
        N_BALLS / 2
        * fr
        * (
            1
            + (BALL_DIAMETER / PITCH_DIAMETER)
            * COS_THETA
        )
    )

    ftf = (
        0.5
        * fr
        * (
            1
            - (BALL_DIAMETER / PITCH_DIAMETER)
            * COS_THETA
        )
    )

    bsf = (
        PITCH_DIAMETER / (2 * BALL_DIAMETER)
        * fr
        * (
            1
            - (BALL_DIAMETER / PITCH_DIAMETER) ** 2
            * COS_THETA ** 2
        )
    )

    return fr, bpfo, bpfi, ftf, bsf


# ------------------------------------------------------------
# 9. ORIGINAL BPFO FEATURE CALCULATION
# ------------------------------------------------------------

def extract_bpfo_exact(signal, rpm):

    frequencies, spectrum = envelope_spectrum_exact(
        signal,
        FS,
        LOWCUT,
        HIGHCUT,
        500
    )

    spectral_power = spectrum ** 2

    total_band = (
        (frequencies >= 20)
        & (frequencies <= 500)
    )

    total_energy = np.sum(
        spectral_power[total_band]
    )

    _, bpfo, _, _, _ = (
        calculate_fault_frequencies_exact(rpm)
    )

    features = {}

    for harmonic in range(1, 6):

        target = harmonic * bpfo

        band = (
            np.abs(frequencies - target)
            <= HARMONIC_TOLERANCE
        )

        harmonic_energy = np.sum(
            spectral_power[band]
        )

        features[
            f"bpfo_{harmonic}x_energy_ratio"
        ] = (
            harmonic_energy
            / (total_energy + 1e-12)
        )

    features["bpfo_1to5_total"] = sum(
        features[
            f"bpfo_{harmonic}x_energy_ratio"
        ]
        for harmonic in range(1, 6)
    )

    return features


# ------------------------------------------------------------
# 10. ORIGINAL BPFI FEATURE CALCULATION
# ------------------------------------------------------------

def extract_bpfi_exact(signal, rpm):

    frequencies, spectrum = envelope_spectrum_exact(
        signal,
        FS,
        LOWCUT,
        HIGHCUT,
        700
    )

    spectral_power = spectrum ** 2

    total_band = (
        (frequencies >= 20)
        & (frequencies <= 700)
    )

    total_energy = np.sum(
        spectral_power[total_band]
    )

    _, _, bpfi, _, _ = (
        calculate_fault_frequencies_exact(rpm)
    )

    features = {}

    for harmonic in range(1, 6):

        target = harmonic * bpfi

        band = (
            np.abs(frequencies - target)
            <= HARMONIC_TOLERANCE
        )

        harmonic_energy = np.sum(
            spectral_power[band]
        )

        features[
            f"bpfi_{harmonic}x_energy_ratio"
        ] = (
            harmonic_energy
            / (total_energy + 1e-12)
        )

    features["bpfi_1to5_total"] = sum(
        features[
            f"bpfi_{harmonic}x_energy_ratio"
        ]
        for harmonic in range(1, 6)
    )

    return features


# ------------------------------------------------------------
# 11. RECOMPUTE ALL 34 FEATURES
# ------------------------------------------------------------

original_26 = extract_vibration_features_exact(
    signal
)

# Select the exact 22 features used by the baseline.
original_22 = {
    feature: original_26[feature]
    for feature in FEATURES_22
}

bpfo_6 = extract_bpfo_exact(
    signal,
    rpm
)

bpfi_6 = extract_bpfi_exact(
    signal,
    rpm
)

recomputed_features = {
    **original_22,
    **bpfo_6,
    **bpfi_6
}


assert len(recomputed_features) == 34

assert list(recomputed_features.keys()) == UNIFIED_FEATURES

print("\nFEATURE RECOMPUTATION")
print("-" * 65)

print("Original 26 features calculated:", len(original_26))
print("Selected original features:", len(original_22))
print("BPFO features:", len(bpfo_6))
print("BPFI features:", len(bpfi_6))
print("Total features:", len(recomputed_features))


# ------------------------------------------------------------
# 12. FIND THE SAVED DEVELOPMENT DATASET ROW
# ------------------------------------------------------------

matching_rows = development_df[
    development_df["file"] == TARGET_FILE
]

if len(matching_rows) != 1:

    raise ValueError(
        f"Expected one matching development row, "
        f"found {len(matching_rows)}."
    )

expected_row = matching_rows.iloc[0]


missing_columns = [
    feature for feature in UNIFIED_FEATURES
    if feature not in development_df.columns
]

if missing_columns:

    raise ValueError(
        "Features missing from development_df: "
        + ", ".join(missing_columns)
    )


# ------------------------------------------------------------
# 13. COMPARE RECOMPUTED FEATURES WITH SAVED FEATURES
# ------------------------------------------------------------

comparison_records = []

for feature in UNIFIED_FEATURES:

    expected_value = float(
        expected_row[feature]
    )

    recomputed_value = float(
        recomputed_features[feature]
    )

    absolute_difference = abs(
        expected_value - recomputed_value
    )

    relative_difference_pct = (
        absolute_difference
        / max(abs(expected_value), 1e-12)
    ) * 100

    matches = np.isclose(
        expected_value,
        recomputed_value,
        rtol=1e-5,
        atol=1e-8
    )

    comparison_records.append({

        "feature": feature,

        "saved_value": expected_value,

        "recomputed_value": recomputed_value,

        "absolute_difference": absolute_difference,

        "relative_difference_pct": (
            relative_difference_pct
        ),

        "matches": matches
    })


comparison_df = pd.DataFrame(
    comparison_records
)


print("\n34-FEATURE PARITY COMPARISON")
print("=" * 75)

print(
    comparison_df.to_string(index=False)
)


# ------------------------------------------------------------
# 14. FINAL PARITY SUMMARY
# ------------------------------------------------------------

matched_count = int(
    comparison_df["matches"].sum()
)

mismatched_count = (
    len(comparison_df) - matched_count
)


print("\n" + "=" * 75)
print("PARITY SUMMARY")
print("=" * 75)

print("Total features:", len(comparison_df))
print("Matching features:", matched_count)
print("Mismatching features:", mismatched_count)

print(
    "Maximum absolute difference:",
    comparison_df["absolute_difference"].max()
)

print(
    "Maximum relative difference (%):",
    comparison_df["relative_difference_pct"].max()
)

if mismatched_count == 0:

    print(
        "\nSUCCESS: All 34 recomputed features match "
        "the saved development dataset within tolerance."
    )

else:

    print(
        "\nWARNING: Some features do not match. "
        "Do not connect this implementation to the model yet."
    )

    print("\nMismatching features:")

    display(
        comparison_df[
            ~comparison_df["matches"]
        ].sort_values(
            "relative_difference_pct",
            ascending=False
        )
    )


print("\nSTEP 9B-1 COMPLETED")
print("No existing model or dataset was modified.")

In [ ]:
# ============================================================
# STEP 9B-2
# RAW MAT FILE TO PACKAGED MODEL PREDICTION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from scipy.io import loadmat


print("=" * 75)
print("STEP 9B-2: END-TO-END RAW SIGNAL INFERENCE TEST")
print("=" * 75)


# ------------------------------------------------------------
# 1. LOAD HELD-OUT TEST FEATURE REFERENCE
# ------------------------------------------------------------

TEST_FEATURES_CSV = (
    BASELINE_DIR / "paderborn_integrated_test_features.csv"
)

test_reference_df = pd.read_csv(
    TEST_FEATURES_CSV
)

print("\nTest reference dataset:", test_reference_df.shape)


# ------------------------------------------------------------
# 2. LOAD THE PACKAGED BASELINE MODEL
# ------------------------------------------------------------

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "paderborn_baseline"
    / "paderborn_baseline_pipeline.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)

baseline_pipeline = joblib.load(MODEL_PATH)

print("Model loaded:", MODEL_PATH)


# Verify model feature count.
if baseline_pipeline.n_features_in_ != 34:
    raise ValueError(
        f"Expected 34 model inputs, "
        f"found {baseline_pipeline.n_features_in_}"
    )

if hasattr(baseline_pipeline, "feature_names_in_"):

    model_feature_order = list(
        baseline_pipeline.feature_names_in_
    )

    if model_feature_order != UNIFIED_FEATURES:
        raise ValueError(
            "Model feature ordering differs from "
            "the reconstructed feature ordering."
        )


# ------------------------------------------------------------
# 3. SELECT ONE HELD-OUT RECORDING PER FAULT CATEGORY
# ------------------------------------------------------------

SELECTED_TEST_BEARINGS = [
    "K006",
    "KA22",
    "KI18"
]

selected_test_rows = []

for bearing in SELECTED_TEST_BEARINGS:

    bearing_rows = test_reference_df[
        test_reference_df["bearing"] == bearing
    ].sort_values(
        ["operating_condition", "measurement_number"]
    )

    if bearing_rows.empty:
        raise ValueError(
            f"No test recording found for {bearing}"
        )

    selected_test_rows.append(
        bearing_rows.iloc[0]
    )


# ------------------------------------------------------------
# 4. EXTRACT FEATURES AND PREDICT
# ------------------------------------------------------------

inference_records = []
parity_records = []

for row in selected_test_rows:

    filename = row["file"]
    bearing = row["bearing"]

    print("\nProcessing:", filename)

    mat_path = (
        RAW_DIR / bearing / filename
    )

    if not mat_path.exists():
        raise FileNotFoundError(mat_path)

    mat = loadmat(
        mat_path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [
        key for key in mat.keys()
        if not key.startswith("__")
    ]

    if len(data_keys) != 1:
        raise ValueError(
            f"Unexpected MATLAB structure: {filename}"
        )

    data = mat[data_keys[0]][0, 0]

    # Extract original vibration signal and measured speed.
    signal = extract_channel_exact(
        data,
        "vibration_1"
    )

    speed = extract_channel_exact(
        data,
        "speed"
    )

    rpm = float(np.mean(speed))

    # Original 22-feature extraction.
    features_26 = extract_vibration_features_exact(
        signal
    )

    features_22 = {
        feature: features_26[feature]
        for feature in FEATURES_22
    }

    # Original BPFO and BPFI feature extraction.
    bpfo_features = extract_bpfo_exact(
        signal,
        rpm
    )

    bpfi_features = extract_bpfi_exact(
        signal,
        rpm
    )

    combined_features = {
        **features_22,
        **bpfo_features,
        **bpfi_features
    }

    # Ensure correct ordering.
    X_raw = pd.DataFrame(
        [[combined_features[f] for f in UNIFIED_FEATURES]],
        columns=UNIFIED_FEATURES
    )

    # Reference features from the existing test CSV.
    X_reference = pd.DataFrame(
        [[row[f] for f in UNIFIED_FEATURES]],
        columns=UNIFIED_FEATURES
    )

    # --------------------------------------------------------
    # FEATURE PARITY CHECK
    # --------------------------------------------------------

    differences = (
        X_raw.to_numpy()
        - X_reference.to_numpy()
    )

    max_absolute_difference = np.max(
        np.abs(differences)
    )

    parity_passed = np.allclose(
        X_raw.to_numpy(),
        X_reference.to_numpy(),
        rtol=1e-5,
        atol=1e-8
    )

    parity_records.append({

        "bearing": bearing,
        "file": filename,
        "features_match": parity_passed,
        "maximum_absolute_difference": (
            max_absolute_difference
        )
    })

    # --------------------------------------------------------
    # MODEL PREDICTION FROM RAW SIGNAL
    # --------------------------------------------------------

    raw_prediction = int(
        baseline_pipeline.predict(X_raw)[0]
    )

    raw_probabilities = (
        baseline_pipeline.predict_proba(X_raw)[0]
    )

    classes = baseline_pipeline.classes_

    positive_index = np.where(
        classes == 1
    )[0][0]

    raw_damaged_probability = float(
        raw_probabilities[positive_index]
    )

    # --------------------------------------------------------
    # MODEL PREDICTION FROM SAVED FEATURES
    # --------------------------------------------------------

    reference_prediction = int(
        baseline_pipeline.predict(X_reference)[0]
    )

    reference_probabilities = (
        baseline_pipeline.predict_proba(X_reference)[0]
    )

    reference_damaged_probability = float(
        reference_probabilities[positive_index]
    )

    predictions_match = (
        raw_prediction == reference_prediction
    )

    probabilities_match = np.allclose(
        raw_probabilities,
        reference_probabilities,
        rtol=1e-5,
        atol=1e-8
    )

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    inference_records.append({

        "bearing": bearing,
        "file": filename,

        "actual_label": int(row["binary_label"]),

        "predicted_label": raw_prediction,

        "damaged_probability": (
            raw_damaged_probability
        ),

        "prediction_from_saved_features": (
            reference_prediction
        ),

        "probability_from_saved_features": (
            reference_damaged_probability
        ),

        "prediction_matches": predictions_match,

        "probability_matches": probabilities_match,

        "feature_parity": parity_passed
    })


# ------------------------------------------------------------
# 5. DISPLAY FEATURE PARITY RESULTS
# ------------------------------------------------------------

parity_df = pd.DataFrame(
    parity_records
)

print("\nFEATURE PARITY RESULTS")
print("=" * 75)

print(
    parity_df.to_string(index=False)
)


# ------------------------------------------------------------
# 6. DISPLAY INFERENCE RESULTS
# ------------------------------------------------------------

inference_df = pd.DataFrame(
    inference_records
)

print("\nRAW SIGNAL INFERENCE RESULTS")
print("=" * 75)

print(
    inference_df.to_string(index=False)
)


# ------------------------------------------------------------
# 7. VERIFY END-TO-END CONSISTENCY
# ------------------------------------------------------------

all_features_match = bool(
    inference_df["feature_parity"].all()
)

all_predictions_match = bool(
    inference_df["prediction_matches"].all()
)

all_probabilities_match = bool(
    inference_df["probability_matches"].all()
)

all_predictions_correct = bool(
    (
        inference_df["actual_label"]
        == inference_df["predicted_label"]
    ).all()
)


print("\n" + "=" * 75)
print("END-TO-END VERIFICATION SUMMARY")
print("=" * 75)

print("Feature parity:", all_features_match)

print("Prediction parity:", all_predictions_match)

print("Probability parity:", all_probabilities_match)

print("All three predictions correct:", all_predictions_correct)


if (
    all_features_match
    and all_predictions_match
    and all_probabilities_match
):

    print(
        "\nSUCCESS: Raw MAT files reproduce the "
        "packaged model predictions."
    )

else:

    print(
        "\nWARNING: A discrepancy was detected. "
        "Inspect the tables before proceeding."
    )


print("\nSTEP 9B-2 COMPLETED")
print("No model retraining or modification performed.")

In [ ]:
# ============================================================
# STEP 9B-3
# FULL HELD-OUT TEST SET RAW INFERENCE VERIFICATION
# ============================================================

import numpy as np
import pandas as pd
import time

from pathlib import Path
from scipy.io import loadmat


print("=" * 75)
print("STEP 9B-3: FULL TEST SET RAW INFERENCE VERIFICATION")
print("=" * 75)


# ------------------------------------------------------------
# 1. VERIFY REQUIRED VARIABLES
# ------------------------------------------------------------

required_variables = [
    "test_reference_df",
    "baseline_pipeline",
    "UNIFIED_FEATURES",
    "extract_channel_exact",
    "extract_vibration_features_exact",
    "extract_bpfo_exact",
    "extract_bpfi_exact",
    "RAW_DIR",
    "PROJECT_ROOT"
]

missing = [
    variable for variable in required_variables
    if variable not in globals()
]

if missing:
    raise RuntimeError(
        "Missing variables. Run the preceding steps first: "
        + ", ".join(missing)
    )


# ------------------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------------------

test_rows = test_reference_df.copy()

if len(test_rows) != 320:
    raise ValueError(
        f"Expected 320 test recordings, found {len(test_rows)}"
    )

if list(baseline_pipeline.feature_names_in_) != UNIFIED_FEATURES:
    raise ValueError(
        "Feature order does not match the packaged model."
    )


# Audit metadata from Step 9A-5P
audit_lookup = audit_df.set_index(
    ["bearing", "file"]
)


# ------------------------------------------------------------
# 3. PROCESS ALL HELD-OUT TEST RECORDINGS
# ------------------------------------------------------------

results = []
failures = []

start_time = time.time()

for i, (_, row) in enumerate(
    test_rows.iterrows(),
    start=1
):

    bearing = str(row["bearing"])
    filename = str(row["file"])

    mat_path = (
        Path(RAW_DIR)
        / bearing
        / filename
    )

    try:

        if not mat_path.exists():
            raise FileNotFoundError(mat_path)

        # -----------------------------------------------
        # Load original MAT file
        # -----------------------------------------------

        mat = loadmat(
            mat_path,
            squeeze_me=False,
            struct_as_record=False
        )

        data_keys = [
            key for key in mat.keys()
            if not key.startswith("__")
        ]

        if len(data_keys) != 1:
            raise ValueError(
                "Unexpected MATLAB structure."
            )

        data = mat[data_keys[0]][0, 0]

        # -----------------------------------------------
        # Extract original vibration and measured RPM
        # -----------------------------------------------

        signal = extract_channel_exact(
            data,
            "vibration_1"
        )

        speed = extract_channel_exact(
            data,
            "speed"
        )

        rpm = float(np.mean(speed))

        # -----------------------------------------------
        # Calculate original 22 features
        # -----------------------------------------------

        features_26 = extract_vibration_features_exact(
            signal
        )

        features_22 = {
            feature: features_26[feature]
            for feature in FEATURES_22
        }

        # -----------------------------------------------
        # Calculate original BPFO and BPFI features
        # -----------------------------------------------

        bpfo_features = extract_bpfo_exact(
            signal,
            rpm
        )

        bpfi_features = extract_bpfi_exact(
            signal,
            rpm
        )

        combined_features = {
            **features_22,
            **bpfo_features,
            **bpfi_features
        }

        # -----------------------------------------------
        # Prepare raw feature vector
        # -----------------------------------------------

        X_raw = pd.DataFrame(
            [
                [
                    combined_features[f]
                    for f in UNIFIED_FEATURES
                ]
            ],
            columns=UNIFIED_FEATURES
        )

        X_reference = pd.DataFrame(
            [
                [
                    float(row[f])
                    for f in UNIFIED_FEATURES
                ]
            ],
            columns=UNIFIED_FEATURES
        )

        # -----------------------------------------------
        # Feature parity
        # -----------------------------------------------

        raw_values = X_raw.to_numpy()

        reference_values = X_reference.to_numpy()

        feature_parity = np.allclose(
            raw_values,
            reference_values,
            rtol=1e-5,
            atol=1e-8
        )

        max_feature_difference = np.max(
            np.abs(
                raw_values - reference_values
            )
        )

        # -----------------------------------------------
        # Prediction from raw features
        # -----------------------------------------------

        raw_prediction = int(
            baseline_pipeline.predict(X_raw)[0]
        )

        raw_probabilities = (
            baseline_pipeline.predict_proba(X_raw)[0]
        )

        positive_index = list(
            baseline_pipeline.classes_
        ).index(1)

        raw_probability = float(
            raw_probabilities[positive_index]
        )

        # -----------------------------------------------
        # Prediction from saved reference features
        # -----------------------------------------------

        reference_prediction = int(
            baseline_pipeline.predict(
                X_reference
            )[0]
        )

        reference_probabilities = (
            baseline_pipeline.predict_proba(
                X_reference
            )[0]
        )

        reference_probability = float(
            reference_probabilities[positive_index]
        )

        prediction_parity = (
            raw_prediction == reference_prediction
        )

        probability_parity = np.allclose(
            raw_probabilities,
            reference_probabilities,
            rtol=1e-5,
            atol=1e-8
        )

        # -----------------------------------------------
        # Retrieve signal-quality audit information
        # -----------------------------------------------

        audit_row = audit_lookup.loc[
            (bearing, filename)
        ]

        results.append({

            "bearing": bearing,

            "file": filename,

            "actual_label": int(row["binary_label"]),

            "raw_prediction": raw_prediction,

            "damaged_probability": raw_probability,

            "feature_parity": feature_parity,

            "prediction_parity": prediction_parity,

            "probability_parity": probability_parity,

            "maximum_feature_difference": (
                max_feature_difference
            ),

            "sample_count": len(signal),

            "timestamp_duration_s": (
                audit_row["timestamp_duration_s"]
            ),

            "anomalous_interval_pct": (
                audit_row["anomalous_interval_pct"]
            ),

            "longest_anomaly_run": (
                audit_row["longest_anomaly_run"]
            )
        })


    except Exception as error:

        failures.append({

            "bearing": bearing,

            "file": filename,

            "error_type": type(error).__name__,

            "error": str(error)
        })


    # -----------------------------------------------
    # Progress report
    # -----------------------------------------------

    if i % 40 == 0 or i == len(test_rows):

        elapsed = time.time() - start_time

        print(
            f"Processed {i}/{len(test_rows)} "
            f"| successful: {len(results)} "
            f"| failed: {len(failures)} "
            f"| elapsed: {elapsed / 60:.2f} minutes"
        )


# ------------------------------------------------------------
# 4. CREATE RESULT TABLES
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

failures_df = pd.DataFrame(failures)


if results_df.empty:
    raise RuntimeError(
        "No test recordings were successfully processed."
    )


# ------------------------------------------------------------
# 5. SUMMARISE PARITY RESULTS
# ------------------------------------------------------------

summary = {

    "total_test_recordings": len(test_rows),

    "successfully_processed": len(results_df),

    "failed_recordings": len(failures_df),

    "feature_parity_passed": int(
        results_df["feature_parity"].sum()
    ),

    "prediction_parity_passed": int(
        results_df["prediction_parity"].sum()
    ),

    "probability_parity_passed": int(
        results_df["probability_parity"].sum()
    ),

    "maximum_absolute_feature_difference": (
        results_df[
            "maximum_feature_difference"
        ].max()
    )
}


summary_df = pd.DataFrame(
    [summary]
)


print("\nFULL TEST SET PARITY SUMMARY")
print("=" * 75)

print(
    summary_df.to_string(index=False)
)


# ------------------------------------------------------------
# 6. CHECK RESULTS BY FAULT CATEGORY
# ------------------------------------------------------------

category_summary = (

    results_df

    .groupby("actual_label")

    .agg(

        recordings=("file", "count"),

        feature_parity_passed=(
            "feature_parity",
            "sum"
        ),

        prediction_parity_passed=(
            "prediction_parity",
            "sum"
        ),

        probability_parity_passed=(
            "probability_parity",
            "sum"
        ),

        maximum_anomaly_percentage=(
            "anomalous_interval_pct",
            "max"
        )
    )

    .reset_index()
)


print("\nPARITY BY CLASS")
print("=" * 75)

print(
    category_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 7. CHECK PARITY ON LONG-ANOMALY RECORDINGS
# ------------------------------------------------------------

long_anomaly_results = results_df[
    results_df["longest_anomaly_run"] >= 50
]


print("\nLONG-ANOMALY RECORDING RESULTS")
print("=" * 75)

print(
    "Recordings with long anomaly runs:",
    len(long_anomaly_results)
)

if not long_anomaly_results.empty:

    print(
        "Feature parity passed:",
        int(long_anomaly_results["feature_parity"].sum())
    )

    print(
        "Prediction parity passed:",
        int(long_anomaly_results["prediction_parity"].sum())
    )

    print(
        "Probability parity passed:",
        int(long_anomaly_results["probability_parity"].sum())
    )


# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

OUTPUT_DIR = (
    Path(PROJECT_ROOT)
    / "results"
    / "raw_inference_verification"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


results_df.to_csv(
    OUTPUT_DIR / "full_test_raw_inference_results.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "full_test_parity_summary.csv",
    index=False
)

category_summary.to_csv(
    OUTPUT_DIR / "parity_by_class.csv",
    index=False
)

failures_df.to_csv(
    OUTPUT_DIR / "raw_inference_failures.csv",
    index=False
)


# ------------------------------------------------------------
# 9. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9B-3 COMPLETED")
print("=" * 75)

if (
    len(results_df) == 320
    and len(failures_df) == 0
    and results_df["feature_parity"].all()
    and results_df["prediction_parity"].all()
    and results_df["probability_parity"].all()
):

    print(
        "SUCCESS: All 320 raw MAT recordings reproduce "
        "their saved feature vectors and predictions."
    )

else:

    print(
        "REVIEW REQUIRED: Some files failed or "
        "parity discrepancies were detected."
    )

print("\nResults saved to:")
print(OUTPUT_DIR)

print("\nNo model training or modification was performed.")

In [ ]:
# ============================================================
# STEP 9B-4
# CREATE A REUSABLE RAW-SIGNAL INFERENCE FUNCTION
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.io import loadmat


print("=" * 75)
print("STEP 9B-4: REUSABLE INFERENCE FUNCTION")
print("=" * 75)


# ------------------------------------------------------------
# 1. DEFINE THE REUSABLE INFERENCE FUNCTION
# ------------------------------------------------------------

def predict_from_signal_window(
    vibration_signal,
    measured_speed,
    model=baseline_pipeline
):
    """
    Predict Healthy/Damaged from a raw vibration signal.

    Parameters
    ----------
    vibration_signal : array-like
        Raw vibration samples.

    measured_speed : float or array-like
        Measured motor RPM. If an array is supplied,
        its mean value is used.

    model : sklearn Pipeline
        Previously trained and packaged 34-feature model.

    Feature construction
    --------------------
    22 original statistical/spectral features:
        Calculated over the entire supplied signal.

    6 BPFO features:
        Calculated using the first 64000 samples.

    6 BPFI features:
        Calculated using the first 64000 samples.

    Returns
    -------
    dict
        Extracted features, prediction and probability.
    """

    # -----------------------------------------------
    # A. Validate the input signal
    # -----------------------------------------------

    signal = np.asarray(
        vibration_signal,
        dtype=np.float64
    ).flatten()

    if signal.size < FS:
        raise ValueError(
            f"At least {FS} samples are required. "
            f"Received {len(signal)}."
        )

    if not np.all(np.isfinite(signal)):
        raise ValueError(
            "Signal contains NaN or infinite values."
        )

    # -----------------------------------------------
    # B. Obtain measured RPM
    # -----------------------------------------------

    speed_values = np.asarray(
        measured_speed,
        dtype=np.float64
    ).reshape(-1)

    if speed_values.size == 0:
        raise ValueError("Speed information is empty.")

    if not np.all(np.isfinite(speed_values)):
        raise ValueError(
            "Speed contains NaN or infinite values."
        )

    rpm = float(np.mean(speed_values))

    if rpm <= 0:
        raise ValueError(
            f"Invalid measured RPM: {rpm}"
        )

    # -----------------------------------------------
    # C. Extract the original 22 features
    # -----------------------------------------------

    features_26 = extract_vibration_features_exact(
        signal
    )

    features_22 = {
        feature: features_26[feature]
        for feature in FEATURES_22
    }

    # -----------------------------------------------
    # D. Extract BPFO and BPFI features
    # -----------------------------------------------

    bpfo_features = extract_bpfo_exact(
        signal,
        rpm
    )

    bpfi_features = extract_bpfi_exact(
        signal,
        rpm
    )

    # -----------------------------------------------
    # E. Combine all 34 features
    # -----------------------------------------------

    combined_features = {
        **features_22,
        **bpfo_features,
        **bpfi_features
    }

    if len(combined_features) != 34:
        raise ValueError(
            f"Expected 34 features, "
            f"generated {len(combined_features)}."
        )

    X = pd.DataFrame(
        [
            [
                combined_features[feature]
                for feature in UNIFIED_FEATURES
            ]
        ],
        columns=UNIFIED_FEATURES
    )

    if not np.all(np.isfinite(X.to_numpy())):
        raise ValueError(
            "Extracted feature vector contains "
            "NaN or infinite values."
        )

    # -----------------------------------------------
    # F. Model inference
    # -----------------------------------------------

    prediction = int(
        model.predict(X)[0]
    )

    probabilities = model.predict_proba(X)[0]

    positive_index = list(
        model.classes_
    ).index(1)

    damaged_probability = float(
        probabilities[positive_index]
    )

    # -----------------------------------------------
    # G. Return the complete result
    # -----------------------------------------------

    return {

        "prediction": prediction,

        "classification": (
            "Damaged"
            if prediction == 1
            else "Healthy"
        ),

        "damaged_probability": damaged_probability,

        "healthy_probability": float(
            probabilities[
                list(model.classes_).index(0)
            ]
        ),

        "rpm": rpm,

        "sample_count": len(signal),

        "feature_vector": X,

        "features": combined_features
    }


print("Reusable inference function created successfully.")


# ------------------------------------------------------------
# 2. TEST IT ON A HELD-OUT LONG-ANOMALY RECORDING
# ------------------------------------------------------------

TARGET_BEARING = "KI18"
TARGET_FILE = "N15_M07_F04_KI18_1.mat"

target_path = (
    Path(RAW_DIR)
    / TARGET_BEARING
    / TARGET_FILE
)

if not target_path.exists():
    raise FileNotFoundError(target_path)


mat = loadmat(
    target_path,
    squeeze_me=False,
    struct_as_record=False
)

data_keys = [
    key for key in mat.keys()
    if not key.startswith("__")
]

if len(data_keys) != 1:
    raise ValueError(
        "Unexpected MATLAB structure."
    )

data = mat[data_keys[0]][0, 0]


# Extract raw vibration and speed.
test_vibration = extract_channel_exact(
    data,
    "vibration_1"
)

test_speed = extract_channel_exact(
    data,
    "speed"
)


# ------------------------------------------------------------
# 3. CALL THE REUSABLE INFERENCE FUNCTION
# ------------------------------------------------------------

result = predict_from_signal_window(
    vibration_signal=test_vibration,
    measured_speed=test_speed
)


# ------------------------------------------------------------
# 4. GET THE SAVED TEST REFERENCE
# ------------------------------------------------------------

reference_row = test_reference_df[
    test_reference_df["file"] == TARGET_FILE
]

if len(reference_row) != 1:
    raise ValueError(
        "Could not identify one unique test reference."
    )

reference_row = reference_row.iloc[0]


reference_features = pd.DataFrame(
    [
        [
            reference_row[feature]
            for feature in UNIFIED_FEATURES
        ]
    ],
    columns=UNIFIED_FEATURES
)


reference_prediction = int(
    baseline_pipeline.predict(
        reference_features
    )[0]
)

reference_probability = float(
    baseline_pipeline.predict_proba(
        reference_features
    )[0][1]
)


# ------------------------------------------------------------
# 5. VERIFY FEATURE AND PREDICTION PARITY
# ------------------------------------------------------------

feature_parity = np.allclose(
    result["feature_vector"].to_numpy(),
    reference_features.to_numpy(),
    rtol=1e-5,
    atol=1e-8
)

prediction_parity = (
    result["prediction"] == reference_prediction
)

probability_parity = np.isclose(
    result["damaged_probability"],
    reference_probability,
    rtol=1e-5,
    atol=1e-8
)


print("\n" + "=" * 75)
print("REUSABLE INFERENCE RESULT")
print("=" * 75)

print("Recording:", TARGET_FILE)
print("Bearing:", TARGET_BEARING)

print("Samples:", result["sample_count"])

print(
    f"Measured RPM: {result['rpm']:.3f}"
)

print(
    "Classification:",
    result["classification"]
)

print(
    f"Healthy probability: "
    f"{result['healthy_probability']:.6f}"
)

print(
    f"Damaged probability: "
    f"{result['damaged_probability']:.6f}"
)


print("\nPARITY VERIFICATION")
print("-" * 65)

print("Feature parity:", feature_parity)

print("Prediction parity:", prediction_parity)

print("Probability parity:", probability_parity)


# ------------------------------------------------------------
# 6. DISPLAY THE FEATURE VECTOR
# ------------------------------------------------------------

print("\nEXTRACTED 34-FEATURE VECTOR")
print("-" * 65)

display(result["feature_vector"].T)


# ------------------------------------------------------------
# 7. FINAL CHECK
# ------------------------------------------------------------

if not all([
    feature_parity,
    prediction_parity,
    probability_parity
]):

    raise AssertionError(
        "Reusable inference failed the parity check."
    )


print("\n" + "=" * 75)
print("STEP 9B-4 COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    "The reusable inference function reproduces "
    "the packaged model's output."
)

print("No model retraining or modification performed.")

In [ ]:
# ============================================================
# STEP 9C-2
# TEST STANDALONE INFERENCE MODULE
# ============================================================

import sys
import importlib
import numpy as np
import pandas as pd

from pathlib import Path


print("=" * 75)
print("STEP 9C-2: STANDALONE MODULE VERIFICATION")
print("=" * 75)


# ------------------------------------------------------------
# 1. SET PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

SRC_DIR = PROJECT_ROOT / "src"

RAW_DIR = PROJECT_ROOT / "data" / "raw"

BASELINE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "baseline"
)

TEST_FEATURES_CSV = (
    BASELINE_DIR / "paderborn_integrated_test_features.csv"
)

if not SRC_DIR.exists():
    raise FileNotFoundError(SRC_DIR)


# ------------------------------------------------------------
# 2. IMPORT THE STANDALONE MODULE
# ------------------------------------------------------------

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import paderborn_inference as pbi

# Reload to ensure the latest saved file is imported.
importlib.reload(pbi)

print("\nModule imported successfully.")
print("Module location:", pbi.__file__)


# ------------------------------------------------------------
# 3. LOAD THE PACKAGED MODEL
# ------------------------------------------------------------

model = pbi.load_model()

print("\nPackaged model loaded successfully.")
print("Model input features:", model.n_features_in_)
print("Module feature count:", len(pbi.FEATURE_NAMES))


# ------------------------------------------------------------
# 4. SELECT THE HELD-OUT RECORDING
# ------------------------------------------------------------

TARGET_BEARING = "KI18"

TARGET_FILE = "N15_M07_F04_KI18_1.mat"

mat_path = (
    RAW_DIR / TARGET_BEARING / TARGET_FILE
)

if not mat_path.exists():
    raise FileNotFoundError(mat_path)


# ------------------------------------------------------------
# 5. RUN INFERENCE USING THE STANDALONE MODULE
# ------------------------------------------------------------

result = pbi.predict_from_mat(
    mat_file=mat_path,
    model=model
)


# ------------------------------------------------------------
# 6. LOAD THE SAVED FEATURE REFERENCE
# ------------------------------------------------------------

reference_df = pd.read_csv(
    TEST_FEATURES_CSV
)

matching_rows = reference_df[
    reference_df["file"] == TARGET_FILE
]

if len(matching_rows) != 1:
    raise ValueError(
        f"Expected one reference row, found {len(matching_rows)}"
    )

reference_row = matching_rows.iloc[0]

reference_features = pd.DataFrame(
    [[
        float(reference_row[name])
        for name in pbi.FEATURE_NAMES
    ]],
    columns=pbi.FEATURE_NAMES
)


# ------------------------------------------------------------
# 7. VERIFY FEATURE PARITY
# ------------------------------------------------------------

feature_difference = (
    result["feature_vector"].to_numpy()
    - reference_features.to_numpy()
)

max_difference = np.max(
    np.abs(feature_difference)
)

feature_parity = np.allclose(
    result["feature_vector"].to_numpy(),
    reference_features.to_numpy(),
    rtol=1e-5,
    atol=1e-8
)


# ------------------------------------------------------------
# 8. VERIFY PREDICTION PARITY
# ------------------------------------------------------------

reference_prediction = int(
    model.predict(reference_features)[0]
)

reference_probabilities = (
    model.predict_proba(reference_features)[0]
)

damaged_index = list(model.classes_).index(1)

reference_damaged_probability = float(
    reference_probabilities[damaged_index]
)

prediction_parity = (
    result["prediction"] == reference_prediction
)

probability_parity = np.isclose(
    result["damaged_probability"],
    reference_damaged_probability,
    rtol=1e-5,
    atol=1e-8
)


# ------------------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nRAW MAT FILE")
print("-" * 65)

print("File:", result["file"])
print("Samples:", result["sample_count"])
print("RPM:", result["rpm"])


print("\nPREDICTION")
print("-" * 65)

print("Classification:", result["classification"])

print(
    "Healthy probability:",
    result["healthy_probability"]
)

print(
    "Damaged probability:",
    result["damaged_probability"]
)


print("\nPARITY VERIFICATION")
print("-" * 65)

print("Feature parity:", feature_parity)

print("Prediction parity:", prediction_parity)

print("Probability parity:", probability_parity)

print("Maximum absolute feature difference:", max_difference)


# ------------------------------------------------------------
# 10. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9C-2 SUMMARY")
print("=" * 75)

if all([
    feature_parity,
    prediction_parity,
    probability_parity
]):

    print("SUCCESS: Standalone module passed all checks.")

else:

    print("WARNING: Standalone module parity check failed.")

print("\nNo model training or modification performed.")

In [ ]:

# ============================================================
# STEP 9C-3
# FULL TEST SET STANDALONE MODULE VERIFICATION
# ============================================================

import sys
import time
import importlib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)


print("=" * 75)
print("STEP 9C-3: FULL TEST SET STANDALONE MODULE VALIDATION")
print("=" * 75)


# ------------------------------------------------------------
# 1. DEFINE PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"

SRC_DIR = PROJECT_ROOT / "src"

TEST_FEATURES_CSV = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "baseline"
    / "paderborn_integrated_test_features.csv"
)

AUDIT_CSV = (
    PROJECT_ROOT
    / "results"
    / "signal_quality_audit"
    / "recording_quality_audit.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "standalone_module_verification"
)


# ------------------------------------------------------------
# 2. VERIFY REQUIRED FILES
# ------------------------------------------------------------

if not RAW_DIR.exists():
    raise FileNotFoundError(RAW_DIR)

if not SRC_DIR.exists():
    raise FileNotFoundError(SRC_DIR)

if not TEST_FEATURES_CSV.exists():
    raise FileNotFoundError(TEST_FEATURES_CSV)


# ------------------------------------------------------------
# 3. IMPORT THE STANDALONE MODULE
# ------------------------------------------------------------

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import paderborn_inference as pbi

importlib.reload(pbi)

print("\nStandalone module imported:")
print(pbi.__file__)


# ------------------------------------------------------------
# 4. LOAD THE PACKAGED MODEL
# ------------------------------------------------------------

model = pbi.load_model()

print("\nModel loaded successfully.")

print("Model input features:", model.n_features_in_)

print("Module feature count:", len(pbi.FEATURE_NAMES))


if model.n_features_in_ != 34:
    raise ValueError(
        "The packaged model does not expect 34 features."
    )

if hasattr(model, "feature_names_in_"):

    if list(model.feature_names_in_) != pbi.FEATURE_NAMES:

        raise ValueError(
            "Model feature order differs from standalone module."
        )


# ------------------------------------------------------------
# 5. LOAD TEST REFERENCE FEATURES
# ------------------------------------------------------------

test_rows = pd.read_csv(
    TEST_FEATURES_CSV
)

print("\nTest recordings loaded:", len(test_rows))


if len(test_rows) != 320:
    raise ValueError(
        f"Expected 320 test recordings, "
        f"found {len(test_rows)}."
    )


# Verify required columns.
required_columns = [
    "bearing",
    "file",
    "binary_label"
] + pbi.FEATURE_NAMES

missing_columns = [
    column
    for column in required_columns
    if column not in test_rows.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns in test reference CSV: "
        + ", ".join(missing_columns)
    )


# ------------------------------------------------------------
# 6. LOAD SIGNAL QUALITY AUDIT
# ------------------------------------------------------------

if AUDIT_CSV.exists():

    audit_df = pd.read_csv(AUDIT_CSV)

    required_audit_columns = [
        "bearing",
        "file",
        "anomalous_interval_pct",
        "longest_anomaly_run",
        "timestamp_duration_s"
    ]

    missing_audit_columns = [
        col for col in required_audit_columns
        if col not in audit_df.columns
    ]

    if missing_audit_columns:
        raise ValueError(
            "Signal audit file is missing columns: "
            + ", ".join(missing_audit_columns)
        )

    # Create a lookup dictionary.
    audit_lookup = {}

    for _, audit_row in audit_df.iterrows():

        key = (
            str(audit_row["bearing"]),
            str(audit_row["file"])
        )

        if key in audit_lookup:
            raise ValueError(
                f"Duplicate audit entry: {key}"
            )

        audit_lookup[key] = audit_row

    print("Signal quality audit loaded:", len(audit_df))

else:

    print(
        "WARNING: Signal quality audit CSV not found. "
        "Audit statistics will be recorded as NaN."
    )

    audit_lookup = {}


# ------------------------------------------------------------
# 7. INITIALIZE RESULT CONTAINERS
# ------------------------------------------------------------

# These must be initialized before the processing loop.

results = []

failures = []

start_time = time.time()

total_recordings = len(test_rows)


# ------------------------------------------------------------
# 8. PROCESS ALL 320 TEST RECORDINGS
# ------------------------------------------------------------

for i, (_, row) in enumerate(
    test_rows.iterrows(),
    start=1
):

    bearing = str(row["bearing"])
    filename = str(row["file"])

    mat_path = (
        RAW_DIR
        / bearing
        / filename
    )

    try:

        # -----------------------------------------------
        # A. Check file existence
        # -----------------------------------------------

        if not mat_path.exists():

            raise FileNotFoundError(
                f"MAT file not found: {mat_path}"
            )


        # -----------------------------------------------
        # B. Run the standalone module
        # -----------------------------------------------

        result = pbi.predict_from_mat(
            mat_file=mat_path,
            model=model
        )


        # -----------------------------------------------
        # C. Create reference feature vector
        # -----------------------------------------------

        reference_values = np.asarray(
            [
                float(row[feature])
                for feature in pbi.FEATURE_NAMES
            ],
            dtype=np.float64
        ).reshape(1, -1)


        reference_X = pd.DataFrame(
            reference_values,
            columns=pbi.FEATURE_NAMES
        )


        module_values = (
            result["feature_vector"].to_numpy()
        )


        # -----------------------------------------------
        # D. Verify feature parity
        # -----------------------------------------------

        feature_parity = np.allclose(
            module_values,
            reference_values,
            rtol=1e-5,
            atol=1e-8
        )

        maximum_difference = np.max(
            np.abs(
                module_values - reference_values
            )
        )


        # -----------------------------------------------
        # E. Verify prediction parity
        # -----------------------------------------------

        reference_prediction = int(
            model.predict(reference_X)[0]
        )

        prediction_parity = (
            result["prediction"]
            == reference_prediction
        )


        # -----------------------------------------------
        # F. Verify probability parity
        # -----------------------------------------------

        reference_probabilities = (
            model.predict_proba(reference_X)[0]
        )

        positive_index = list(
            model.classes_
        ).index(1)

        reference_damaged_probability = float(
            reference_probabilities[positive_index]
        )

        probability_parity = np.isclose(
            result["damaged_probability"],
            reference_damaged_probability,
            rtol=1e-5,
            atol=1e-8
        )


        # -----------------------------------------------
        # G. Retrieve signal quality metadata
        # -----------------------------------------------

        audit_key = (
            bearing,
            filename
        )

        audit_row = audit_lookup.get(audit_key)

        if audit_row is not None:

            anomaly_pct = float(
                audit_row["anomalous_interval_pct"]
            )

            longest_run = int(
                audit_row["longest_anomaly_run"]
            )

            duration = float(
                audit_row["timestamp_duration_s"]
            )

        else:

            anomaly_pct = np.nan
            longest_run = np.nan
            duration = np.nan


        # -----------------------------------------------
        # H. Save successful result
        # -----------------------------------------------

        results.append({

            "bearing": bearing,

            "file": filename,

            "actual_label": int(
                row["binary_label"]
            ),

            "module_prediction": (
                result["prediction"]
            ),

            "damaged_probability": (
                result["damaged_probability"]
            ),

            "reference_prediction": (
                reference_prediction
            ),

            "feature_parity": (
                feature_parity
            ),

            "prediction_parity": (
                prediction_parity
            ),

            "probability_parity": (
                probability_parity
            ),

            "maximum_feature_difference": (
                maximum_difference
            ),

            "sample_count": (
                result["sample_count"]
            ),

            "timestamp_duration_s": duration,

            "anomalous_interval_pct": anomaly_pct,

            "longest_anomaly_run": longest_run

        })


    except Exception as error:

        # Record the failed recording without stopping
        # the remaining test-set processing.

        failures.append({

            "bearing": bearing,

            "file": filename,

            "error_type": type(error).__name__,

            "error": str(error)

        })


    # -----------------------------------------------
    # I. Progress report
    # -----------------------------------------------

    if i % 40 == 0 or i == total_recordings:

        elapsed = time.time() - start_time

        print(
            f"Processed {i}/{total_recordings} "
            f"| successful: {len(results)} "
            f"| failed: {len(failures)} "
            f"| elapsed: {elapsed / 60:.2f} minutes"
        )


# ------------------------------------------------------------
# 9. CREATE RESULT DATAFRAMES
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

failures_df = pd.DataFrame(failures)


if results_df.empty:

    raise RuntimeError(
        "No recordings were processed successfully."
    )


# ------------------------------------------------------------
# 10. FULL TEST SET PARITY SUMMARY
# ------------------------------------------------------------

summary = {

    "total_test_recordings": total_recordings,

    "successfully_processed": len(results_df),

    "failed_recordings": len(failures_df),

    "feature_parity_passed": int(
        results_df["feature_parity"].sum()
    ),

    "prediction_parity_passed": int(
        results_df["prediction_parity"].sum()
    ),

    "probability_parity_passed": int(
        results_df["probability_parity"].sum()
    ),

    "maximum_absolute_feature_difference": (
        results_df[
            "maximum_feature_difference"
        ].max()
    )

}


summary_df = pd.DataFrame(
    [summary]
)


print("\nFULL TEST SET PARITY SUMMARY")
print("=" * 75)

print(
    summary_df.to_string(index=False)
)


# ------------------------------------------------------------
# 11. CLASSIFICATION PERFORMANCE
# ------------------------------------------------------------

y_true = results_df["actual_label"]

y_pred = results_df["module_prediction"]


print("\nCLASSIFICATION PERFORMANCE")
print("=" * 75)

print(
    "Accuracy:",
    accuracy_score(y_true, y_pred)
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(y_true, y_pred)
)

print(
    "F1 score:",
    f1_score(y_true, y_pred, zero_division=0)
)

print(
    "\nConfusion matrix:"
)

print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )
)


# ------------------------------------------------------------
# 12. PARITY BY FAULT CLASS
# ------------------------------------------------------------

category_summary = (

    results_df

    .groupby("actual_label")

    .agg(

        recordings=("file", "count"),

        feature_parity_passed=(
            "feature_parity",
            "sum"
        ),

        prediction_parity_passed=(
            "prediction_parity",
            "sum"
        ),

        probability_parity_passed=(
            "probability_parity",
            "sum"
        ),

        maximum_anomaly_percentage=(
            "anomalous_interval_pct",
            "max"
        )

    )

    .reset_index()

)


print("\nPARITY BY CLASS")
print("=" * 75)

print(
    category_summary.to_string(index=False)
)


# ------------------------------------------------------------
# 13. CHECK LONG-ANOMALY RECORDINGS
# ------------------------------------------------------------

long_anomaly_df = results_df[
    results_df["longest_anomaly_run"] >= 50
]


print("\nLONG-ANOMALY RECORDING RESULTS")
print("=" * 75)

print(
    "Recordings with long anomaly runs:",
    len(long_anomaly_df)
)


if not long_anomaly_df.empty:

    print(
        "Feature parity:",
        int(long_anomaly_df["feature_parity"].sum()),
        "/",
        len(long_anomaly_df)
    )

    print(
        "Prediction parity:",
        int(long_anomaly_df["prediction_parity"].sum()),
        "/",
        len(long_anomaly_df)
    )

    print(
        "Probability parity:",
        int(long_anomaly_df["probability_parity"].sum()),
        "/",
        len(long_anomaly_df)
    )


# ------------------------------------------------------------
# 14. DISPLAY FAILURES
# ------------------------------------------------------------

print("\nFAILED RECORDINGS")
print("=" * 75)

if failures_df.empty:

    print("No failures.")

else:

    display(failures_df)


# ------------------------------------------------------------
# 15. SAVE RESULTS
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

results_df.to_csv(
    OUTPUT_DIR / "standalone_module_test_results.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "standalone_module_summary.csv",
    index=False
)

category_summary.to_csv(
    OUTPUT_DIR / "standalone_module_parity_by_class.csv",
    index=False
)

failures_df.to_csv(
    OUTPUT_DIR / "standalone_module_failures.csv",
    index=False
)


print("\nResults saved to:")
print(OUTPUT_DIR)


# ------------------------------------------------------------
# 16. FINAL VERIFICATION
# ------------------------------------------------------------

all_passed = (

    len(results_df) == 320

    and len(failures_df) == 0

    and results_df["feature_parity"].all()

    and results_df["prediction_parity"].all()

    and results_df["probability_parity"].all()

)


print("\n" + "=" * 75)
print("STEP 9C-3 COMPLETED")
print("=" * 75)

print(
    "All 320 standalone module checks passed:",
    all_passed
)

print(
    "No model retraining or modification performed."
)


In [ ]:
# ============================================================
# STEP 9C-4
# SIMULATED EDGE SIGNAL BUFFER AND INFERENCE
# ============================================================

import sys
import importlib
import time
import numpy as np
import pandas as pd

from pathlib import Path


print("=" * 75)
print("STEP 9C-4: SIMULATED EDGE SIGNAL BUFFER")
print("=" * 75)


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"

SRC_DIR = PROJECT_ROOT / "src"

TEST_CSV = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "baseline"
    / "paderborn_integrated_test_features.csv"
)

TARGET_BEARING = "KI18"
TARGET_FILE = "N15_M07_F04_KI18_1.mat"

CHUNK_SIZE = 4096


# ------------------------------------------------------------
# 2. IMPORT MODULE AND LOAD MODEL
# ------------------------------------------------------------

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import paderborn_inference as pbi

importlib.reload(pbi)

model = pbi.load_model()

print("\nStandalone module loaded.")
print("Model input features:", model.n_features_in_)


# ------------------------------------------------------------
# 3. LOAD THE TEST RECORDING
# ------------------------------------------------------------

mat_path = (
    RAW_DIR
    / TARGET_BEARING
    / TARGET_FILE
)

if not mat_path.exists():
    raise FileNotFoundError(mat_path)


load_start = time.perf_counter()

original_signal, measured_speed = (
    pbi.load_signal_and_rpm(mat_path)
)

mat_load_time = (
    time.perf_counter() - load_start
)


print("\nSOURCE RECORDING")
print("-" * 65)

print("File:", TARGET_FILE)
print("Bearing:", TARGET_BEARING)
print("Original samples:", len(original_signal))
print("Measured RPM:", np.mean(measured_speed))


# ------------------------------------------------------------
# 4. CREATE A SIGNAL BUFFER CLASS
# ------------------------------------------------------------

class SignalBuffer:
    """
    Accumulates incoming signal chunks.

    The complete signal is reconstructed when finish()
    is called. No sample is intentionally removed or altered.
    """

    def __init__(self):

        self.chunks = []
        self.total_samples = 0
        self.closed = False

    def add_chunk(self, chunk):

        if self.closed:
            raise RuntimeError(
                "Cannot add data to a finalized buffer."
            )

        chunk = np.asarray(
            chunk,
            dtype=np.float64
        ).reshape(-1)

        if chunk.size == 0:
            return

        if not np.all(np.isfinite(chunk)):
            raise ValueError(
                "Incoming chunk contains invalid values."
            )

        self.chunks.append(chunk.copy())

        self.total_samples += len(chunk)

    def finish(self):

        if self.closed:
            raise RuntimeError(
                "Buffer has already been finalized."
            )

        if not self.chunks:
            raise ValueError(
                "No signal data was received."
            )

        complete_signal = np.concatenate(
            self.chunks
        )

        self.closed = True

        return complete_signal


# ------------------------------------------------------------
# 5. SIMULATE INCOMING SIGNAL CHUNKS
# ------------------------------------------------------------

buffer = SignalBuffer()

number_of_chunks = int(
    np.ceil(len(original_signal) / CHUNK_SIZE)
)

print("\nSIMULATED SIGNAL TRANSMISSION")
print("-" * 65)

print("Chunk size:", CHUNK_SIZE)
print("Expected number of chunks:", number_of_chunks)


replay_start = time.perf_counter()

for start in range(
    0,
    len(original_signal),
    CHUNK_SIZE
):

    end = min(
        start + CHUNK_SIZE,
        len(original_signal)
    )

    incoming_chunk = original_signal[start:end]

    buffer.add_chunk(incoming_chunk)


buffered_signal = buffer.finish()

replay_time = (
    time.perf_counter() - replay_start
)


print("\nBUFFER INFORMATION")
print("-" * 65)

print("Chunks received:", len(buffer.chunks))

print("Total samples received:", buffer.total_samples)

print("Reconstructed samples:", len(buffered_signal))

print(
    "Reconstructed signal identical:",
    np.array_equal(
        original_signal,
        buffered_signal
    )
)


# ------------------------------------------------------------
# 6. VERIFY NO SAMPLES WERE LOST OR MODIFIED
# ------------------------------------------------------------

buffer_integrity = (

    len(original_signal) == len(buffered_signal)

    and np.array_equal(
        original_signal,
        buffered_signal
    )

)

if not buffer_integrity:
    raise AssertionError(
        "Signal buffer failed data-integrity verification."
    )


# ------------------------------------------------------------
# 7. RUN INFERENCE ON THE RECONSTRUCTED SIGNAL
# ------------------------------------------------------------

inference_start = time.perf_counter()

result = pbi.predict_from_signal(
    vibration_signal=buffered_signal,
    measured_speed=measured_speed,
    model=model
)

inference_time = (
    time.perf_counter() - inference_start
)


# ------------------------------------------------------------
# 8. LOAD SAVED TEST REFERENCE
# ------------------------------------------------------------

reference_df = pd.read_csv(TEST_CSV)

reference_rows = reference_df[
    reference_df["file"] == TARGET_FILE
]

if len(reference_rows) != 1:
    raise ValueError(
        "Unique test reference was not found."
    )

reference_row = reference_rows.iloc[0]


X_reference = pd.DataFrame(
    [[
        float(reference_row[name])
        for name in pbi.FEATURE_NAMES
    ]],
    columns=pbi.FEATURE_NAMES
)


# ------------------------------------------------------------
# 9. VERIFY FEATURE AND PREDICTION PARITY
# ------------------------------------------------------------

feature_parity = np.allclose(
    result["feature_vector"].to_numpy(),
    X_reference.to_numpy(),
    rtol=1e-5,
    atol=1e-8
)

reference_prediction = int(
    model.predict(X_reference)[0]
)

reference_probability = float(
    model.predict_proba(X_reference)[0][1]
)


prediction_parity = (
    result["prediction"] == reference_prediction
)

probability_parity = np.isclose(
    result["damaged_probability"],
    reference_probability,
    rtol=1e-5,
    atol=1e-8
)


# ------------------------------------------------------------
# 10. DISPLAY EDGE REPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SIMULATED EDGE INFERENCE RESULT")
print("=" * 75)

print("Recording:", TARGET_FILE)

print("Chunks received:", len(buffer.chunks))

print("Samples processed:", len(buffered_signal))

print("RPM:", result["rpm"])

print("Classification:", result["classification"])

print(
    "Healthy probability:",
    result["healthy_probability"]
)

print(
    "Damaged probability:",
    result["damaged_probability"]
)


# ------------------------------------------------------------
# 11. PERFORMANCE MEASUREMENTS
# ------------------------------------------------------------

total_processing_time = (
    mat_load_time
    + replay_time
    + inference_time
)


print("\nPROCESSING TIME")
print("-" * 65)

print(
    f"MAT file loading: "
    f"{mat_load_time * 1000:.3f} ms"
)

print(
    f"Chunk buffering: "
    f"{replay_time * 1000:.3f} ms"
)

print(
    f"Feature extraction + prediction: "
    f"{inference_time * 1000:.3f} ms"
)

print(
    f"Total measured processing: "
    f"{total_processing_time * 1000:.3f} ms"
)


# ------------------------------------------------------------
# 12. PARITY SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9C-4 VERIFICATION")
print("=" * 75)

print("Buffer integrity:", buffer_integrity)

print("Feature parity:", feature_parity)

print("Prediction parity:", prediction_parity)

print("Probability parity:", probability_parity)


all_passed = all([

    buffer_integrity,

    feature_parity,

    prediction_parity,

    probability_parity

])


print("\nOverall result:", all_passed)

if not all_passed:

    raise AssertionError(
        "Simulated edge verification failed."
    )


print("\nSTEP 9C-4 COMPLETED SUCCESSFULLY")

print(
    "The reconstructed signal produced the same "
    "result as the original standalone inference pipeline."
)

print(
    "\nNote: This is accelerated chunk replay, "
    "not a real-time hardware benchmark."
)

In [ ]:
# ============================================================
# STEP 9C-6
# AUDIT CHRONOLOGICAL SPACING OF SEQUENCE MEASUREMENTS
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.io import loadmat
from datetime import datetime


print("=" * 75)
print("STEP 9C-6: MEASUREMENT CHRONOLOGY AUDIT")
print("=" * 75)


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

BEARING = "K001"
CONDITION = "N09_M07_F10"

MEASUREMENTS = range(1, 7)

FS = 64000

RAW_DIR = Path(RAW_DIR)


# ------------------------------------------------------------
# 2. HELPER: UNWRAP MATLAB VALUES
# ------------------------------------------------------------

def unwrap_mat_value(value):

    if isinstance(value, np.ndarray):

        if value.size == 1:
            return unwrap_mat_value(value.item())

        if value.dtype.kind in ("U", "S"):
            return "".join(
                map(str, value.reshape(-1))
            ).strip()

        if value.size <= 10:
            return [
                unwrap_mat_value(x)
                for x in value.reshape(-1)
            ]

        return f"<Array shape={value.shape}>"

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace"
        ).strip()

    if isinstance(value, np.generic):
        return value.item()

    return value


# ------------------------------------------------------------
# 3. HELPER: RECURSIVELY INSPECT MATLAB METADATA
# ------------------------------------------------------------

def collect_metadata(obj, prefix="", depth=0, output=None):

    if output is None:
        output = {}

    if depth > 8:
        return output

    if hasattr(obj, "_fieldnames"):

        for field in obj._fieldnames:

            value = getattr(obj, field)

            field_path = (
                f"{prefix}.{field}"
                if prefix else field
            )

            collect_metadata(
                value,
                field_path,
                depth + 1,
                output
            )

    elif isinstance(obj, np.ndarray):

        if obj.size == 1:

            collect_metadata(
                obj.item(),
                prefix,
                depth + 1,
                output
            )

        elif obj.dtype.kind in ("U", "S"):

            output[prefix] = "".join(
                map(str, obj.reshape(-1))
            ).strip()

        elif obj.dtype == object and obj.size <= 10:

            for index, item in enumerate(obj.reshape(-1)):

                collect_metadata(
                    item,
                    f"{prefix}[{index}]",
                    depth + 1,
                    output
                )

        else:

            output[prefix] = (
                f"<Array shape={obj.shape}, "
                f"dtype={obj.dtype}>"
            )

    else:

        output[prefix] = unwrap_mat_value(obj)

    return output


# ------------------------------------------------------------
# 4. LOAD THE SIX CONSECUTIVE RECORDINGS
# ------------------------------------------------------------

recording_records = []
metadata_records = []

for measurement in MEASUREMENTS:

    filename = (
        f"{CONDITION}_{BEARING}_{measurement}.mat"
    )

    mat_path = (
        RAW_DIR / BEARING / filename
    )

    if not mat_path.exists():

        print("Missing file:", filename)
        continue

    mat = loadmat(
        mat_path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [
        key for key in mat
        if not key.startswith("__")
    ]

    if len(data_keys) != 1:
        raise ValueError(
            f"Unexpected MAT structure: {filename}"
        )

    data = mat[data_keys[0]][0, 0]


    # --------------------------------------------------------
    # 5. EXTRACT VIBRATION SIGNAL AND TIME AXIS
    # --------------------------------------------------------

    y_channels = np.asarray(data.Y).flatten()

    vibration_channel = None
    speed_channel = None

    for channel in y_channels:

        name = unwrap_mat_value(
            channel.Name
        )

        if str(name) == "vibration_1":
            vibration_channel = channel

        elif str(name) == "speed":
            speed_channel = channel

    if vibration_channel is None:
        raise ValueError(
            f"Vibration channel missing: {filename}"
        )

    vibration = np.asarray(
        vibration_channel.Data,
        dtype=float
    ).squeeze()

    x_index = int(
        np.asarray(
            vibration_channel.XIndex
        ).squeeze()
    )

    vibration_time_axis = np.asarray(
        data.X,
        dtype=object
    ).flatten()[x_index - 1]

    vibration_timestamps = np.asarray(
        vibration_time_axis.Data,
        dtype=float
    ).squeeze()

    if speed_channel is None:
        raise ValueError(
            f"Speed channel missing: {filename}"
        )

    speed = np.asarray(
        speed_channel.Data,
        dtype=float
    ).squeeze()


    # --------------------------------------------------------
    # 6. SIGNAL TIMING STATISTICS
    # --------------------------------------------------------

    dt = np.diff(vibration_timestamps)

    recording_duration = (
        vibration_timestamps[-1]
        - vibration_timestamps[0]
    )

    median_dt = np.median(dt)

    effective_fs = 1 / median_dt

    anomaly_mask = (
        np.abs(dt - 1 / FS)
        > 0.05 * (1 / FS)
    )


    # --------------------------------------------------------
    # 7. EXTRACT DESCRIPTION AND INFO METADATA
    # --------------------------------------------------------

    description = getattr(
        data,
        "Description",
        None
    )

    info = getattr(
        data,
        "Info",
        None
    )

    if description is not None:

        description_metadata = collect_metadata(
            description,
            prefix="Description"
        )

        for key, value in description_metadata.items():

            metadata_records.append({

                "measurement": measurement,

                "file": filename,

                "metadata_field": key,

                "metadata_value": str(value)
            })


    if info is not None:

        info_metadata = collect_metadata(
            info,
            prefix="Info"
        )

        for key, value in info_metadata.items():

            metadata_records.append({

                "measurement": measurement,

                "file": filename,

                "metadata_field": key,

                "metadata_value": str(value)
            })


    # --------------------------------------------------------
    # 8. RECORD SUMMARY
    # --------------------------------------------------------

    recording_records.append({

        "measurement": measurement,

        "file": filename,

        "vibration_samples": len(vibration),

        "speed_samples": len(speed),

        "recording_duration_s": recording_duration,

        "start_timestamp": vibration_timestamps[0],

        "end_timestamp": vibration_timestamps[-1],

        "median_interval_us": median_dt * 1e6,

        "effective_sampling_rate_hz": effective_fs,

        "anomalous_interval_count": int(
            np.sum(anomaly_mask)
        ),

        "anomalous_interval_pct": (
            np.mean(anomaly_mask) * 100
        ),

        "mean_speed": np.mean(speed)

    })


# ------------------------------------------------------------
# 9. DISPLAY MEASUREMENT SUMMARY
# ------------------------------------------------------------

recording_df = pd.DataFrame(
    recording_records
)

metadata_df = pd.DataFrame(
    metadata_records
)


print("\nRECORDING SUMMARY")
print("=" * 75)

display(recording_df)


# ------------------------------------------------------------
# 10. DISPLAY ACQUISITION METADATA
# ------------------------------------------------------------

print("\nMAT FILE DESCRIPTION / INFO METADATA")
print("=" * 75)

if not metadata_df.empty:

    display(metadata_df)

else:

    print("No description or info metadata found.")


# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "sequence_chronology_audit"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

recording_df.to_csv(
    OUTPUT_DIR / "K001_measurement_timing.csv",
    index=False
)

metadata_df.to_csv(
    OUTPUT_DIR / "K001_acquisition_metadata.csv",
    index=False
)


print("\nResults saved to:")
print(OUTPUT_DIR)

print("\n" + "=" * 75)
print("STEP 9C-6 COMPLETED")
print("=" * 75)

print(
    "No original recordings or sequence datasets were modified."
)

In [ ]:
# ============================================================
# STEP 9C-7
# CALCULATE REAL-TIME GAPS BETWEEN MEASUREMENTS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 75)
print("STEP 9C-7: ACTUAL MEASUREMENT TIME INTERVALS")
print("=" * 75)


# ------------------------------------------------------------
# 1. VERIFY METADATA FROM STEP 9C-6
# ------------------------------------------------------------

if "metadata_df" not in globals():
    raise RuntimeError(
        "metadata_df is missing. Run Step 9C-6 first."
    )


# ------------------------------------------------------------
# 2. EXTRACT IMPORTANT METADATA FIELDS
# ------------------------------------------------------------

required_fields = [
    "Description.General.DateTime",
    "Description.Measurement.StartTimestamp",
    "Description.Measurement.StopTimestamp"
]

available_fields = metadata_df[
    "metadata_field"
].unique()

for field in required_fields:

    if field not in available_fields:
        raise ValueError(
            f"Required metadata field not found: {field}"
        )


selected_metadata = metadata_df[
    metadata_df["metadata_field"].isin(required_fields)
].copy()


# ------------------------------------------------------------
# 3. CONVERT METADATA INTO ONE ROW PER MEASUREMENT
# ------------------------------------------------------------

measurement_metadata = selected_metadata.pivot_table(
    index=["measurement", "file"],
    columns="metadata_field",
    values="metadata_value",
    aggfunc="first"
).reset_index()


measurement_metadata.columns.name = None


# ------------------------------------------------------------
# 4. PARSE ACTUAL RECORDING DATE/TIME
# ------------------------------------------------------------

measurement_metadata["acquisition_datetime"] = (
    pd.to_datetime(
        measurement_metadata[
            "Description.General.DateTime"
        ],
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce"
    )
)


measurement_metadata = measurement_metadata.sort_values(
    "measurement"
).reset_index(drop=True)


if measurement_metadata["acquisition_datetime"].isna().any():

    print(
        "WARNING: Some acquisition date/time values "
        "could not be parsed."
    )


# ------------------------------------------------------------
# 5. CALCULATE TIME DIFFERENCE BETWEEN RECORDINGS
# ------------------------------------------------------------

measurement_metadata["start_to_start_gap_s"] = (
    measurement_metadata[
        "acquisition_datetime"
    ].diff().dt.total_seconds()
)

# Approximate gap after the previous four-second recording.
measurement_metadata["approx_gap_after_previous_s"] = (
    measurement_metadata["start_to_start_gap_s"] - 4.0
)


# ------------------------------------------------------------
# 6. CONVERT RELATIVE START/STOP TIMESTAMPS
# ------------------------------------------------------------

measurement_metadata[
    "relative_start_timestamp_s"
] = pd.to_numeric(
    measurement_metadata[
        "Description.Measurement.StartTimestamp"
    ],
    errors="coerce"
)

measurement_metadata[
    "relative_stop_timestamp_s"
] = pd.to_numeric(
    measurement_metadata[
        "Description.Measurement.StopTimestamp"
    ],
    errors="coerce"
)


# ------------------------------------------------------------
# 7. DISPLAY CHRONOLOGICAL RESULTS
# ------------------------------------------------------------

display_columns = [
    "measurement",
    "file",
    "acquisition_datetime",
    "relative_start_timestamp_s",
    "relative_stop_timestamp_s",
    "start_to_start_gap_s",
    "approx_gap_after_previous_s"
]


print("\nCHRONOLOGICAL MEASUREMENT TABLE")
print("=" * 75)

display(
    measurement_metadata[display_columns]
)


# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

print("\nSUMMARY")
print("=" * 75)

print(
    "Number of measurements:",
    len(measurement_metadata)
)

print(
    "Median start-to-start interval:",
    measurement_metadata[
        "start_to_start_gap_s"
    ].median(),
    "seconds"
)

print(
    "Median estimated gap between recordings:",
    measurement_metadata[
        "approx_gap_after_previous_s"
    ].median(),
    "seconds"
)


print("\nSTEP 9C-7 COMPLETED")

In [ ]:

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn
import sys

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

FILES = [
    "stage1_scaled_sequences.joblib",
    "stage2_scaled_sequences.joblib"
]

for filename in FILES:
    matches = list(PROJECT_ROOT.rglob(filename))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}, "
            f"but found: {matches}"
        )

    path = matches[0]
    data = joblib.load(path)

    print("\n" + "=" * 65)
    print("FILE:", filename)
    print("LOCATION:", path)
    print("TOP-LEVEL KEYS:", list(data.keys()))

    for feature_set, split_data in data.items():
        print("\nFEATURE SET:", feature_set)

        if not isinstance(split_data, dict):
            print("Unexpected structure:", type(split_data))
            continue

        print("SPLIT KEYS:", list(split_data.keys()))

        for key, value in split_data.items():
            shape = getattr(value, "shape", None)
            dtype = getattr(value, "dtype", None)

            print(f"\n{key}:")
            print("  Type:", type(value).__name__)
            print("  Shape:", shape)
            print("  Dtype:", dtype)

            try:
                arr = (
                    value.to_numpy()
                    if hasattr(value, "to_numpy")
                    else np.asarray(value)
                )

                if key.lower().startswith("y") or "label" in key.lower():
                    unique, counts = np.unique(
                        arr, return_counts=True
                    )
                    print("  Class distribution:",
                          dict(zip(unique.tolist(), counts.tolist())))

                elif any(term in key.lower() for term in
                         ["group", "bearing", "condition"]):
                    unique, counts = np.unique(
                        arr.astype(str), return_counts=True
                    )
                    print("  Unique values:", len(unique))
                    print("  Value counts:",
                          dict(zip(unique.tolist(), counts.tolist())))

                elif key.lower().startswith("x"):
                    if np.issubdtype(arr.dtype, np.number):
                        print("  All finite:", np.isfinite(arr).all())

            except Exception as error:
                print("  Inspection note:", error)


In [ ]:

from pathlib import Path
import joblib

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

file_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "stage1_scaled_sequences.joblib"
)

stage1_data = joblib.load(file_path)
dataset = stage1_data["all_22"]

for split in ["train", "val", "test"]:
    meta = dataset[f"meta_{split}"]

    print("\n" + "=" * 55)
    print(f"{split.upper()} METADATA")
    print("=" * 55)

    print("Columns:", meta.columns.tolist())
    print("\nFirst five rows:")
    print(meta.head().to_string(index=False))

    for column in meta.columns:
        if "bearing" in column.lower():
            print(f"\nUnique bearings in {split}:")
            print(sorted(meta[column].astype(str).unique()))
            print("Number of bearings:", meta[column].nunique())


In [ ]:

# ============================================================
# STAGE 1: 1D CNN — Healthy vs Damaged
# Same all_22 data and bearing-level splits as previous models
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

print("TensorFlow version:", tf.__version__)

# ------------------------------------------------------------
# 2. Load the existing Stage 1 sequences
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "stage1_scaled_sequences.joblib"
)

data = joblib.load(DATA_PATH)
dataset = data["all_22"]

X_train = np.asarray(dataset["X_train"], dtype=np.float32)
y_train = np.asarray(dataset["y_train"], dtype=np.int32)

X_val = np.asarray(dataset["X_val"], dtype=np.float32)
y_val = np.asarray(dataset["y_val"], dtype=np.int32)

X_test = np.asarray(dataset["X_test"], dtype=np.float32)
y_test = np.asarray(dataset["y_test"], dtype=np.int32)

# ------------------------------------------------------------
# 3. Verify shapes and bearing-level separation
# ------------------------------------------------------------

assert X_train.shape[1:] == (5, 22)
assert X_val.shape[1:] == (5, 22)
assert X_test.shape[1:] == (5, 22)

assert set(np.unique(y_train)) == {0, 1}
assert set(np.unique(y_val)) == {0, 1}
assert set(np.unique(y_test)) == {0, 1}

bearing_groups = {
    split: set(
        dataset[f"meta_{split}"]["bearing"].astype(str).unique()
    )
    for split in ["train", "val", "test"]
}

assert bearing_groups["train"].isdisjoint(bearing_groups["val"])
assert bearing_groups["train"].isdisjoint(bearing_groups["test"])
assert bearing_groups["val"].isdisjoint(bearing_groups["test"])

print("\nInput shapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nPhysical bearing counts:")
for split, bearings in bearing_groups.items():
    print(f"{split}: {len(bearings)} bearings")

print("\nTraining class distribution:")
print(dict(zip(*np.unique(y_train, return_counts=True))))

# ------------------------------------------------------------
# 4. Handle class imbalance
# ------------------------------------------------------------

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes.tolist(), weights.tolist()))

print("\nClass weights:", class_weights)

# ------------------------------------------------------------
# 5. Build the 1D CNN
# ------------------------------------------------------------

model = Sequential([
    Input(shape=(5, 22)),

    Conv1D(
        filters=32,
        kernel_size=2,
        padding="same",
        activation="relu"
    ),

    Conv1D(
        filters=64,
        kernel_size=2,
        padding="same",
        activation="relu"
    ),

    GlobalAveragePooling1D(),

    Dense(32, activation="relu"),

    Dropout(0.30),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("\nCNN architecture:")
model.summary()

print("\nTotal trainable parameters:", model.count_params())

# ------------------------------------------------------------
# 6. Train using training and validation data only
# ------------------------------------------------------------

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)

# ------------------------------------------------------------
# 7. Evaluate on validation data only
# ------------------------------------------------------------

val_probabilities = model.predict(
    X_val,
    verbose=0
).ravel()

val_predictions = (
    val_probabilities >= 0.5
).astype(int)

val_metrics = {
    "accuracy": accuracy_score(y_val, val_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_val, val_predictions
    ),
    "precision": precision_score(
        y_val, val_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_val, val_predictions, zero_division=0
    ),
    "f1": f1_score(
        y_val, val_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_val, val_probabilities
    )
}

print("\n" + "=" * 55)
print("CNN VALIDATION RESULTS")
print("=" * 55)

for metric, value in val_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_predictions))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_predictions,
        target_names=["Healthy", "Damaged"],
        zero_division=0
    )
)

# ------------------------------------------------------------
# 8. Save model and training history
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "cnn_sequence_stage1"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / "cnn_stage1_all22.keras"
history_path = OUTPUT_DIR / "cnn_stage1_history.csv"
metrics_path = OUTPUT_DIR / "cnn_stage1_validation_metrics.json"

model.save(model_path)

pd.DataFrame(history.history).to_csv(
    history_path,
    index=False
)

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(val_metrics, file, indent=4)

print("\nSaved files:")
print(model_path)
print(history_path)
print(metrics_path)

print("\nCNN training and validation completed.")
print("The test set has NOT been evaluated in this cell.")


In [ ]:

# ============================================================
# STAGE 1 CNN: ONE-TIME TEST EVALUATION
# Do not tune the model or threshold using these results.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "stage1_scaled_sequences.joblib"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "cnn_sequence_stage1"
    / "cnn_stage1_all22.keras"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "cnn_sequence_stage1"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load saved data and best CNN model
data = joblib.load(DATA_PATH)
dataset = data["all_22"]

model = tf.keras.models.load_model(MODEL_PATH)

X_test = np.asarray(dataset["X_test"], dtype=np.float32)
y_test = np.asarray(dataset["y_test"], dtype=np.int32)

test_meta = dataset["meta_test"].reset_index(drop=True).copy()

assert X_test.shape == (320, 5, 22)
assert y_test.shape == (320,)
assert len(test_meta) == len(y_test)
assert model.input_shape[1:] == (5, 22)

# 2. Verify that test bearings are separate from train/validation
train_bearings = set(
    dataset["meta_train"]["bearing"].astype(str).unique()
)
val_bearings = set(
    dataset["meta_val"]["bearing"].astype(str).unique()
)
test_bearings = set(
    test_meta["bearing"].astype(str).unique()
)

assert train_bearings.isdisjoint(test_bearings)
assert val_bearings.isdisjoint(test_bearings)

# 3. Predict using the fixed 0.5 threshold
test_probabilities = model.predict(
    X_test,
    verbose=0
).ravel()

test_predictions = (
    test_probabilities >= 0.5
).astype(int)

# 4. Calculate metrics
test_metrics = {
    "accuracy": float(
        accuracy_score(y_test, test_predictions)
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_test, test_predictions)
    ),
    "precision": float(
        precision_score(y_test, test_predictions, zero_division=0)
    ),
    "recall": float(
        recall_score(y_test, test_predictions, zero_division=0)
    ),
    "f1": float(
        f1_score(y_test, test_predictions, zero_division=0)
    ),
    "roc_auc": float(
        roc_auc_score(y_test, test_probabilities)
    )
}

print("=" * 60)
print("CNN — STAGE 1 TEST RESULTS")
print("=" * 60)

for name, value in test_metrics.items():
    print(f"{name}: {value:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1]
)
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Healthy", "Damaged"],
        zero_division=0
    )
)

# 5. Per-bearing evaluation
test_results = test_meta.copy()

test_results["actual"] = y_test
test_results["predicted"] = test_predictions
test_results["probability_damaged"] = test_probabilities

per_bearing = (
    test_results.groupby("bearing")
    .apply(
        lambda group: pd.Series({
            "sequences": len(group),
            "actual_class": int(group["actual"].iloc[0]),
            "actual_classes_present": group["actual"].nunique(),
            "accuracy": accuracy_score(
                group["actual"], group["predicted"]
            ),
            "errors": int(
                (group["actual"] != group["predicted"]).sum()
            ),
            "mean_damage_probability": group[
                "probability_damaged"
            ].mean()
        }),
        include_groups=False
    )
    .reset_index()
)

print("\nPer-bearing results:")
print(per_bearing.to_string(index=False))

# 6. Save evaluation artifacts
with open(
    OUTPUT_DIR / "cnn_stage1_test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(test_metrics, file, indent=4)

test_results.to_csv(
    OUTPUT_DIR / "cnn_stage1_test_predictions.csv",
    index=False
)

per_bearing.to_csv(
    OUTPUT_DIR / "cnn_stage1_per_bearing_results.csv",
    index=False
)

print("\nSaved test evaluation files in:")
print(OUTPUT_DIR)


In [ ]:

# ============================================================
# STAGE 2: 1D CNN — Outer vs Inner vs Both
# Uses the same architecture as Stage 1
# Input: 5 measurements x 12 selected features
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# 2. Load existing Stage 2 sequences
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "stage2_scaled_sequences.joblib"
)

data = joblib.load(DATA_PATH)
dataset = data["top_12"]

X_train = np.asarray(dataset["X_train"], dtype=np.float32)
y_train = np.asarray(dataset["y_train"], dtype=np.int32)

X_val = np.asarray(dataset["X_val"], dtype=np.float32)
y_val = np.asarray(dataset["y_val"], dtype=np.int32)

X_test = np.asarray(dataset["X_test"], dtype=np.float32)
y_test = np.asarray(dataset["y_test"], dtype=np.int32)

# ------------------------------------------------------------
# 3. Verify data
# ------------------------------------------------------------

assert X_train.shape == (1152, 5, 12)
assert X_val.shape == (254, 5, 12)
assert X_test.shape == (256, 5, 12)

assert set(np.unique(y_train)) == {0, 1, 2}
assert set(np.unique(y_val)) == {0, 1, 2}
assert set(np.unique(y_test)) == {0, 1, 2}

train_bearings = set(
    dataset["meta_train"]["bearing"].astype(str).unique()
)

val_bearings = set(
    dataset["meta_val"]["bearing"].astype(str).unique()
)

test_bearings = set(
    dataset["meta_test"]["bearing"].astype(str).unique()
)

assert train_bearings.isdisjoint(val_bearings)
assert train_bearings.isdisjoint(test_bearings)
assert val_bearings.isdisjoint(test_bearings)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print("\nTraining distribution:")
print(dict(zip(*np.unique(y_train, return_counts=True))))

print("\nValidation distribution:")
print(dict(zip(*np.unique(y_val, return_counts=True))))

# ------------------------------------------------------------
# 4. Calculate class weights
# ------------------------------------------------------------

classes = np.array([0, 1, 2])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(label): float(weight)
    for label, weight in zip(classes, weights)
}

print("\nClass weights:")
print(class_weights)

# ------------------------------------------------------------
# 5. Build the same CNN architecture
# ------------------------------------------------------------

model = Sequential([
    Input(shape=(5, 12)),

    Conv1D(
        filters=32,
        kernel_size=2,
        padding="same",
        activation="relu"
    ),

    Conv1D(
        filters=64,
        kernel_size=2,
        padding="same",
        activation="relu"
    ),

    GlobalAveragePooling1D(),

    Dense(32, activation="relu"),

    Dropout(0.30),

    Dense(3, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nCNN architecture:")
model.summary()

print("Trainable parameters:", model.count_params())

# ------------------------------------------------------------
# 6. Train
# ------------------------------------------------------------

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)

# ------------------------------------------------------------
# 7. Validation evaluation
# ------------------------------------------------------------

val_probabilities = model.predict(
    X_val,
    verbose=0
)

val_predictions = np.argmax(
    val_probabilities,
    axis=1
)

metrics = {
    "accuracy": accuracy_score(y_val, val_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_val, val_predictions
    ),
    "macro_precision": precision_score(
        y_val, val_predictions,
        average="macro",
        zero_division=0
    ),
    "macro_recall": recall_score(
        y_val, val_predictions,
        average="macro",
        zero_division=0
    ),
    "macro_f1": f1_score(
        y_val, val_predictions,
        average="macro",
        zero_division=0
    ),
    "weighted_f1": f1_score(
        y_val, val_predictions,
        average="weighted",
        zero_division=0
    )
}

print("\n" + "=" * 60)
print("STAGE 2 CNN — VALIDATION RESULTS")
print("=" * 60)

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        val_predictions,
        labels=[0, 1, 2]
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_predictions,
        labels=[0, 1, 2],
        target_names=["Outer", "Inner", "Both"],
        zero_division=0
    )
)

# ------------------------------------------------------------
# 8. Save model, history and validation metrics
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "cnn_sequence_stage2"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save(
    OUTPUT_DIR / "cnn_stage2_top12.keras"
)

pd.DataFrame(history.history).to_csv(
    OUTPUT_DIR / "cnn_stage2_history.csv",
    index=False
)

with open(
    OUTPUT_DIR / "cnn_stage2_validation_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {key: float(value) for key, value in metrics.items()},
        file,
        indent=4
    )

print("\nStage 2 CNN training and validation completed.")
print("Test data has NOT been evaluated.")


In [ ]:

# ============================================================
# STAGE 2 CNN: TEST EVALUATION
# Outer vs Inner vs Both
# Fixed saved model; no retraining or tuning
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "stage2_scaled_sequences.joblib"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "cnn_sequence_stage2"
    / "cnn_stage2_top12.keras"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "cnn_sequence_stage2"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Outer", "Inner", "Both"]

# ------------------------------------------------------------
# 1. Load saved model and test data
# ------------------------------------------------------------

data = joblib.load(DATA_PATH)
dataset = data["top_12"]

model = tf.keras.models.load_model(MODEL_PATH)

X_test = np.asarray(
    dataset["X_test"],
    dtype=np.float32
)

y_test = np.asarray(
    dataset["y_test"],
    dtype=np.int32
)

test_meta = (
    dataset["meta_test"]
    .reset_index(drop=True)
    .copy()
)

assert X_test.shape == (256, 5, 12)
assert y_test.shape == (256,)
assert len(test_meta) == len(y_test)
assert model.input_shape[1:] == (5, 12)

# ------------------------------------------------------------
# 2. Verify physical bearing separation
# ------------------------------------------------------------

train_bearings = set(
    dataset["meta_train"]["bearing"].astype(str).unique()
)

val_bearings = set(
    dataset["meta_val"]["bearing"].astype(str).unique()
)

test_bearings = set(
    test_meta["bearing"].astype(str).unique()
)

assert train_bearings.isdisjoint(test_bearings)
assert val_bearings.isdisjoint(test_bearings)

print("Test physical bearings:", sorted(test_bearings))
print("Number of test bearings:", len(test_bearings))

# ------------------------------------------------------------
# 3. Predict
# ------------------------------------------------------------

test_probabilities = model.predict(
    X_test,
    verbose=0
)

test_predictions = np.argmax(
    test_probabilities,
    axis=1
)

# ------------------------------------------------------------
# 4. Calculate overall metrics
# ------------------------------------------------------------

metrics = {
    "accuracy": float(
        accuracy_score(y_test, test_predictions)
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_test, test_predictions)
    ),
    "macro_precision": float(
        precision_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        )
    ),
    "macro_recall": float(
        recall_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        )
    ),
    "macro_f1": float(
        f1_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        )
    ),
    "weighted_f1": float(
        f1_score(
            y_test,
            test_predictions,
            average="weighted",
            zero_division=0
        )
    )
}

print("\n" + "=" * 60)
print("STAGE 2 CNN — TEST RESULTS")
print("=" * 60)

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

# ------------------------------------------------------------
# 5. Confusion matrix and classification report
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1, 2]
)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 6. Per-sequence prediction results
# ------------------------------------------------------------

test_results = test_meta.copy()

test_results["actual"] = y_test
test_results["predicted"] = test_predictions
test_results["actual_name"] = [
    CLASS_NAMES[i] for i in y_test
]
test_results["predicted_name"] = [
    CLASS_NAMES[i] for i in test_predictions
]

for class_id, class_name in enumerate(CLASS_NAMES):
    test_results[
        f"probability_{class_name.lower()}"
    ] = test_probabilities[:, class_id]

# ------------------------------------------------------------
# 7. Per-bearing evaluation
# ------------------------------------------------------------

per_bearing_rows = []

for bearing, group in test_results.groupby("bearing"):
    actual = group["actual"].to_numpy()
    predicted = group["predicted"].to_numpy()

    row = {
        "bearing": bearing,
        "sequences": len(group),
        "actual_class": CLASS_NAMES[int(actual[0])],
        "actual_classes_present": len(np.unique(actual)),
        "accuracy": float(
            accuracy_score(actual, predicted)
        ),
        "errors": int(
            np.sum(actual != predicted)
        )
    }

    for class_id, class_name in enumerate(CLASS_NAMES):
        row[
            f"mean_probability_{class_name.lower()}"
        ] = float(
            group[
                f"probability_{class_name.lower()}"
            ].mean()
        )

    per_bearing_rows.append(row)

per_bearing = pd.DataFrame(per_bearing_rows)

print("\nPer-bearing results:")
print(per_bearing.to_string(index=False))

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

with open(
    OUTPUT_DIR / "cnn_stage2_test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(metrics, file, indent=4)

test_results.to_csv(
    OUTPUT_DIR / "cnn_stage2_test_predictions.csv",
    index=False
)

per_bearing.to_csv(
    OUTPUT_DIR / "cnn_stage2_per_bearing_results.csv",
    index=False
)

pd.DataFrame(
    cm,
    index=[f"Actual_{name}" for name in CLASS_NAMES],
    columns=[f"Predicted_{name}" for name in CLASS_NAMES]
).to_csv(
    OUTPUT_DIR / "cnn_stage2_test_confusion_matrix.csv"
)

print("\nSaved evaluation files:")
print(OUTPUT_DIR)


In [ ]:

# ============================================================
# STAGE 2: PHYSICAL-BEARING SUPPORT AUDIT
# Counts independent bearings supporting each fault class
# ============================================================

from pathlib import Path
import joblib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "stage2_scaled_sequences.joblib"
)

data = joblib.load(DATA_PATH)
dataset = data["top_12"]

CLASS_NAMES = {
    0: "Outer",
    1: "Inner",
    2: "Both"
}

all_tables = []

for split in ["train", "val", "test"]:

    meta = dataset[f"meta_{split}"].reset_index(drop=True).copy()
    labels = np.asarray(dataset[f"y_{split}"])

    assert len(meta) == len(labels)

    meta["class_id"] = labels
    meta["fault_class"] = meta["class_id"].map(CLASS_NAMES)

    # Number of sequences belonging to each bearing and fault class
    bearing_table = (
        meta.groupby(["bearing", "fault_class"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for class_name in CLASS_NAMES.values():
        if class_name not in bearing_table.columns:
            bearing_table[class_name] = 0

    bearing_table["partition"] = split

    all_tables.append(bearing_table)

    print("\n" + "=" * 60)
    print(f"{split.upper()} — BEARING-LEVEL CLASS SUPPORT")
    print("=" * 60)

    print(bearing_table.to_string(index=False))

    print("\nIndependent bearings supporting each class:")

    for class_name in CLASS_NAMES.values():
        supported = int(
            (bearing_table[class_name] > 0).sum()
        )
        sequences = int(bearing_table[class_name].sum())

        print(
            f"{class_name}: {supported} bearings, "
            f"{sequences} sequences"
        )

final_audit = pd.concat(all_tables, ignore_index=True)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "results"
    / "cnn_sequence_stage2"
    / "stage2_bearing_support_audit.csv"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final_audit.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)
